In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

PROJECT_FOLDER = Path(
    "/content/drive/MyDrive/London_Air_Quality_Project"
)

DATA_FOLDER = PROJECT_FOLDER / "data"
DATA_FOLDER.mkdir(parents=True, exist_ok=True)

print(DATA_FOLDER)

In [ ]:
import shutil

source = "/content/london_air_quality_2010_2023_top5_sites.csv"

destination = (
    DATA_FOLDER / "london_air_quality_2010_2023_top5_sites.csv"
)

if not destination.exists():
    shutil.copy2(source, destination)
    print("✅ Fichier sauvegardé dans Google Drive.")
else:
    print("✅ Le fichier existe déjà dans Google Drive.")

print("Emplacement :", destination)

In [ ]:
import pandas as pd

file_path = (
    "/content/drive/MyDrive/London_Air_Quality_Project/"
    "data/london_air_quality_2010_2023_top5_sites.csv"
)

df = pd.read_csv(file_path)

print("✅ Données chargées :", df.shape)
df.head()

# 🌍 Clean Skies London: Forecasting Daily NO₂ Levels

## 1. Project Objective

London continues to face significant air-quality challenges, particularly from pollutants such as nitrogen dioxide (NO₂), ozone (O₃), and particulate matter (PM10 and PM2.5). Exposure to high NO₂ concentrations can negatively affect respiratory health, especially among children, older adults, and people with pre-existing health conditions.

The organisation **Clean Skies London** wants to use historical air-quality data to better understand pollution patterns and predict future pollution levels. Reliable forecasts could help policymakers anticipate periods of high pollution and take preventive action.

### Main objective

The objective of this project is to develop and compare different time-series forecasting approaches for predicting the **daily average NO₂ concentration** at the **Kensington and Chelsea – North Ken** monitoring station.

### Project scope

The original dataset contains:

- hourly air-quality measurements;
- six different pollutants;
- five monitoring locations;
- observations covering several years.

For this analysis, the data will be reduced to:

- **Location:** Kensington and Chelsea – North Ken;
- **Pollutant:** Concentration Nitrogen Dioxide NO₂ (µg/m³);
- **Time frequency:** daily averages instead of hourly measurements;
- **Forecasting target:** future daily average NO₂ levels.

### Modelling approaches

Several forecasting approaches will be evaluated:

1. Baseline forecasting methods;
2. Classical statistical models such as AR, MA, ARIMA and SARIMA;
3. Prophet, including UK public holidays;
4. HistGradientBoostingRegressor using lag, rolling, calendar and Fourier features

The models will be evaluated using chronological train-test splitting and time-based cross-validation. Their performance will be compared using metrics such as MAE and RMSE, with MAPE used only when appropriate.

### Main research question

> Which forecasting approach provides the most accurate and reliable predictions of daily NO₂ levels in Kensington and Chelsea: classical statistical models, Prophet, or feature-based machine learning with HistGradientBoostingRegressor?

## 2. Data Loading and Understanding

Before focusing on a single location and pollutant, it's important to first understand the raw dataset as a whole: its structure, size, time coverage, and the different sites and pollutants it contains.

In this section, we load the full dataset and inspect:
- the shape and column types;
- the monitoring sites available;
- the time range covered;
- a first overview of missing values across the dataset.

This step ensures we know exactly what we're working with before reducing the scope to Kensington and Chelsea / NO₂.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


def load_air_quality_data(filepath: str) -> pd.DataFrame:
    """Load the raw London air-quality dataset and parse datetime."""

    return pd.read_csv(
        filepath,
        parse_dates=["datetime"]
    )


def summarize_dataset(df: pd.DataFrame) -> None:
    """Display a quick overview of the raw dataset."""

    print("Shape:", df.shape)
    print("\nColumn types:\n", df.dtypes)

    print("\nMonitoring sites:")
    display(
        df[["SiteCode", "SiteName"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    print(
        "\nDate range:",
        df["datetime"].min(),
        "to",
        df["datetime"].max()
    )

    print(
        "\nMissing values per column:\n",
        df.isna().sum()
    )


# Permanent file path in Google Drive
FILE_PATH = (
    "/content/drive/MyDrive/London_Air_Quality_Project/"
    "data/london_air_quality_2010_2023_top5_sites.csv"
)


# Load the dataset
df: pd.DataFrame = load_air_quality_data(FILE_PATH)


# Display the dataset summary
summarize_dataset(df)


# Preview the first five rows
df.head()

### Interpretation

The dataset contains **600,685 hourly observations and 14 columns**, covering the period from **1 January 2010 to 30 December 2023**.

Five monitoring stations are available in the dataset. The station selected for this project is:

- **Site code:** `KC1`
- **Site name:** `Kensington and Chelsea - North Ken`

The `datetime` column was successfully loaded as `datetime64[ns]`, which means it can be used directly for time-series operations.

The station information and geographical variables contain no missing values. However, the pollutant columns contain missing observations. Across all five stations, the target variable `no2_level` has **26,970 missing hourly values**.

Since this number concerns the complete dataset, missing NO₂ values will be examined again after filtering the data for the `KC1` monitoring station.
### What should we remember?

The dataset is ready for filtering. In the next section, we will retain only the `KC1` station and the variables required to study NO₂ levels.

## 3. Filtering Kensington and Chelsea and NO₂ Data

The original dataset contains several monitoring sites and pollutants. This project focuses exclusively on nitrogen dioxide measurements recorded at the Kensington and Chelsea monitoring station.

In this section, we will:

- select the monitoring station with the site code `KC1`;
- retain only the `datetime` and `no2_level` columns;
- arrange the observations chronologically;
- examine the date range and missing NO₂ measurements;
- check for duplicated timestamps.

In [ ]:
def filter_site_pollutant(
    df: pd.DataFrame,
    site_code: str,
    pollutant_col: str
) -> pd.DataFrame:
    """Filter the dataset to one site and pollutant, sorted chronologically."""
    subset = df.loc[
        df["SiteCode"] == site_code,
        ["datetime", pollutant_col]
    ].copy()

    return subset.sort_values("datetime").reset_index(drop=True)


def report_missing(df: pd.DataFrame, col: str) -> None:
    """Report the date range, missing values and duplicated timestamps."""
    print("Shape:", df.shape)
    print("Date range:", df["datetime"].min(), "to", df["datetime"].max())
    print("Missing values:", df[col].isna().sum())
    print("Missing %:", round(100 * df[col].isna().mean(), 2))
    print("Duplicated timestamps:", df["datetime"].duplicated().sum())


kc1_no2: pd.DataFrame = filter_site_pollutant(
    df,
    site_code="KC1",
    pollutant_col="no2_level"
)

report_missing(kc1_no2, "no2_level")
display(kc1_no2.head())

### Interpretation

After filtering, the dataset contains **121,807 hourly observations** for the `Kensington and Chelsea - North Ken` monitoring station (`KC1`). The data covers the period from **1 January 2010 at 00:00** to **30 December 2023 at 23:00**.

The target variable `no2_level` contains **2,952 missing hourly measurements**, representing **2.42%** of the filtered dataset.

There are **no duplicated timestamps**, meaning that each recorded date and hour appears only once.

The missing values will not be treated yet. We will first convert the hourly measurements into daily averages, since a day may still have enough valid hourly observations to calculate a representative daily mean.

### What should we remember?

The analysis now focuses exclusively on hourly NO₂ measurements from the Kensington and Chelsea station.

The dataset is chronologically ordered and contains no duplicated timestamps. However, 2.42% of its NO₂ measurements are missing.

In the next section, the hourly measurements will be aggregated into daily average NO₂ levels.

## 4. Converting Hourly Measurements into Daily Averages

The original NO₂ measurements are recorded hourly, but the objective is to forecast daily NO₂ levels.

For each day, we will calculate:

- the average NO₂ level;
- the number of valid hourly measurements.

A daily average will only be kept when at least 18 valid hourly measurements are available. Days with fewer than 18 valid measurements will be considered missing.

In [ ]:
# Use datetime as the index
kc1_no2_indexed = kc1_no2.set_index("datetime")

# Calculate the daily mean
daily_mean = kc1_no2_indexed["no2_level"].resample("D").mean()

# Count the valid hourly measurements for each day
daily_count = kc1_no2_indexed["no2_level"].resample("D").count()

# Create the daily DataFrame
kc1_no2_daily = pd.DataFrame({
    "no2_level": daily_mean,
    "hourly_count": daily_count
})

# Keep the daily mean only when at least 18 hourly values are available
kc1_no2_daily.loc[
    kc1_no2_daily["hourly_count"] < 18,
    "no2_level"
] = np.nan

# Display the results
print("Total days:", len(kc1_no2_daily))

print(
    "Days with fewer than 18 hourly measurements:",
    (kc1_no2_daily["hourly_count"] < 18).sum()
)

print(
    "Missing daily values:",
    kc1_no2_daily["no2_level"].isna().sum()
)

print(
    "Missing %:",
    round(
        100 * kc1_no2_daily["no2_level"].isna().sum()
        / len(kc1_no2_daily),
        2
    )
)

print(
    "Date range:",
    kc1_no2_daily.index.min(),
    "to",
    kc1_no2_daily.index.max()
)

kc1_no2_daily.head()

### Interpretation

The hourly NO₂ measurements were converted into **5,112 daily observations**, covering the period from **1 January 2010 to 30 December 2023**.

A daily average was retained only when at least **18 valid hourly measurements** were available. In total, **160 days** contained fewer than 18 valid hourly observations and were therefore considered missing.

These 160 missing daily values represent **3.13%** of the complete daily time series.

For example, the first five days of 2010 each contain 24 valid hourly measurements. Their daily averages are therefore considered sufficiently representative and have been retained.

### What should we remember?

The daily NO₂ time series contains 5,112 observations. After applying the minimum quality threshold of 18 valid hourly measurements per day, 160 daily values remain missing.

This quality rule ensures that daily averages based on too few hourly measurements are not used in the analysis. The distribution and duration of these 160 missing values will be examined in the next section.

## 5. Missing Daily Values: Analysis and Treatment

After applying the minimum quality threshold of 18 valid hourly measurements per day, 160 daily NO₂ values remain missing.

In this section, we will first examine how these missing values are distributed across the time series. We will then select an appropriate treatment while avoiding data leakage between the training and test periods.

### 5.1 Analysing Missing Daily Values

Before treating the 160 missing daily values, we examine:

- the number of separate missing periods;
- the duration of each period;
- the longest sequence of consecutive missing days;
- the distribution of gap lengths.

This analysis will help determine whether interpolation is appropriate.

In [ ]:
def analyse_missing_gaps(
    df: pd.DataFrame,
    value_col: str
) -> pd.DataFrame:
    """Identify consecutive periods of missing daily observations."""

    missing_mask: pd.Series = df[value_col].isna()

    # Create a new group whenever the missing status changes
    gap_groups: pd.Series = (
        missing_mask
        .ne(missing_mask.shift())
        .cumsum()
    )

    missing_gaps: pd.DataFrame = (
        df.loc[missing_mask]
        .assign(gap_group=gap_groups[missing_mask])
        .groupby("gap_group")
        .apply(
            lambda gap: pd.Series({
                "start_date": gap.index.min(),
                "end_date": gap.index.max(),
                "gap_length": len(gap)
            }),
            include_groups=False
        )
        .reset_index(drop=True)
    )

    return missing_gaps


missing_gaps: pd.DataFrame = analyse_missing_gaps(
    kc1_no2_daily,
    value_col="no2_level"
)

print(
    "Total missing days:",
    kc1_no2_daily["no2_level"].isna().sum()
)

print(
    "Number of missing periods:",
    len(missing_gaps)
)

print(
    "Longest missing period:",
    missing_gaps["gap_length"].max(),
    "days"
)

print("\nDistribution of gap lengths:")

print(
    missing_gaps["gap_length"]
    .value_counts()
    .sort_index()
)

display(
    missing_gaps
    .sort_values("gap_length", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

### Interpretation of Missing Periods

A total of 160 daily NO₂ values are missing across 58 separate periods.

Most missing periods are short:

- 26 periods contain one missing day;
- 23 periods contain two consecutive missing days;
- only nine periods are longer than two days.

Therefore, 49 of the 58 missing periods last no more than two days, suggesting that most gaps can reasonably be treated using time-based interpolation.

However, three substantially longer interruptions were identified:

- 29 days, from 6 January to 3 February 2022;
- 21 days, from 29 January to 18 February 2010;
- 13 days, from 25 May to 6 June 2011.

The longest gap contains 29 consecutive missing days. Interpolation across these extended periods is less reliable because it estimates pollution levels over several weeks without real measurements.

Overall, time-based interpolation is appropriate for maintaining a continuous daily series, particularly for the numerous short gaps. Nevertheless, values estimated within the longest missing periods must be interpreted cautiously and should not be considered actual pollution measurements.

The chronological train-test split will be performed before interpolation to prevent information from the 2023 test period from being used to complete the training data.

### 5.2 Treating Missing Daily Values

The missing values must be treated without allowing information from the test period to influence the training data.

The complete daily time series is therefore divided chronologically into:

- a training set containing observations from 1 January 2010 to 31 December 2022;
- a test set containing observations from 1 January 2023 to 30 December 2023.

The training set contains 4,748 observations, representing approximately 92.88% of the complete series. The test set contains 364 observations, representing approximately 7.12%.

Before treatment, the training set contains 153 missing NO₂ values, while the test set contains 7 missing NO₂ values.

Time-based interpolation is applied separately to the training and test sets. This prevents observations from the 2023 test period from being used to estimate missing values in the training period and therefore reduces the risk of data leakage.

A copy of the test set, named `test_original`, is preserved before interpolation. It retains the 7 missing NO₂ values and will later allow the forecasting models to be evaluated only on dates containing real observations.

In [ ]:
# Split the time series chronologically
train: pd.DataFrame = (
    kc1_no2_daily
    .loc[:"2022-12-31"]
    .copy()
)

test: pd.DataFrame = (
    kc1_no2_daily
    .loc["2023-01-01":]
    .copy()
)

# Preserve the original test set before interpolation
test_original: pd.DataFrame = test.copy()

# Calculate the actual proportions
total_observations: int = len(kc1_no2_daily)

train_percentage: float = (
    len(train) / total_observations * 100
)

test_percentage: float = (
    len(test) / total_observations * 100
)

# Display the periods
print(
    "Training period:",
    train.index.min().date(),
    "to",
    train.index.max().date()
)

print(
    "Test period:",
    test.index.min().date(),
    "to",
    test.index.max().date()
)

# Display the number and percentage of observations
print(
    f"Training observations: {len(train)} "
    f"({train_percentage:.2f}%)"
)

print(
    f"Test observations: {len(test)} "
    f"({test_percentage:.2f}%)"
)

# Display missing values before treatment
print(
    "Missing training values:",
    train["no2_level"].isna().sum()
)

print(
    "Missing test values:",
    test["no2_level"].isna().sum()
)

print(
    "Missing values in test_original:",
    test_original["no2_level"].isna().sum()
)

In [ ]:
# Interpolate missing values separately
train["no2_level"] = (
    train["no2_level"]
    .interpolate(method="time")
)

test["no2_level"] = (
    test["no2_level"]
    .interpolate(method="time")
)

# Fill any missing values remaining at the end of the training set
train["no2_level"] = train["no2_level"].ffill()

# Check the missing values after treatment
print(
    "Missing values in treated train:",
    train["no2_level"].isna().sum()
)

print(
    "Missing values in treated test:",
    test["no2_level"].isna().sum()
)

print(
    "Missing values in test_original:",
    test_original["no2_level"].isna().sum()
)

### Interpretation

After separate treatment, the training and test sets contain no missing NO₂ values. The original test set retains its seven missing values so that final model evaluation can use only dates with real observations.

## 6. Exploratory Time-Series Analysis and Visualization

This section examines the temporal behaviour of daily NO₂ levels using only the training data from 2010 to 2022. It explores the long-term trend, seasonal patterns, variability and unusual pollution peaks while keeping 2023 reserved for final model evaluation.

In [ ]:
# Plot the daily NO₂ time series for the training period
fig: plt.Figure
ax: plt.Axes

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(
    train.index,
    train["no2_level"],
    color="steelblue",
    linewidth=0.8
)

ax.set_title("Daily NO₂ Levels at North Kensington (2010–2022)")
ax.set_xlabel("Date")
ax.set_ylabel("Daily Mean NO₂ Level (µg/m³)")
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

### Interpretation

The daily NO₂ series shows strong short-term fluctuations and several peaks above 100 µg/m³, especially during the earlier years.

Despite these variations, NO₂ levels generally decreased between 2010 and 2022. Recurring high and low periods suggest seasonality, while variability also appears to decline over time.

These patterns will be examined further using aggregation, decomposition and statistical tests.

In [ ]:
import matplotlib.dates as mdates

# Calculate the annual mean NO₂ level
annual_no2: pd.Series = (
    train["no2_level"]
    .resample("YE")
    .mean()
)

# Create the figure
fig: plt.Figure
ax: plt.Axes

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    annual_no2.index,
    annual_no2,
    marker="o",
    color="darkorange",
    linewidth=2
)

ax.set_title(
    "Annual Mean NO₂ Levels at North Kensington (2010–2022)"
)
ax.set_xlabel("Year")
ax.set_ylabel("Annual Mean NO₂ Level (µg/m³)")

# Display one tick for each year
ax.set_xticks(annual_no2.index)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

### Interpretation of the Annual Trend

Annual mean NO₂ concentrations show a clear long-term downward trend. They decreased from approximately 36.4 µg/m³ in 2010 to around 16.0 µg/m³ in 2022, representing a reduction of approximately 56%.

The decline was not continuous: temporary increases occurred in 2012–2013 and 2016. Nevertheless, concentrations fell markedly after 2017, with the sharpest annual decrease occurring between 2019 and 2020.

This changing mean indicates that the original NO₂ series is unlikely to be stationary.

In [ ]:
# Calculate the average NO₂ level for each month
monthly_no2: pd.Series = (
    train["no2_level"]
    .groupby(train.index.month)
    .mean()
)

# Month labels
month_labels: list[str] = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

# Create the figure
fig: plt.Figure
ax: plt.Axes

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    monthly_no2.index,
    monthly_no2.values,
    marker="o",
    color="seagreen",
    linewidth=2
)

ax.set_title("Average Monthly NO₂ Pattern at North Kensington (2010–2022)")
ax.set_xlabel("Month")
ax.set_ylabel("Mean NO₂ Level (µg/m³)")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

### Interpretation of the Monthly Pattern

The monthly averages reveal a clear seasonal pattern. NO₂ concentrations are highest in January, at approximately 39.5 µg/m³, and remain relatively high in November and December.

Concentrations generally decrease from winter to summer, reaching their lowest level in July at approximately 19.8 µg/m³. They then increase again from August to November.

The difference of approximately 20 µg/m³ between January and July indicates substantial annual seasonality, with higher average NO₂ levels during colder months and lower levels during warmer months. The graph describes this recurring pattern but does not, by itself, establish its causes.

## 7. Chronological Train-Test Split

The daily NO₂ series was split chronologically before missing-value treatment:

- training set: 4,748 observations, from 1 January 2010 to 31 December 2022;
- test set: 364 observations, from 1 January 2023 to 30 December 2023.

The year 2023 is reserved for final model evaluation, allowing performance to be assessed across almost all four seasons without using future observations during training.

In [ ]:
# Visualize the chronological train-test split
fig: plt.Figure
ax: plt.Axes

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(
    train.index,
    train["no2_level"],
    label="Training set (2010–2022)",
    color="steelblue",
    linewidth=0.8
)

ax.plot(
    test.index,
    test["no2_level"],
    label="Test set (2023)",
    color="darkorange",
    linewidth=1
)

ax.axvline(
    pd.Timestamp("2023-01-01"),
    color="black",
    linestyle="--",
    linewidth=1.2,
    label="Train-test boundary"
)

ax.set_title("Chronological Train-Test Split of Daily NO₂ Levels")
ax.set_xlabel("Date")
ax.set_ylabel("Daily Mean NO₂ Level (µg/m³)")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# Reconstruct the complete daily NO₂ series
full_daily_no2: pd.DataFrame = pd.concat([train, test]).sort_index()

# Plot the complete daily NO₂ time series
fig: plt.Figure
ax: plt.Axes

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(
    full_daily_no2.index,
    full_daily_no2["no2_level"],
    color="steelblue",
    linewidth=0.8,
    label="Daily mean NO₂"
)

# Mark the beginning of the unseen test period
ax.axvline(
    pd.Timestamp("2023-01-01"),
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="Start of test period (2023)"
)

ax.set_title("Daily NO₂ Levels at North Kensington (2010–2023)")
ax.set_xlabel("Date")
ax.set_ylabel("Daily Mean NO₂ Level (µg/m³)")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 8. Baseline Models

Baseline models provide simple reference forecasts for evaluating more advanced models.

Four approaches will be tested:

- Mean: predicts the training-set average;
- Naïve: uses the latest observed value;
- Seasonal Naïve: uses the value observed on the same day of the previous year;
- Drift: extends the overall trend between the first and last training observations.

Their forecasts will be compared with the real 2023 observations using MAE, RMSE and MAPE.

In [ ]:
# Create a DataFrame for the baseline forecasts
baseline_forecasts: pd.DataFrame = pd.DataFrame(
    index=test.index
)

# 1. Mean baseline
baseline_forecasts["Mean"] = train["no2_level"].mean()

# 2. Naïve baseline
baseline_forecasts["Naive"] = train["no2_level"].iloc[-1]

# 3. Seasonal naïve baseline
# Use the observation from the same calendar date one year earlier
previous_year_dates: pd.DatetimeIndex = (
    test.index - pd.DateOffset(years=1)
)

baseline_forecasts["Seasonal_Naive"] = (
    train["no2_level"]
    .reindex(previous_year_dates)
    .to_numpy()
)

# 4. Drift baseline
first_value: float = train["no2_level"].iloc[0]
last_value: float = train["no2_level"].iloc[-1]
n_train: int = len(train)

forecast_horizon: np.ndarray = np.arange(
    1,
    len(test) + 1
)

baseline_forecasts["Drift"] = (
    last_value
    + forecast_horizon
    * (last_value - first_value)
    / (n_train - 1)
)

# Add the treated test observations
baseline_forecasts["Actual"] = test["no2_level"]

display(baseline_forecasts.head())

In [ ]:
print("Mean forecast:", baseline_forecasts["Mean"].iloc[0])
print("Naïve forecast:", baseline_forecasts["Naive"].iloc[0])

print(
    "Missing seasonal naïve forecasts:",
    baseline_forecasts["Seasonal_Naive"].isna().sum()
)

print(
    "Baseline forecast dimensions:",
    baseline_forecasts.shape
)

In [ ]:
# Plot actual values and baseline forecasts
fig: plt.Figure
ax: plt.Axes

fig, ax = plt.subplots(figsize=(16, 7))

ax.plot(
    baseline_forecasts.index,
    baseline_forecasts["Actual"],
    label="Actual NO₂",
    color="black",
    linewidth=1.2
)

ax.plot(
    baseline_forecasts.index,
    baseline_forecasts["Mean"],
    label="Mean",
    linestyle="--"
)

ax.plot(
    baseline_forecasts.index,
    baseline_forecasts["Naive"],
    label="Naïve",
    linestyle="--"
)

ax.plot(
    baseline_forecasts.index,
    baseline_forecasts["Seasonal_Naive"],
    label="Seasonal Naïve",
    color="seagreen",
    linewidth=1
)

ax.plot(
    baseline_forecasts.index,
    baseline_forecasts["Drift"],
    label="Drift",
    linestyle="--"
)

ax.set_title("Baseline Forecasts for Daily NO₂ Levels in 2023")
ax.set_xlabel("Date")
ax.set_ylabel("Daily Mean NO₂ Level (µg/m³)")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

### Interpretation of Baseline Forecasts

The baseline models produce very different forecasts for 2023:

- The Mean model predicts a constant value of approximately 30 µg/m³ and overestimates most actual observations.
- The Naïve model remains constant at approximately 7.4 µg/m³, the final observed value of 2022. It captures the general level of some low-pollution days but cannot follow daily fluctuations.
- The Drift model starts near the Naïve forecast and gradually decreases to approximately 5 µg/m³. It also fails to represent the variability observed in 2023.
- The Seasonal Naïve model reproduces the daily pattern from 2022. It captures some fluctuations better than the constant models, but many peaks do not coincide with the actual 2023 peaks and are sometimes strongly overestimated.

Overall, none of the baseline models follows the actual daily NO₂ series consistently. Performance metrics are therefore required to identify the strongest baseline objectively.

## 9. Trend, Seasonality and Anomaly Analysis

This section examines the training data to identify:

- the long-term trend in NO₂ levels;
- recurring seasonal patterns;
- unusual observations or pollution peaks.

Rolling statistics and aggregated patterns will be used to distinguish these components before applying time-series decomposition.

In [ ]:
# Calculate rolling averages
rolling_30_days: pd.Series = (
    train["no2_level"]
    .rolling(window=30, center=True)
    .mean()
)

rolling_365_days: pd.Series = (
    train["no2_level"]
    .rolling(window=365, center=True)
    .mean()
)

# Plot the daily series and rolling averages
fig: plt.Figure
ax: plt.Axes

fig, ax = plt.subplots(figsize=(16, 7))

ax.plot(
    train.index,
    train["no2_level"],
    label="Daily NO₂",
    color="lightsteelblue",
    linewidth=0.6,
    alpha=0.6
)

ax.plot(
    train.index,
    rolling_30_days,
    label="30-day rolling mean",
    color="darkorange",
    linewidth=1.5
)

ax.plot(
    train.index,
    rolling_365_days,
    label="365-day rolling mean",
    color="darkred",
    linewidth=2.5
)

ax.set_title("NO₂ Trend Using Rolling Averages (2010–2022)")
ax.set_xlabel("Date")
ax.set_ylabel("Daily Mean NO₂ Level (µg/m³)")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

### Interpretation of Rolling Averages

The 30-day rolling mean shows recurring short-term and seasonal fluctuations, with generally higher NO₂ levels during colder periods and lower levels during warmer periods.

The 365-day rolling mean reveals a clear long-term downward trend, decreasing from approximately 36 µg/m³ in 2010 to around 16 µg/m³ in 2022. A temporary increase is visible around 2017, followed by a stronger decline after 2019.

These changing averages confirm that the original NO₂ series is not stationary.

In [ ]:
# Prepare monthly mean NO₂ data
monthly_seasonality: pd.DataFrame = (
    train["no2_level"]
    .resample("ME")
    .mean()
    .to_frame()
)

# Plot one line per year
sns.relplot(
    kind="line",
    data=monthly_seasonality,
    x=monthly_seasonality.index.month,
    y="no2_level",
    hue=monthly_seasonality.index.year,
    palette="tab20",
    marker="o",
    height=6,
    aspect=1.7
).set(
    title="Monthly Seasonality of NO₂ Levels (2010–2022)",
    xlabel="Month",
    ylabel="Monthly Mean NO₂ Level (µg/m³)"
)

plt.xticks(
    range(1, 13),
    [
        "Jan", "Feb", "Mar", "Apr", "May", "Jun",
        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
    ]
)

plt.show()

### Interpretation of Monthly Seasonality

The seasonal plot shows a broadly recurring annual pattern in NO₂ concentrations between 2010 and 2022.

For most years, concentrations generally decrease from the beginning of the year toward summer, reaching their lowest levels around June or July. They then increase again from August toward autumn and winter, particularly in November and December.

However, the pattern is not identical every year. Some months show substantial year-to-year variation, especially January, March and December. For example, January 2017 and December 2010–2016 contain particularly high monthly averages.

The curves for 2020–2022 are generally lower than those of earlier years. This reflects the long-term decline in NO₂ levels already observed in the rolling averages, rather than seasonality alone.

Overall, the figure provides evidence of annual seasonality, with generally higher NO₂ concentrations during colder months and lower concentrations during summer. The strength and exact timing of this seasonal pattern vary across years.

In [ ]:
# Calculate the centred 30-day rolling mean and standard deviation
rolling_mean_30: pd.Series = (
    train["no2_level"]
    .rolling(window=30, center=True)
    .mean()
)

rolling_std_30: pd.Series = (
    train["no2_level"]
    .rolling(window=30, center=True)
    .std()
)

# Calculate the local z-score
local_z_score: pd.Series = (
    (train["no2_level"] - rolling_mean_30)
    / rolling_std_30
)

# Identify observations more than three standard deviations
# away from their local 30-day mean
anomaly_mask: pd.Series = local_z_score.abs() > 3

anomalies: pd.DataFrame = train.loc[
    anomaly_mask,
    ["no2_level"]
].copy()

anomalies["local_z_score"] = local_z_score.loc[anomaly_mask]

print(f"Number of potential anomalies: {len(anomalies)}")
display(anomalies.sort_values("local_z_score", ascending=False).head(10))

In [ ]:
# Plot the daily series and potential anomalies
fig: plt.Figure
ax: plt.Axes

fig, ax = plt.subplots(figsize=(16, 7))

ax.plot(
    train.index,
    train["no2_level"],
    label="Daily NO₂",
    color="steelblue",
    linewidth=0.7,
    alpha=0.7
)

ax.scatter(
    anomalies.index,
    anomalies["no2_level"],
    label="Potential anomalies",
    color="red",
    s=30,
    zorder=3
)

ax.set_title("Potential Anomalies in Daily NO₂ Levels (2010–2022)")
ax.set_xlabel("Date")
ax.set_ylabel("Daily Mean NO₂ Level (µg/m³)")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

### Interpretation of Potential Anomalies

The graph highlights several daily observations that differ substantially from the local behaviour of NO₂ concentrations.

The red points represent observations located more than three standard deviations from their centred 30-day rolling mean. They are therefore considered potential statistical anomalies.

The highest anomalies mainly occur between 2014 and 2017, with some concentrations exceeding 110–120 µg/m³. After 2018, the detected anomalies generally have lower values, consistent with the long-term decline in NO₂ concentrations.

An observation does not need to be extremely high across the entire dataset to be locally unusual. For example, the point detected in 2022 is low compared with earlier pollution peaks but remains unusual relative to the surrounding observations.

These observations will not be removed automatically. They may represent genuine short-term pollution events, unusual conditions or measurement errors. Without external evidence confirming their cause, they will be retained and described as potential anomalies.

## 10. Time-Series Decomposition

Time-series decomposition separates the observed NO₂ series into:

- **trend:** the long-term evolution of NO₂ concentrations;
- **seasonality:** the recurring annual pattern;
- **residuals:** the remaining unexplained variations.

### Why Was an Additive Decomposition Selected?

The additive model was selected based on evidence calculated exclusively from the training set.

- **Seasonality is not proportional to the series level.** Between 2010–2013 and 2019–2022, the average NO₂ level decreased by 43%, from 36.4 to 20.9 µg/m³. However, the winter–summer seasonal amplitude decreased by only 22%, from 21.8 to 17.1 µg/m³. This indicates that seasonal fluctuations remain relatively stable in absolute units.

- **There is no marked heteroscedasticity.** The correlation between the trend level and absolute residual magnitude is close to zero for both models: approximately 0.14 for the additive decomposition and −0.11 for the multiplicative decomposition. Residual variability therefore does not clearly increase with the series level.

- **Both models explain almost the same variance.** On the same µg/m³ scale, the additive model explains 41.53% of the variance, compared with 41.63% for the multiplicative model. This difference of 0.10 percentage points is negligible.

- **The additive model is easier to interpret.** Its components are expressed directly in µg/m³, making the results more meaningful for public-health and policy applications.

The additive model therefore provides the best balance between empirical validity, simplicity and interpretability:

\[
Y_t = T_t + S_t + R_t
\]

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose, DecomposeResult

# Perform additive decomposition on the training series
decomposition: DecomposeResult = seasonal_decompose(
    train["no2_level"],
    model="additive",
    period=365
)

# Display the traditional decomposition plot
fig = decomposition.plot()
fig.set_size_inches(16, 10)
fig.suptitle(
    "Additive Decomposition of Daily NO₂ Levels (2010–2022)",
    fontsize=14,
    y=1.02
)

fig.tight_layout()
plt.show()

### Interpretation of the Additive Decomposition

The decomposition reveals three important characteristics of the daily NO₂ series:

- **Trend:** The long-term trend generally decreases from approximately 36–37 µg/m³ in 2010 to around 16 µg/m³ in 2022. A temporary increase appears around 2016–2017, followed by a stronger decline after 2019. This confirms the long-term reduction already observed with the 365-day rolling mean.

- **Seasonality:** A strong annual pattern repeats throughout the training period. Seasonal effects are generally positive during colder periods and negative during summer, indicating higher NO₂ concentrations in winter and lower concentrations in warmer months.

- **Residuals:** Most residuals fluctuate around zero, but several large positive and negative values remain. The largest positive residuals occur around 2016–2017 and represent pollution peaks that cannot be explained by the estimated trend or annual seasonality.

Overall, the decomposition captures the main downward trend and recurring annual cycle. However, the remaining residual variation indicates that daily NO₂ concentrations are also affected by short-term factors not represented by this basic decomposition, such as weather, traffic conditions or exceptional pollution events.

The missing trend and residual values at the beginning and end of the series are expected because the decomposition estimates the trend using a centred moving average.

## 11. Stationarity Analysis

Before fitting AR, MA, ARIMA or SARIMA models, the stationarity of the NO₂ time series must be examined.

A stationary series has statistical properties that remain approximately constant over time, particularly:

- a stable mean;
- a stable variance;
- an autocorrelation structure that does not change over time.

The previous analyses revealed a long-term downward trend and annual seasonality. Therefore, the original NO₂ series is unlikely to be stationary.

Stationarity will be evaluated using:

1. visual inspection of rolling statistics;
2. the Augmented Dickey–Fuller (ADF) test;
3. differencing if the original series is found to be non-stationary.

### 11.1 Visual Inspection of Rolling Statistics

Rolling statistics help assess whether the statistical properties of the NO₂ series remain stable over time.

A 365-day window is used to reduce short-term and seasonal fluctuations:

- the rolling mean shows whether the average level changes over time;
- the rolling standard deviation shows whether the variability remains stable.

A stationary series should have rolling statistics that are approximately constant over time.

In [ ]:
# Calculate 365-day rolling statistics
rolling_mean_365: pd.Series = (
    train["no2_level"]
    .rolling(window=365)
    .mean()
)

rolling_std_365: pd.Series = (
    train["no2_level"]
    .rolling(window=365)
    .std()
)

# Plot the original series and rolling statistics
fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(16, 10),
    sharex=True
)

# Original series and rolling mean
axes[0].plot(
    train.index,
    train["no2_level"],
    color="lightsteelblue",
    linewidth=0.7,
    alpha=0.6,
    label="Daily NO₂"
)

axes[0].plot(
    rolling_mean_365.index,
    rolling_mean_365,
    color="darkred",
    linewidth=2,
    label="365-day rolling mean"
)

axes[0].set_title("Daily NO₂ Levels and 365-Day Rolling Mean")
axes[0].set_ylabel("NO₂ Level (µg/m³)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Rolling standard deviation
axes[1].plot(
    rolling_std_365.index,
    rolling_std_365,
    color="darkgreen",
    linewidth=2,
    label="365-day rolling standard deviation"
)

axes[1].set_title("365-Day Rolling Standard Deviation")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Standard Deviation (µg/m³)")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()

### Interpretation of Rolling Statistics

The 365-day rolling statistics are not constant over time, providing visual evidence that the original NO₂ series is non-stationary.

- **Rolling mean:** The annual rolling mean decreases from approximately 36 µg/m³ in 2011 to around 16 µg/m³ in 2022. A temporary increase appears around 2017, followed by a stronger decline after 2019. This confirms that the mean changes over time.

- **Rolling standard deviation:** The annual rolling standard deviation also varies considerably. It rises to nearly 20 µg/m³ around 2017 before decreasing to approximately 10–11 µg/m³ by 2022. Therefore, the variability of the series is not stable either.

Overall, both the mean and variance evolve over time. The original daily NO₂ series therefore does not appear stationary based on this visual inspection. The Augmented Dickey–Fuller test will now be used to assess stationarity statistically.

The rolling curves begin later than the original series because a complete 365-day window is required before the first rolling statistic can be calculated.

### 11.2 Augmented Dickey–Fuller Test

The Augmented Dickey–Fuller (ADF) test is used to assess statistically whether the original training series is stationary.

The hypotheses are:

- **H₀:** the series has a unit root and is non-stationary;
- **H₁:** the series has no unit root and is stationary.

At a 5% significance level:

- if the p-value is below 0.05, H₀ is rejected;
- otherwise, H₀ cannot be rejected.

In [ ]:
from statsmodels.tsa.stattools import adfuller

# Run the ADF test on the original training series
adf_result = adfuller(
    train["no2_level"].dropna(),
    autolag="AIC"
)

# Extract the results
adf_statistic: float = adf_result[0]
p_value: float = adf_result[1]
lags_used: int = adf_result[2]
observations_used: int = adf_result[3]
critical_values: dict = adf_result[4]

# Display the results
print("Augmented Dickey–Fuller Test")
print("-" * 40)
print(f"ADF statistic: {adf_statistic:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"Lags used: {lags_used}")
print(f"Observations used: {observations_used}")

print("\nCritical values:")
for significance_level, critical_value in critical_values.items():
    print(f"  {significance_level}: {critical_value:.4f}")

# Interpret the result
alpha: float = 0.05

if p_value < alpha:
    print("\nDecision: Reject H₀.")
    print("No statistical evidence of a unit root was found.")
    print("This result alone does not confirm complete stationarity.")
else:
    print("\nDecision: Fail to reject H₀.")
    print("The series is not statistically stationary according to the ADF test.")

The ADF test rejects the null hypothesis of a unit root because the p-value (0.0001) is below 0.05 and the ADF statistic (−4.8113) is lower than the critical values.

Therefore, no statistical evidence of a unit root is found in the original training series. However, the rolling statistics still show a changing mean and variance, while the previous analysis identified annual seasonality.

Consequently, the ADF result alone is not sufficient to conclude that the series is fully stationary. Additional diagnostics are required before deciding whether trend or seasonal differencing should be applied.

### 11.3 KPSS Test

The KPSS test complements the ADF test by examining stationarity from the opposite perspective.

The hypotheses are:

- **H₀:** the series is stationary around a constant or a deterministic trend;
- **H₁:** the series is non-stationary.

At a 5% significance level:

- if the p-value is below 0.05, H₀ is rejected;
- otherwise, H₀ cannot be rejected.

Two KPSS specifications are tested:

- `regression="c"` tests stationarity around a constant;
- `regression="ct"` tests stationarity around a deterministic trend.

In [ ]:
from statsmodels.tsa.stattools import kpss
import warnings

def run_kpss_test(
    series: pd.Series,
    regression: str,
    test_name: str
) -> None:
    """Run and display the KPSS test results."""

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        result = kpss(
            series.dropna(),
            regression=regression,
            nlags="auto"
        )

    kpss_statistic: float = result[0]
    p_value: float = result[1]
    lags_used: int = result[2]
    critical_values: dict = result[3]

    print(test_name)
    print("-" * 50)
    print(f"KPSS statistic: {kpss_statistic:.4f}")
    print(f"p-value: {p_value:.4f}")
    print(f"Lags used: {lags_used}")

    print("\nCritical values:")
    for level, value in critical_values.items():
        print(f"  {level}: {value:.4f}")

    alpha: float = 0.05

    if p_value < alpha:
        print("\nDecision: Reject H₀.")
        print("The series is not stationary under this specification.")
    else:
        print("\nDecision: Fail to reject H₀.")
        print("The test does not find sufficient evidence of non-stationarity.")

    print("\n")


# Test stationarity around a constant
run_kpss_test(
    series=train["no2_level"],
    regression="c",
    test_name="KPSS Test — Level Stationarity"
)

# Test stationarity around a deterministic trend
run_kpss_test(
    series=train["no2_level"],
    regression="ct",
    test_name="KPSS Test — Trend Stationarity"
)

### Interpretation of the KPSS Tests

Both KPSS specifications reject their null hypothesis at the 5% significance level.

For level stationarity, the KPSS statistic is 5.0308, which is greater than the 1% critical value of 0.7390. The p-value of 0.0100 therefore leads to the rejection of the hypothesis that the series is stationary around a constant mean.

For trend stationarity, the KPSS statistic is 0.3484, which also exceeds the 1% critical value of 0.2160. The p-value of 0.0100 indicates that the series is not stationary even around a deterministic linear trend.

The ADF and KPSS results are therefore mixed. The ADF test rejects the presence of a unit root, while both KPSS tests reject stationarity. This may result from annual seasonality, a non-linear trend, changing variance or structural changes that are not fully represented by a simple deterministic trend.

Combined with the rolling-statistics analysis, the KPSS results provide strong evidence that the original NO₂ series should not be modelled directly as a fully stationary series. Appropriate transformations, particularly seasonal and/or regular differencing, should therefore be evaluated.

### 11.4 Conclusion on the Original Series

The combined rolling-statistics, ADF and KPSS results indicate that the original NO₂ series should not be considered fully stationary.

Although the ADF test rejects the presence of a unit root, both KPSS specifications reject stationarity. The rolling mean and standard deviation also change over time.

Therefore, a transformation of the original series should be evaluated before modelling.

## 12. Differencing

### 12.1 First-Order Differencing

First-order differencing calculates the change in NO₂ concentration between two consecutive days:

\[
\Delta Y_t = Y_t - Y_{t-1}
\]

This transformation is applied to remove changes in level and reduce the long-term trend. The resulting series represents daily changes in NO₂ rather than the original concentration levels.

The transformed series will be examined visually and statistically to determine whether one regular difference is sufficient.

In [ ]:
# Calculate the first-order difference
first_difference: pd.Series = (
    train["no2_level"]
    .diff(periods=1)
    .dropna()
)

print("First-order differenced series:")
display(first_difference.head())

print(f"\nNumber of observations: {len(first_difference)}")
print(f"Mean daily change: {first_difference.mean():.4f} µg/m³")
print(f"Standard deviation: {first_difference.std():.4f} µg/m³")

In [ ]:
# Plot the first-order differenced series
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(
    first_difference.index,
    first_difference,
    color="steelblue",
    linewidth=0.7,
    alpha=0.8
)

ax.axhline(
    y=0,
    color="black",
    linestyle="--",
    linewidth=1
)

ax.set_title("First-Order Differenced Daily NO₂ Series")
ax.set_xlabel("Date")
ax.set_ylabel("Daily Change in NO₂ (µg/m³)")
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

### Interpretation of First-Order Differencing

After first-order differencing, the series fluctuates around zero and no clear long-term trend remains. This indicates that regular differencing has effectively stabilised the mean of the NO₂ series.

The average daily change is approximately −0.0064 µg/m³, which is very close to zero. Positive values represent increases relative to the previous day, while negative values represent decreases.

Daily changes remain variable, with a standard deviation of approximately 11.94 µg/m³ and several extreme positive and negative fluctuations. The variability also appears slightly lower toward the end of the period.

Overall, first-order differencing appears to remove the long-term trend, but statistical tests and additional diagnostics are required to determine whether the transformed series is stationary and whether annual seasonality remains.

### 12.2 Stationarity Tests After First-Order Differencing

The ADF and KPSS tests are now applied to the first-order differenced series.

The purpose is to determine whether regular differencing is sufficient to make the series stationary before considering seasonal differencing.

In [ ]:
# ADF test on the first-order differenced series
adf_first_diff = adfuller(
    first_difference,
    autolag="AIC"
)

print("ADF Test — First-Order Differenced Series")
print("-" * 50)
print(f"ADF statistic: {adf_first_diff[0]:.4f}")
print(f"p-value: {adf_first_diff[1]:.4f}")
print(f"Lags used: {adf_first_diff[2]}")
print(f"Observations used: {adf_first_diff[3]}")

print("\nCritical values:")
for level, value in adf_first_diff[4].items():
    print(f"  {level}: {value:.4f}")

if adf_first_diff[1] < 0.05:
    print("\nDecision: Reject H₀.")
    print("No statistical evidence of a unit root was found.")
else:
    print("\nDecision: Fail to reject H₀.")
    print("The test does not provide sufficient evidence against a unit root.")

### Interpretation of the ADF Test After First-Order Differencing

The ADF statistic for the first-order differenced series is −20.1447, which is substantially lower than all the critical values. The p-value is also below 0.05.

Therefore, the null hypothesis of a unit root is rejected. There is strong statistical evidence that the first-order differenced series does not contain a unit root.

Compared with the original series, whose ADF statistic was −4.8113, the much more negative statistic obtained after differencing provides stronger evidence against the presence of a unit root.

However, this result only addresses the presence of a unit root. The KPSS results and seasonal diagnostics must also be examined before concluding that first-order differencing is sufficient or selecting \(d=1\).

In [ ]:
# KPSS tests after first-order differencing
run_kpss_test(
    series=first_difference,
    regression="c",
    test_name="KPSS Test — First Difference, Level Stationarity"
)

run_kpss_test(
    series=first_difference,
    regression="ct",
    test_name="KPSS Test — First Difference, Trend Stationarity"
)

### Interpretation of Stationarity Tests After First-Order Differencing

The ADF and KPSS tests provide consistent evidence that the first-order differenced series is stationary.

The ADF test strongly rejects the null hypothesis of a unit root, with a statistic of −20.1447 and a p-value below 0.05.

For the KPSS level-stationarity test, the statistic is 0.0600, which is substantially below the 5% critical value of 0.4630. Similarly, the trend-stationarity statistic is 0.0469, below the corresponding 5% critical value of 0.1460. Both reported p-values are 0.1000, so the null hypotheses of stationarity cannot be rejected.

Therefore, first-order differencing successfully stabilises the mean of the series and produces a statistically stationary transformation according to both tests. This supports using one regular difference, corresponding to \(d=1\), if an integrated model is selected.

However, these results do not establish that the annual seasonal dependence has disappeared. The ACF of the differenced series must still be examined, particularly around lag 365, before deciding whether seasonal differencing is also required.

### 12.3 Seasonal Dependence After First-Order Differencing

Although first-order differencing produced a statistically stationary series, annual dependence may still remain.

The autocorrelation function (ACF) is examined up to lag 400, with particular attention to lag 365. A significant autocorrelation around this lag would suggest that daily NO₂ changes remain related to those observed approximately one year earlier.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import acf

# Plot the ACF up to 400 daily lags
fig, ax = plt.subplots(figsize=(16, 6))

plot_acf(
    first_difference,
    lags=400,
    alpha=0.05,
    fft=True,
    zero=False,
    ax=ax
)

# Highlight the annual lag
ax.axvline(
    x=365,
    color="darkred",
    linestyle="--",
    linewidth=2,
    label="Annual lag (365 days)"
)

ax.set_title("ACF of First-Order Differenced NO₂ Series")
ax.set_xlabel("Lag (days)")
ax.set_ylabel("Autocorrelation")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# Calculate ACF values up to lag 365
acf_values = acf(
    first_difference,
    nlags=365,
    fft=True
)

annual_acf: float = acf_values[365]

# Approximate 95% confidence limit
confidence_limit: float = 1.96 / np.sqrt(len(first_difference))

print(f"ACF at lag 365: {annual_acf:.4f}")
print(
    f"Approximate 95% confidence limits: "
    f"±{confidence_limit:.4f}"
)

if abs(annual_acf) > confidence_limit:
    print("The autocorrelation at lag 365 is statistically significant.")
else:
    print("The autocorrelation at lag 365 is not statistically significant.")

**Interpretation of Annual Dependence**

The autocorrelation at lag 365 is 0.0036, which lies well within the approximate 95% confidence limits of ±0.0284. Therefore, the annual autocorrelation is not statistically significant after first-order differencing.

This result indicates that no meaningful annual dependence remains at lag 365. Consequently, an additional annual seasonal difference is not supported by this diagnostic.

### 12.4 Selection of Differencing Orders

The stationarity analysis supports the following differencing orders:

\[
d=1,\qquad D=0
\]

One regular difference is required (d=1), while no seasonal difference is required (D=0.)

Annual seasonality may still be represented through seasonal model terms if later model comparisons and residual diagnostics show that these terms improve forecasting performance.

## 13. ACF and PACF Analysis

After selecting one regular difference (\(d=1\)) and no seasonal difference (\(D=0\)), the ACF and PACF of the first-order differenced series are examined to identify possible ARIMA parameters.

- The ACF helps identify possible moving-average terms and the order \(q\).
- The PACF helps identify possible autoregressive terms and the order \(p\).
- The first 60 lags are examined to focus on short-term dependencies.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Plot the ACF and PACF of the first-order differenced series
fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(16, 11)
)

plot_acf(
    first_difference,
    lags=60,
    alpha=0.05,
    zero=False,
    fft=True,
    ax=axes[0]
)

axes[0].set_title("ACF of First-Order Differenced NO₂ Series")
axes[0].set_xlabel("Lag (days)")
axes[0].set_ylabel("Autocorrelation")
axes[0].grid(alpha=0.3)

plot_pacf(
    first_difference,
    lags=60,
    alpha=0.05,
    zero=False,
    method="ywm",
    ax=axes[1]
)

axes[1].set_title("PACF of First-Order Differenced NO₂ Series")
axes[1].set_xlabel("Lag (days)")
axes[1].set_ylabel("Partial Autocorrelation")
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()

**Detailed Interpretation of the ACF and PACF**

The ACF and PACF are analysed by separating the lags into several zones. This helps distinguish short-term dependence from the recurring weekly pattern.

**Zone 1 — Lags 1 and 2: strong short-term correction**

The most prominent feature of both plots is the strong negative dependence at lags 1 and 2. The largest negative spike appears around lag 2.

A negative correlation means that a positive daily change in NO₂ concentration tends to be followed by a negative change one or two days later. Conversely, a sharp decrease may be followed by a partial increase.

This suggests a short-term correction or mean-reversion mechanism: unusually large daily movements are not generally maintained in the same direction over the following days.

The ACF and PACF are identical at lag 1 by construction because there are no intermediate lags whose effects need to be removed.

These first two lags provide evidence that short-term AR or MA terms may be required. In particular, moving-average orders \(q=1\) and \(q=2\) should be evaluated.

---

**Zone 2 — Lags 3 to 6: declining short-term dependence**

Between lags 3 and 6, the negative correlations become progressively smaller.

In the ACF, the correlations weaken considerably after the first two lags, although some bars may remain outside the confidence interval. Therefore, the ACF does not show a perfectly clean cut-off after lag 2.

In the PACF, several negative values remain significant and decrease more gradually. This means that part of the relationship with previous daily changes remains after controlling for the effects of the intermediate lags.

The contrast between a relatively rapid decline in the ACF and a more gradual decline in the PACF suggests a possible moving-average-dominated structure. However, because neither plot presents a perfectly clear theoretical cut-off, a mixed ARMA structure is also possible.

Consequently, small non-seasonal orders should be tested, such as:

- \(p=0,1,2\)
- \(q=1,2\)

The exact values of \(p\) and \(q\) cannot be selected reliably from visual inspection alone.

---

**Zone 3 — Lags 7, 14, 21, 28 and other multiples of 7: weekly pattern**

The ACF displays recurring positive spikes approximately at:

\[
7,\ 14,\ 21,\ 28,\ 35,\ 42,\ 49,\ 56
\]

These lags correspond to intervals of one, two, three and more weeks.

The regular repetition of the spikes indicates that daily NO₂ changes retain a weekly dependence structure. In practical terms, the variation recorded on a particular day tends to resemble, to some extent, the variation recorded on the same day of previous weeks.

This weekly pattern may be associated with:

- differences between weekdays and weekends;
- recurring road-traffic patterns;
- commuting and professional activity;
- weekly cycles in local emissions;
- recurring meteorological conditions.

The relevant seasonal period is therefore:

\[
m=7
\]

This is a weekly seasonal period and should not be confused with the annual lag of 365 previously examined.

The absence of significant autocorrelation at lag 365 supported \(D=0\) for annual seasonal differencing. It does not imply that every type of seasonality has disappeared. The ACF indicates that a shorter weekly dependence remains.

---

**Zone 4 — PACF at the weekly lags: no clear direct seasonal order**

The PACF does not show an equally clear and isolated spike at lag 7 or at all its multiples.

This suggests that the weekly correlations visible in the ACF may not represent a simple direct relationship between the current daily change and the change observed exactly seven days earlier.

Some of the weekly relationship may be transmitted through the intermediate daily lags or may originate from recurring day-of-week effects.

However, this difference between the ACF and PACF is not sufficient to conclude automatically that:

\[
P=0 \quad \text{and} \quad Q=1
\]

A theoretical seasonal MA(1) process would normally produce a strong seasonal ACF spike followed by a clearer cut-off across the seasonal lags. The repeated pattern visible here is more complex and may also reflect deterministic weekday effects.

Therefore, the plots establish the presence of weekly dependence, but they do not identify the seasonal AR and MA orders with certainty.

---

**Zone 5 — Remaining lags: weak but statistically detectable correlations**

Outside the first lags and the multiples of seven, most correlations are relatively close to zero. Nevertheless, some bars still cross the confidence limits.

Because the training series contains several thousand observations, the confidence interval is narrow. Consequently, even small correlations can be statistically significant.

Statistical significance does not necessarily mean that every lag is important for forecasting. The magnitude and practical relevance of the correlations must also be considered.

Adding one AR or MA parameter for every significant bar would produce an unnecessarily complex model and increase the risk of overfitting.

---

**Overall Interpretation**

The ACF and PACF reveal two main structures:

1. A strong negative short-term dependence during the first few days, indicating a correction or mean-reversion pattern.
2. A recurring weekly dependence at multiples of seven, indicating that the behaviour of NO₂ changes is partially associated with the day of the week.

The short-term pattern supports testing small non-seasonal AR and MA orders. The relatively rapid weakening of the ACF makes \(q=1\) and \(q=2\) reasonable candidates, while the gradual PACF behaviour suggests testing small AR orders such as \(p=1\) or \(p=2\).

The weekly peaks support evaluating models with a seasonal period of \(m=7\). Nevertheless, the graphs do not provide enough evidence to determine the seasonal orders \(P\) and \(Q\) definitively.

The ACF and PACF are therefore used to generate candidate models rather than select the final model. Model selection will be based on:

- AIC(Akaike Information Criterion) and BIC(Bayesian Information Criterion);
- forecast accuracy on the test set;
- residual autocorrelation;
- the Ljung–Box test;
- model simplicity and interpretability.

## 14. Autoregressive and Moving Average Models: AR and MA

The ACF and PACF analysis revealed short-term dependence in the first-order differenced NO₂ series. Before combining autoregressive and moving-average components within an ARIMA model, AR and MA models are first evaluated separately.

Because these models require a stationary series, they are fitted to the first-order differenced training series.

- An autoregressive model, AR(\(p\)), predicts the current value using previous values of the series.
- A moving-average model, MA(\(q\)), predicts the current value using current and previous forecast errors.
- The orders \(p\) and \(q\) represent the number of lags included in each model.

The ACF and PACF did not show perfectly clear cut-offs. Therefore, several small AR and MA orders are compared rather than selecting one specification from visual inspection alone.

### 14.1 Autoregressive Models — AR(p)

An autoregressive model assumes that the current value of the stationary series depends linearly on its previous values.

An AR(\(p\)) model can be written as:

\[
X_t = c + \phi_1X_{t-1} + \phi_2X_{t-2}
      + \cdots + \phi_pX_{t-p} + \varepsilon_t
\]

where:

- \(X_t\) is the current value of the differenced NO₂ series;
- \(c\) is a constant;
- \(\phi_1,\ldots,\phi_p\) are the autoregressive coefficients;
- \(p\) is the number of previous observations included;
- \(\varepsilon_t\) is a random error term.

Because the PACF does not show a clear cut-off, AR models with orders from 1 to 6 are evaluated.

In [ ]:
import warnings

import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

# Stationary first-order differenced training series
ar_ma_training_series: pd.Series = (
    train["no2_level"]
    .astype(float)
    .diff()
    .dropna()
)

ar_results: list[dict] = []
fitted_ar_models: dict[int, object] = {}

# Compare AR models from AR(1) to AR(6)
for p in range(1, 7):
    try:
        with warnings.catch_warnings(record=True) as captured_warnings:
            warnings.simplefilter("always")

            # AR(p) is equivalent to ARIMA(p, 0, 0)
            ar_model = ARIMA(
                ar_ma_training_series,
                order=(p, 0, 0),
                trend="c",
                enforce_stationarity=False
            )

            fitted_ar_model = ar_model.fit()

        fitted_ar_models[p] = fitted_ar_model

        ar_results.append({
            "Model": f"AR({p})",
            "Order": p,
            "AIC": fitted_ar_model.aic,
            "BIC": fitted_ar_model.bic,
            "Log-Likelihood": fitted_ar_model.llf,
            "Converged": fitted_ar_model.mle_retvals.get(
                "converged",
                None
            ),
            "Warnings": len(captured_warnings)
        })

    except Exception as error:
        ar_results.append({
            "Model": f"AR({p})",
            "Order": p,
            "AIC": None,
            "BIC": None,
            "Log-Likelihood": None,
            "Converged": False,
            "Warnings": None,
            "Error": str(error)
        })

ar_comparison: pd.DataFrame = (
    pd.DataFrame(ar_results)
    .sort_values(
        by=["AIC", "BIC"],
        na_position="last"
    )
    .reset_index(drop=True)
)

display(
    ar_comparison.style.format({
        "AIC": "{:.2f}",
        "BIC": "{:.2f}",
        "Log-Likelihood": "{:.2f}"
    })
)

### 14.2 Moving-Average Models — MA(q)

A moving-average model represents the current value using current and previous forecast errors.

An MA(\(q\)) model can be written as:

\[
X_t = \mu + \varepsilon_t
      + \theta_1\varepsilon_{t-1}
      + \theta_2\varepsilon_{t-2}
      + \cdots
      + \theta_q\varepsilon_{t-q}
\]

where:

- \(X_t\) is the current value of the differenced NO₂ series;
- \(\mu\) is the mean of the stationary series;
- \(\varepsilon_t\) is the current random error;
- \(\theta_1,\ldots,\theta_q\) are the moving-average coefficients;
- \(q\) is the number of previous errors included.

The strong negative ACF values at the first two lags suggest that MA(1) and MA(2) are important candidates. Additional small orders are also tested because the ACF does not present a perfectly clear cut-off.

In [ ]:
ma_results: list[dict] = []
fitted_ma_models: dict[int, object] = {}

# Compare MA models from MA(1) to MA(6)
for q in range(1, 7):
    try:
        with warnings.catch_warnings(record=True) as captured_warnings:
            warnings.simplefilter("always")

            # MA(q) is equivalent to ARIMA(0, 0, q)
            ma_model = ARIMA(
                ar_ma_training_series,
                order=(0, 0, q),
                trend="c",
                enforce_invertibility=False
            )

            fitted_ma_model = ma_model.fit()

        fitted_ma_models[q] = fitted_ma_model

        ma_results.append({
            "Model": f"MA({q})",
            "Order": q,
            "AIC": fitted_ma_model.aic,
            "BIC": fitted_ma_model.bic,
            "Log-Likelihood": fitted_ma_model.llf,
            "Converged": fitted_ma_model.mle_retvals.get(
                "converged",
                None
            ),
            "Warnings": len(captured_warnings)
        })

    except Exception as error:
        ma_results.append({
            "Model": f"MA({q})",
            "Order": q,
            "AIC": None,
            "BIC": None,
            "Log-Likelihood": None,
            "Converged": False,
            "Warnings": None,
            "Error": str(error)
        })

ma_comparison: pd.DataFrame = (
    pd.DataFrame(ma_results)
    .sort_values(
        by=["AIC", "BIC"],
        na_position="last"
    )
    .reset_index(drop=True)
)

display(
    ma_comparison.style.format({
        "AIC": "{:.2f}",
        "BIC": "{:.2f}",
        "Log-Likelihood": "{:.2f}"
    })
)

### 14.3 Comparison of AR and MA Models

All AR and MA models converged successfully without warnings, allowing their AIC and BIC values to be compared reliably.

**Autoregressive models**

Among the autoregressive models, AR(6) produced the lowest information criteria:

- AIC: 36,203.40
- BIC: 36,255.11

Therefore, AR(6) is the best AR candidate according to both AIC and BIC. The continuous decrease in these criteria from AR(1) to AR(6) suggests that several previous daily changes contribute to explaining the current change in NO₂.

However, because AR(6) is the highest order evaluated, the optimal autoregressive order may not have been fully identified. It is therefore considered the best AR model within the tested range, rather than the definitive AR specification.

**Moving-average models**

Within the MA family, AIC and BIC select slightly different orders:

- MA(5) has the lowest AIC: 35,972.59.
- MA(4) has the lowest BIC: 36,017.44.
- The BIC of MA(5) is 36,017.84, only 0.40 points higher than that of MA(4).

AIC favours MA(5) because its additional parameter slightly improves the model fit. BIC favours the simpler MA(4) because it applies a stronger penalty for model complexity.

The extremely small BIC difference indicates that MA(4) and MA(5) receive practically equivalent statistical support.

**Comparison between the AR and MA families**

The leading MA models have substantially lower AIC and BIC values than the leading AR model. For example:

- AR(6) AIC: 36,203.40
- MA(5) AIC: 35,972.59

The AIC difference is approximately 230.81 points in favour of MA(5). This suggests that the short-term dependence in the differenced NO₂ series is better represented by previous forecast errors than by a pure autoregressive structure.

This result is consistent with the earlier ACF and PACF analysis, which suggested a moving-average-dominated short-term structure.

Based on the information criteria:

- AR(6) is the leading AR candidate;
- MA(4) is the most parsimonious MA candidate;
- MA(5) is the best-fitting MA candidate.

However, AIC and BIC measure in-sample model quality and do not directly determine forecasting accuracy. Therefore, all fitted AR and MA candidates will now be evaluated on the unseen 2023 test set using MAE, RMSE and MAPE. Their residuals will then be examined in Section 14.4.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
# Combine all fitted AR and MA candidate models
all_ar_ma_models = {
    **{
        f"AR({order})": model
        for order, model in fitted_ar_models.items()
    },
    **{
        f"MA({order})": model
        for order, model in fitted_ma_models.items()
    }
}

# Actual NO₂ values during the test period
ar_ma_test_series = test["no2_level"].astype(float)

# Last observed value before the test period
last_training_value = train["no2_level"].iloc[-1]

ar_ma_forecasts = {}
ar_ma_forecast_results = []

for model_name, fitted_model in all_ar_ma_models.items():

    # Forecast daily changes
    forecasted_changes = fitted_model.forecast(
        steps=len(ar_ma_test_series)
    )

    # Convert predicted changes back to NO₂ levels
    forecasted_levels = (
        last_training_value
        + np.asarray(forecasted_changes).cumsum()
    )

    forecast = pd.Series(
        forecasted_levels,
        index=ar_ma_test_series.index,
        name=model_name
    )

    ar_ma_forecasts[model_name] = forecast

    # Calculate evaluation metrics
    mae = mean_absolute_error(
        ar_ma_test_series,
        forecast
    )

    rmse = np.sqrt(
        mean_squared_error(
            ar_ma_test_series,
            forecast
        )
    )

    non_zero_mask = ar_ma_test_series != 0

    mape = np.mean(
        np.abs(
            (
                ar_ma_test_series[non_zero_mask]
                - forecast[non_zero_mask]
            )
            / ar_ma_test_series[non_zero_mask]
        )
    ) * 100

    ar_ma_forecast_results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape
    })

# Comparison table
ar_ma_forecast_comparison = (
    pd.DataFrame(ar_ma_forecast_results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(
    ar_ma_forecast_comparison.style
    .format({
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "MAPE (%)": "{:.2f}"
    })
    .highlight_min(
        subset=["MAE", "RMSE", "MAPE (%)"],
        color="lightgreen"
    )
)

The out-of-sample evaluation shows that MA(2) achieved the lowest MAE (8.168) and RMSE (11.288), making it the most accurate AR–MA model on the 2023 test set.

AR(1) obtained the lowest MAPE (52.89%), but its MAE (10.190) and RMSE (14.881) were considerably higher. This disagreement occurs because MAPE is highly sensitive to actual NO₂ values close to zero. Therefore, MAE and RMSE are considered more reliable for the final comparison.

Interestingly, the models favoured by AIC and BIC, particularly AR(6), MA(4) and MA(5), did not provide the best out-of-sample forecasts. This confirms that better in-sample fit does not necessarily produce better forecasting accuracy.

Consequently, MA(2) is retained as the best AR–MA forecasting model, while AR(1) is noted for achieving the lowest percentage error.

### 14.4 Residual Diagnostics

A well-specified time-series model should leave residuals that behave approximately like white noise. This means that the remaining errors should fluctuate randomly around zero, show no systematic pattern and contain no meaningful autocorrelation.

The residuals of the following models are examined:

- AR(6), the leading autoregressive model according to AIC and BIC;
- MA(2), the best-performing AR–MA model on the unseen 2023 test set according to MAE and RMSE.

The diagnostics include:

- residual time-series plots;
- residual ACF plots;
- the Ljung–Box test.

The Ljung–Box test evaluates whether the residual autocorrelations are jointly significant.

- A p-value greater than 0.05 means that the null hypothesis of no residual autocorrelation cannot be rejected.
- A p-value less than or equal to 0.05 indicates that significant temporal dependence remains in the residuals.

These diagnostics help determine whether the models adequately captured the temporal structure of the differenced NO₂ series.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox

# Models selected for residual diagnostics
models_for_diagnostics = {
    "AR(6)": fitted_ar_models[6],
    "MA(2)": fitted_ma_models[2]
}

residual_diagnostics = []

for model_name, fitted_model in models_for_diagnostics.items():

    # Extract residuals and remove missing values
    residuals = fitted_model.resid.dropna()

    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(16, 8)
    )

    # Residuals over time
    axes[0].plot(
        residuals.index,
        residuals,
        color="steelblue",
        linewidth=0.8
    )

    axes[0].axhline(
        y=0,
        color="black",
        linestyle="--",
        linewidth=1
    )

    axes[0].set_title(f"Residuals of {model_name}")
    axes[0].set_xlabel("Date")
    axes[0].set_ylabel("Residual")
    axes[0].grid(alpha=0.3)

    # Residual autocorrelation
    plot_acf(
        residuals,
        lags=60,
        zero=False,
        alpha=0.05,
        fft=True,
        ax=axes[1]
    )

    axes[1].set_title(f"Residual ACF of {model_name}")
    axes[1].set_xlabel("Lag (days)")
    axes[1].set_ylabel("Autocorrelation")
    axes[1].grid(alpha=0.3)

    fig.tight_layout()
    plt.show()

    # Ljung–Box test
    ljung_box = acorr_ljungbox(
        residuals,
        lags=[7, 14, 30],
        return_df=True
    )

    for lag, row in ljung_box.iterrows():
        residual_diagnostics.append({
            "Model": model_name,
            "Lag": lag,
            "Ljung-Box Statistic": row["lb_stat"],
            "p-value": row["lb_pvalue"],
            "White Noise at 5%": row["lb_pvalue"] > 0.05
        })

# Create the diagnostic results table
residual_diagnostics_df = pd.DataFrame(
    residual_diagnostics
)

display(
    residual_diagnostics_df.style.format({
        "Ljung-Box Statistic": "{:.2f}",
        "p-value": "{:.4f}"
    })
)

The residuals of AR(6) and MA(2) fluctuate approximately around zero without a clear long-term trend. However, their residual ACF plots show several statistically significant autocorrelations.

The Ljung–Box test confirms this result. For both models, all p-values at lags 7, 14 and 30 are below 0.05. Therefore, the null hypothesis of no residual autocorrelation is rejected, and the residuals cannot be considered white noise.

Although the remaining autocorrelations appear relatively small in the ACF plots, they are statistically significant because of the large number of observations. MA(2) also produces larger Ljung–Box statistics than AR(6), indicating stronger remaining residual dependence.

Consequently, neither AR(6) nor MA(2) fully captures the temporal structure of the differenced NO₂ series. Nevertheless, MA(2) remains the best AR–MA forecasting model according to its lower MAE and RMSE on the unseen 2023 test set. More flexible ARIMA and seasonal models are therefore required.

## 15. ARIMA and SARIMA Models

The pure AR and MA models captured part of the short-term dependence in the differenced NO₂ series. However, none of them produced fully white-noise residuals. In particular, recurring residual autocorrelations around multiples of seven indicated that a weekly dependence structure remained unexplained.

This section evaluates two related model families:

- ARIMA models, which combine non-seasonal autoregressive, differencing and moving-average components;
- SARIMA models, which extend ARIMA by adding seasonal components capable of representing the remaining weekly dependence.

The analysis is conducted progressively. Non-seasonal ARIMA models are evaluated first. Weekly SARIMA models are then introduced to determine whether the seasonal extension improves residual behaviour and out-of-sample forecasting performance.

### 15.1 ARIMA Models

#### 15.1.a ARIMA Candidate Models

The stationarity analysis established that one regular difference is required. Therefore, the differencing order is fixed at:

\[
d=1
\]

The ACF and PACF did not identify a single unambiguous combination of autoregressive and moving-average orders. Moreover, the analysis of the pure AR and MA models showed that:

- the moving-average family performed better than the autoregressive family;
- MA(4) and MA(5) obtained the most favourable information criteria;
- a small autoregressive component may still improve the representation of the series.

Consequently, several ARIMA candidates are evaluated by combining:

\[
p \in \{0,1,2\}
\]

and:

\[
q \in \{1,2,3,4,5\}
\]

This produces 15 candidate specifications. The orders remain relatively small to limit unnecessary model complexity and reduce the risk of overfitting.

The original, non-differenced training series is used. Each ARIMA model performs the first-order differencing internally through \(d=1\).

The candidate models are compared using:

- Akaike Information Criterion (AIC);
- Bayesian Information Criterion (BIC);
- log-likelihood;
- convergence status;
- number of fitting warnings.

Lower AIC and BIC values indicate a better compromise between model fit and complexity. However, the final model will also be evaluated using residual diagnostics and out-of-sample forecast accuracy.

In [ ]:
import warnings

import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

# Use the original training series.
# ARIMA performs the first-order differencing internally because d=1.
arima_training_series: pd.Series = (
    train["no2_level"]
    .astype(float)
    .dropna()
)

# Define the candidate ARIMA orders
candidate_arima_orders: list[tuple[int, int, int]] = [
    (p, 1, q)
    for p in range(0, 3)
    for q in range(1, 6)
]

print(f"Number of candidate models: {len(candidate_arima_orders)}")
print(candidate_arima_orders)

In [ ]:
arima_results: list[dict] = []
fitted_arima_models: dict[tuple[int, int, int], object] = {}

for order in candidate_arima_orders:
    try:
        with warnings.catch_warnings(record=True) as captured_warnings:
            warnings.simplefilter("always")

            arima_model = ARIMA(
                arima_training_series,
                order=order,
                trend=None,
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            fitted_model = arima_model.fit()

        converged = fitted_model.mle_retvals.get(
            "converged",
            None
        )

        fitted_arima_models[order] = fitted_model

        arima_results.append({
            "Model": f"ARIMA{order}",
            "p": order[0],
            "d": order[1],
            "q": order[2],
            "AIC": fitted_model.aic,
            "BIC": fitted_model.bic,
            "Log-Likelihood": fitted_model.llf,
            "Converged": converged,
            "Warnings": len(captured_warnings),
            "Error": None
        })

    except Exception as error:
        arima_results.append({
            "Model": f"ARIMA{order}",
            "p": order[0],
            "d": order[1],
            "q": order[2],
            "AIC": None,
            "BIC": None,
            "Log-Likelihood": None,
            "Converged": False,
            "Warnings": None,
            "Error": str(error)
        })

arima_comparison: pd.DataFrame = (
    pd.DataFrame(arima_results)
    .sort_values(
        by=["AIC", "BIC"],
        ascending=True,
        na_position="last"
    )
    .reset_index(drop=True)
)

display(
    arima_comparison.style.format({
        "AIC": "{:.2f}",
        "BIC": "{:.2f}",
        "Log-Likelihood": "{:.2f}"
    })
)


All candidate ARIMA models converged without warnings or errors. ARIMA(2,1,5) achieved the lowest AIC, while ARIMA(0,1,5) achieved the lowest BIC.

Since their AIC difference is small (1.03), ARIMA(0,1,5) provides a similar fit with fewer parameters. Therefore, ARIMA(0,1,5) and ARIMA(2,1,5) are retained for residual diagnostics and forecast evaluation.

#### 15.1.b ARIMA Forecast Evaluation

ARIMA(0,1,5) and ARIMA(2,1,5) are evaluated on the unseen test period because they achieved the lowest BIC and AIC, respectively.

Each fitted model generates a multi-step forecast covering the complete test set. Forecast accuracy is measured using:

- Mean Absolute Error (MAE);
- Root Mean Squared Error (RMSE);
- Mean Absolute Percentage Error (MAPE).

Lower values indicate better forecasting performance. The predicted and observed NO₂ values are also plotted to visually compare their behaviour over the test period.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error

# Actual values from the test period
arima_test_series: pd.Series = (
    test["no2_level"]
    .astype(float)
    .dropna()
)

# Leading ARIMA candidates
arima_models_for_forecasting = {
    "ARIMA(0,1,5)": fitted_arima_models[(0, 1, 5)],
    "ARIMA(2,1,5)": fitted_arima_models[(2, 1, 5)]
}

arima_forecasts: dict[str, pd.Series] = {}
arima_forecast_results: list[dict] = []

for model_name, fitted_model in arima_models_for_forecasting.items():

    # Forecast the complete test period
    forecast = fitted_model.forecast(
        steps=len(arima_test_series)
    )

    # Assign the test dates to the predictions
    forecast = pd.Series(
        forecast.to_numpy(),
        index=arima_test_series.index,
        name=model_name
    )

    arima_forecasts[model_name] = forecast

    # Calculate evaluation metrics
    mae = mean_absolute_error(
        arima_test_series,
        forecast
    )

    rmse = np.sqrt(
        mean_squared_error(
            arima_test_series,
            forecast
        )
    )

    # Avoid division by zero when calculating MAPE
    non_zero_mask = arima_test_series != 0

    mape = np.mean(
        np.abs(
            (
                arima_test_series[non_zero_mask]
                - forecast[non_zero_mask]
            )
            / arima_test_series[non_zero_mask]
        )
    ) * 100

    arima_forecast_results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape
    })

# Create the comparison table
arima_forecast_comparison: pd.DataFrame = (
    pd.DataFrame(arima_forecast_results)
    .sort_values(by="RMSE")
    .reset_index(drop=True)
)

display(
    arima_forecast_comparison.style.format({
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "MAPE (%)": "{:.2f}"
    })
)

In [ ]:
plt.figure(figsize=(16, 6))

plt.plot(
    arima_test_series.index,
    arima_test_series,
    label="Actual NO₂",
    color="black",
    linewidth=1.5
)

for model_name, forecast in arima_forecasts.items():
    plt.plot(
        forecast.index,
        forecast,
        label=model_name,
        linewidth=1.2
    )

plt.title("ARIMA Forecasts versus Actual NO₂ Values")
plt.xlabel("Date")
plt.ylabel("Daily Mean NO₂ Level")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The two ARIMA models produced almost identical forecasts and quickly converged toward a constant NO₂ level of approximately 19. ARIMA(2,1,5) performed marginally better, with the lowest MAE (9.311), RMSE (11.658) and MAPE (100.06%).

However, both models failed to reproduce the strong daily fluctuations and seasonal changes observed in the test data. The very high MAPE also indicates poor predictive accuracy, although this metric is sensitive to the very low NO₂ values.

Overall, ARIMA(2,1,5) is the best ARIMA candidate, but its flat long-term forecast shows that a non-seasonal ARIMA model is insufficient. This supports evaluating SARIMA models with a weekly seasonal period.

### 15.2 SARIMA Model Specification

The ARIMA models produced nearly constant long-term forecasts and failed to reproduce the fluctuations observed in the test period. In addition, the earlier ACF analysis showed recurring correlations around multiples of seven days.

A Seasonal ARIMA model extends ARIMA by including seasonal components. It is written as:

\[
SARIMA(p,d,q)(P,D,Q)_s
\]

where:

- \(p,d,q\) represent the non-seasonal ARIMA parameters;
- \(P\) is the seasonal autoregressive order;
- \(D\) is the seasonal differencing order;
- \(Q\) is the seasonal moving-average order;
- \(s\) is the length of the seasonal cycle.

Because the data are recorded daily and the analysis revealed a weekly pattern, the seasonal period is fixed at:

\[
s=7
\]

The regular differencing order remains \(d=1\), based on the stationarity analysis. Small seasonal orders will be tested to determine whether weekly AR or MA components improve the model.

The candidate SARIMA models will be compared using AIC, BIC, convergence and out-of-sample forecast accuracy.

#### 15.2.a SARIMA Candidate Models

The non-seasonal ARIMA results showed that moving-average terms were important, with ARIMA(2,1,5) providing the best forecasting performance among the evaluated ARIMA candidates.

To assess the contribution of weekly seasonality without creating an excessively large search space, the leading non-seasonal structures are extended using small seasonal orders:

\[
P \in \{0,1\}, \qquad D \in \{0,1\}, \qquad Q \in \{0,1\}
\]

with a fixed seasonal period:

\[
s=7
\]

The configuration \((P,D,Q)=(0,0,0)\) is excluded because it contains no seasonal component and is equivalent to a non-seasonal ARIMA model.

The SARIMA candidates are compared using AIC, BIC, log-likelihood, convergence status and fitting warnings. Forecast accuracy will be evaluated separately on the test set.

In [ ]:
import warnings
import pandas as pd

from statsmodels.tsa.statespace.sarimax import SARIMAX

# Leading non-seasonal ARIMA structures
sarima_base_orders = [
    (0, 1, 5),
    (2, 1, 5)
]

# Small weekly seasonal structures
seasonal_orders = [
    (P, D, Q, 7)
    for P in range(0, 2)
    for D in range(0, 2)
    for Q in range(0, 2)
    if (P, D, Q) != (0, 0, 0)
]

candidate_sarima_orders = [
    (order, seasonal_order)
    for order in sarima_base_orders
    for seasonal_order in seasonal_orders
]

print(
    f"Number of candidate SARIMA models: "
    f"{len(candidate_sarima_orders)}"
)

In [ ]:
sarima_results = []
fitted_sarima_models = {}

for order, seasonal_order in candidate_sarima_orders:
    try:
        with warnings.catch_warnings(record=True) as captured_warnings:
            warnings.simplefilter("always")

            sarima_model = SARIMAX(
                arima_training_series,
                order=order,
                seasonal_order=seasonal_order,
                trend=None,
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            fitted_model = sarima_model.fit(
                disp=False,
                maxiter=200
            )

        converged = fitted_model.mle_retvals.get(
            "converged",
            None
        )

        fitted_sarima_models[
            (order, seasonal_order)
        ] = fitted_model

        sarima_results.append({
            "Model": (
                f"SARIMA{order}"
                f"{seasonal_order}"
            ),
            "Order": order,
            "Seasonal Order": seasonal_order,
            "AIC": fitted_model.aic,
            "BIC": fitted_model.bic,
            "Log-Likelihood": fitted_model.llf,
            "Converged": converged,
            "Warnings": len(captured_warnings),
            "Error": None
        })

    except Exception as error:
        sarima_results.append({
            "Model": (
                f"SARIMA{order}"
                f"{seasonal_order}"
            ),
            "Order": order,
            "Seasonal Order": seasonal_order,
            "AIC": None,
            "BIC": None,
            "Log-Likelihood": None,
            "Converged": False,
            "Warnings": None,
            "Error": str(error)
        })

sarima_comparison = (
    pd.DataFrame(sarima_results)
    .sort_values(
        by=["AIC", "BIC"],
        ascending=True,
        na_position="last"
    )
    .reset_index(drop=True)
)

display(
    sarima_comparison.style.format({
        "AIC": "{:.2f}",
        "BIC": "{:.2f}",
        "Log-Likelihood": "{:.2f}"
    })
)

All 14 SARIMA models converged without warnings or errors. SARIMA(0,1,5)(0,1,1,7) achieved both the lowest AIC (35,604.45) and the lowest BIC (35,649.68), making it the leading candidate. The results also show that models with seasonal differencing (\(D=1\)) generally performed better, confirming the importance of the weekly seasonal structure.

#### 15.2.b SARIMA Forecast Evaluation

The leading SARIMA model, SARIMA(0,1,5)(0,1,1,7), is evaluated on the unseen test period. Its forecasting performance is measured using MAE, RMSE and MAPE, then compared with the actual daily NO₂ values.

In [ ]:
# Select the leading SARIMA model
best_sarima_order = (0, 1, 5)
best_seasonal_order = (0, 1, 1, 7)

best_sarima_model = fitted_sarima_models[
    (best_sarima_order, best_seasonal_order)
]

# Generate predictions for the complete test period
sarima_forecast = best_sarima_model.forecast(
    steps=len(arima_test_series)
)

# Assign the test dates to the forecast
sarima_forecast = pd.Series(
    sarima_forecast.to_numpy(),
    index=arima_test_series.index,
    name="SARIMA(0,1,5)(0,1,1,7)"
)

# Calculate evaluation metrics
sarima_mae = mean_absolute_error(
    arima_test_series,
    sarima_forecast
)

sarima_rmse = np.sqrt(
    mean_squared_error(
        arima_test_series,
        sarima_forecast
    )
)

# Exclude zero values from MAPE
non_zero_mask = arima_test_series != 0

sarima_mape = np.mean(
    np.abs(
        (
            arima_test_series[non_zero_mask]
            - sarima_forecast[non_zero_mask]
        )
        / arima_test_series[non_zero_mask]
    )
) * 100

sarima_forecast_evaluation = pd.DataFrame({
    "Model": ["SARIMA(0,1,5)(0,1,1,7)"],
    "MAE": [sarima_mae],
    "RMSE": [sarima_rmse],
    "MAPE (%)": [sarima_mape]
})

display(
    sarima_forecast_evaluation.style.format({
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "MAPE (%)": "{:.2f}"
    })
)

In [ ]:
plt.figure(figsize=(16, 6))

plt.plot(
    arima_test_series.index,
    arima_test_series,
    label="Actual NO₂",
    color="black",
    linewidth=1.5
)

plt.plot(
    sarima_forecast.index,
    sarima_forecast,
    label="SARIMA(0,1,5)(0,1,1,7)",
    color="darkorange",
    linewidth=1.2
)

plt.title("SARIMA Forecast versus Actual NO₂ Values")
plt.xlabel("Date")
plt.ylabel("Daily Mean NO₂ Level")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

SARIMA(0,1,5)(0,1,1,7) achieved an MAE of 8.889, an RMSE of 11.381 and a MAPE of 89.29%, slightly outperforming the best ARIMA model. It captures a regular weekly pattern, but its forecasts remain too smooth and fail to reproduce the large daily NO₂ fluctuations. Therefore, SARIMA improves the forecast, but its overall predictive accuracy remains limited.

### 15.3 Final ARIMA–SARIMA Comparison

The best non-seasonal ARIMA model and the leading weekly SARIMA model are compared using their forecasting performance on the same unseen test period.

In [ ]:
final_arima_sarima_comparison = pd.DataFrame({
    "Model": [
        "ARIMA(2,1,5)",
        "SARIMA(0,1,5)(0,1,1,7)"
    ],
    "MAE": [
        arima_forecast_comparison.loc[
            arima_forecast_comparison["Model"] == "ARIMA(2,1,5)",
            "MAE"
        ].iloc[0],
        sarima_mae
    ],
    "RMSE": [
        arima_forecast_comparison.loc[
            arima_forecast_comparison["Model"] == "ARIMA(2,1,5)",
            "RMSE"
        ].iloc[0],
        sarima_rmse
    ],
    "MAPE (%)": [
        arima_forecast_comparison.loc[
            arima_forecast_comparison["Model"] == "ARIMA(2,1,5)",
            "MAPE (%)"
        ].iloc[0],
        sarima_mape
    ]
}).sort_values("RMSE").reset_index(drop=True)

display(
    final_arima_sarima_comparison.style
    .format({
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "MAPE (%)": "{:.2f}"
    })
    .highlight_min(
        subset=["MAE", "RMSE", "MAPE (%)"],
        color="lightgreen"
    )
)

SARIMA(0,1,5)(0,1,1,7) outperformed ARIMA(2,1,5) across all three forecasting metrics. Its MAE decreased from 9.311 to 8.889, its RMSE from 11.658 to 11.381, and its MAPE from 100.06% to 89.29%.

The improvement confirms that including the weekly seasonal structure provides additional forecasting value. However, the gain remains modest, and both models produce overly smooth forecasts that fail to reproduce the strongest daily NO₂ fluctuations.

Therefore, SARIMA(0,1,5)(0,1,1,7) is retained as the best classical statistical forecasting model. Its residuals will be examined in the next section to determine whether significant temporal dependence remains unexplained.

## 16. Residual Diagnostics

SARIMA(0,1,5)(0,1,1,7) achieved the best forecasting performance among the classical statistical models. Residual diagnostics are now performed to determine whether the model has captured the temporal structure of the training series adequately.

A well-specified forecasting model should produce residuals that:

- fluctuate randomly around zero;
- have approximately constant variance;
- contain no significant remaining autocorrelation;
- resemble white noise.

The residuals are examined using a time-series plot, a distribution plot, an ACF plot and the Ljung–Box test.

### 16.1 SARIMA Residual Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.graphics.tsaplots import plot_acf

# Extract residuals from the best SARIMA model
sarima_residuals = best_sarima_model.resid.dropna()

fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(16, 12)
)

# Residuals over time
axes[0].plot(
    sarima_residuals.index,
    sarima_residuals,
    color="steelblue",
    linewidth=0.8
)

axes[0].axhline(
    y=0,
    color="black",
    linestyle="--",
    linewidth=1
)

axes[0].set_title(
    "Residuals of SARIMA(0,1,5)(0,1,1,7)"
)
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Residual")
axes[0].grid(alpha=0.3)

# Residual distribution
sns.histplot(
    sarima_residuals,
    bins=40,
    kde=True,
    color="steelblue",
    ax=axes[1]
)

axes[1].axvline(
    x=0,
    color="black",
    linestyle="--",
    linewidth=1
)

axes[1].set_title("Distribution of SARIMA Residuals")
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Frequency")
axes[1].grid(alpha=0.3)

# Residual ACF
plot_acf(
    sarima_residuals,
    lags=60,
    zero=False,
    alpha=0.05,
    fft=True,
    ax=axes[2]
)

axes[2].set_title("ACF of SARIMA Residuals")
axes[2].set_xlabel("Lag (days)")
axes[2].set_ylabel("Autocorrelation")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

The SARIMA residuals fluctuate around zero without a clear trend or recurring pattern, although several large errors and outliers remain. Their distribution is slightly right-skewed and is therefore not perfectly normal. However, the residual autocorrelations are very small and remain mostly within the confidence interval, suggesting that the model has captured most of the temporal dependence. The Ljung–Box test will be used to confirm whether the remaining residual autocorrelation is statistically significant.

### 16.2 Ljung–Box Test

The Ljung–Box test is used to determine whether the residual autocorrelations are jointly significant.

The hypotheses are:

- \(H_0\): the residuals are independently distributed and behave like white noise;
- \(H_1\): significant autocorrelation remains in the residuals.

A p-value greater than 0.05 means that the null hypothesis cannot be rejected.

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

# Test residual autocorrelation at several time horizons
ljung_box_results = acorr_ljungbox(
    sarima_residuals,
    lags=[7, 14, 30],
    return_df=True
)

# Improve the presentation of the results
ljung_box_results = (
    ljung_box_results
    .rename(columns={
        "lb_stat": "Ljung-Box Statistic",
        "lb_pvalue": "p-value"
    })
    .reset_index()
    .rename(columns={"index": "Lag"})
)

ljung_box_results["White Noise at 5%"] = (
    ljung_box_results["p-value"] > 0.05
)

display(
    ljung_box_results.style.format({
        "Ljung-Box Statistic": "{:.3f}",
        "p-value": "{:.4f}"
    })
)

At lags 7, 14 and 30, all p-values are greater than 0.05. Therefore, the null hypothesis cannot be rejected: no significant autocorrelation remains in the residuals. The residuals behave like white noise, indicating that the SARIMA model successfully captured the main temporal structure of the training series.

## 17. Model Evaluation Using MAE, RMSE and MAPE

All forecasting models are evaluated on the same unseen 2023 test set using three complementary metrics:

- **MAE** measures the average absolute difference between the actual and predicted NO₂ values.
- **RMSE** gives more importance to large forecasting errors.
- **MAPE** expresses the average error as a percentage.

Lower values indicate better forecasting performance. However, MAPE can become very high when actual NO₂ values are close to zero. Therefore, MAE and RMSE are given greater importance when selecting the best model.

## 18. Comparison of Classical Models on the Test Set

The forecasting performance of all classical models is compared on the same unseen 2023 test set. The comparison includes the baseline models and the best-performing models from the AR, MA, ARIMA and SARIMA families.

MAE and RMSE are given greater importance in the final selection because MAPE is highly sensitive to actual NO₂ values close to zero. Residual diagnostics and time-based cross-validation will also be considered before selecting the final classical model.

In [ ]:
# Actual values from the 2023 test set
actual_test_values = baseline_forecasts["Actual"].astype(float)

# ---------------------------------------------------------
# 1. Evaluate all baseline models
# ---------------------------------------------------------

baseline_models = [
    "Mean",
    "Naive",
    "Seasonal_Naive",
    "Drift"
]

baseline_evaluation_results = []

for model_name in baseline_models:

    forecast = baseline_forecasts[model_name].astype(float)

    mae = mean_absolute_error(
        actual_test_values,
        forecast
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual_test_values,
            forecast
        )
    )

    non_zero_mask = actual_test_values != 0

    mape = np.mean(
        np.abs(
            (
                actual_test_values[non_zero_mask]
                - forecast[non_zero_mask]
            )
            / actual_test_values[non_zero_mask]
        )
    ) * 100

    baseline_evaluation_results.append({
        "Family": "Baseline",
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape
    })

baseline_evaluation = pd.DataFrame(
    baseline_evaluation_results
)

# ---------------------------------------------------------
# 2. Select the best AR and MA models according to RMSE
# ---------------------------------------------------------

best_ar_result = (
    ar_ma_forecast_comparison[
        ar_ma_forecast_comparison["Model"].str.startswith("AR(")
    ]
    .nsmallest(1, "RMSE")
    .copy()
)

best_ar_result.insert(
    0,
    "Family",
    "AR"
)

best_ma_result = (
    ar_ma_forecast_comparison[
        ar_ma_forecast_comparison["Model"].str.startswith("MA(")
    ]
    .nsmallest(1, "RMSE")
    .copy()
)

best_ma_result.insert(
    0,
    "Family",
    "MA"
)

# ---------------------------------------------------------
# 3. Select the best ARIMA model according to RMSE
# ---------------------------------------------------------

best_arima_result = (
    arima_forecast_comparison
    .nsmallest(1, "RMSE")
    .copy()
)

best_arima_result.insert(
    0,
    "Family",
    "ARIMA"
)

# ---------------------------------------------------------
# 4. Add the selected SARIMA model
# ---------------------------------------------------------

best_sarima_result = (
    sarima_forecast_evaluation
    .copy()
)

best_sarima_result.insert(
    0,
    "Family",
    "SARIMA"
)

# ---------------------------------------------------------
# 5. Final comparison of classical models
# ---------------------------------------------------------

classical_models_comparison = pd.concat(
    [
        baseline_evaluation,
        best_ar_result,
        best_ma_result,
        best_arima_result,
        best_sarima_result
    ],
    ignore_index=True
)

classical_models_comparison = (
    classical_models_comparison
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(
    classical_models_comparison.style
    .format({
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "MAPE (%)": "{:.2f}"
    })
    .highlight_min(
        subset=["MAE", "RMSE", "MAPE (%)"],
        color="lightgreen"
    )
)

The comparison shows that MA(2) achieved the lowest MAE (8.168) and RMSE (11.288), making it the most accurate classical model on the unseen 2023 test set.

SARIMA(0,1,5)(0,1,1,7) ranked second, with an MAE of 8.889 and an RMSE of 11.381. Its RMSE is only slightly higher than that of MA(2), indicating very similar forecasting performance.

ARIMA(2,1,5) ranked third, while all baseline models produced higher MAE and RMSE values. This confirms that the fitted statistical models generally improved upon the simple forecasting approaches.

The Naive baseline achieved the lowest MAPE (50.10%). However, its MAE and RMSE were considerably higher than those of MA(2) and SARIMA. This disagreement occurs because MAPE is highly sensitive to actual NO₂ values close to zero. Therefore, MAE and RMSE are given greater importance.

Although MA(2) provides the best test-set accuracy, its residual diagnostics revealed significant remaining autocorrelation. SARIMA produced slightly higher test errors but left residuals that behaved like white noise. Time-based cross-validation is therefore required before selecting the final classical model.

## 19. Time-Based Cross-Validation

The 2023 test-set comparison identified MA(2) and SARIMA(0,1,5)(0,1,1,7) as the two leading classical models.

However, performance on a single test period may not fully represent a model's forecasting ability. Time-based cross-validation is therefore used to evaluate both models across three chronological validation periods.

An expanding training window is applied. At each fold, the models are trained only on past observations and evaluated on the following 365 unseen days. This preserves the temporal order and prevents data leakage.

MAE and RMSE are given greater importance than MAPE when comparing the models.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.model_selection import TimeSeriesSplit

# Complete training series used for cross-validation
cv_series = train["no2_level"].astype(float).copy()

# Three validation periods of 365 days
time_series_cv = TimeSeriesSplit(
    n_splits=3,
    test_size=365
)

cross_validation_results = []

for fold, (train_indices, validation_indices) in enumerate(
    time_series_cv.split(cv_series),
    start=1
):
    fold_train = cv_series.iloc[train_indices]
    fold_validation = cv_series.iloc[validation_indices]

    # -----------------------------------------------------
    # MA(2): fitted to the differenced training series
    # -----------------------------------------------------

    fold_train_differenced = fold_train.diff().dropna()

    ma_model = ARIMA(
        fold_train_differenced,
        order=(0, 0, 2),
        trend="n"
    )

    fitted_ma_model = ma_model.fit()

    # Forecast changes and reconstruct NO₂ levels
    ma_forecasted_changes = fitted_ma_model.forecast(
        steps=len(fold_validation)
    )

    ma_forecast = pd.Series(
        fold_train.iloc[-1]
        + np.asarray(ma_forecasted_changes).cumsum(),
        index=fold_validation.index
    )

    # -----------------------------------------------------
    # SARIMA(0,1,5)(0,1,1,7)
    # -----------------------------------------------------

    sarima_model = SARIMAX(
        fold_train,
        order=(0, 1, 5),
        seasonal_order=(0, 1, 1, 7),
        trend=None,
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    fitted_sarima_model = sarima_model.fit(
        disp=False,
        maxiter=200
    )

    sarima_forecast = fitted_sarima_model.forecast(
        steps=len(fold_validation)
    )

    sarima_forecast = pd.Series(
        np.asarray(sarima_forecast),
        index=fold_validation.index
    )

    # -----------------------------------------------------
    # Evaluate both models
    # -----------------------------------------------------

    fold_forecasts = {
        "MA(2)": ma_forecast,
        "SARIMA(0,1,5)(0,1,1,7)": sarima_forecast
    }

    for model_name, forecast in fold_forecasts.items():

        mae = mean_absolute_error(
            fold_validation,
            forecast
        )

        rmse = np.sqrt(
            mean_squared_error(
                fold_validation,
                forecast
            )
        )

        non_zero_mask = fold_validation != 0

        mape = np.mean(
            np.abs(
                (
                    fold_validation[non_zero_mask]
                    - forecast[non_zero_mask]
                )
                / fold_validation[non_zero_mask]
            )
        ) * 100

        cross_validation_results.append({
            "Fold": fold,
            "Model": model_name,
            "Validation Start": fold_validation.index.min(),
            "Validation End": fold_validation.index.max(),
            "MAE": mae,
            "RMSE": rmse,
            "MAPE (%)": mape
        })

cross_validation_results_df = pd.DataFrame(
    cross_validation_results
)

display(
    cross_validation_results_df.style.format({
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "MAPE (%)": "{:.2f}"
    })
)



In [ ]:
# Average cross-validation performance for each model
cross_validation_summary = (
    cross_validation_results_df
    .groupby("Model")[["MAE", "RMSE", "MAPE (%)"]]
    .agg(["mean", "std"])
)

display(
    cross_validation_summary.style.format("{:.3f}")
)

Across the three validation folds, SARIMA(0,1,5)(0,1,1,7) achieved slightly better average performance than MA(2).

SARIMA obtained a mean MAE of 10.957 and a mean RMSE of 12.792, compared with 10.997 and 12.904 for MA(2). It also achieved a lower mean MAPE of 94.281%, compared with 96.390% for MA(2).

The differences remain relatively small, indicating that both models provide comparable forecasting accuracy. However, SARIMA also produced lower standard deviations for all three metrics, showing slightly more stable performance across the validation periods.

MAPE remains very high because some actual NO₂ values are close to zero. Therefore, MAE and RMSE are considered more reliable for the final selection.

Overall, SARIMA demonstrates a small but consistent advantage in both accuracy and stability. Combined with its satisfactory residual diagnostics, these results support selecting SARIMA(0,1,5)(0,1,1,7) as the best classical forecasting model.

## 20. Selection of the Best Classical Model

The final classical model is selected by considering three complementary elements:

- forecasting performance on the unseen 2023 test set;
- performance and stability across the time-based cross-validation folds;
- residual diagnostics.

MA(2) achieved the lowest MAE and RMSE on the 2023 test set. However, its residuals contained significant autocorrelation, indicating that the model did not fully capture the temporal structure of the series.

SARIMA(0,1,5)(0,1,1,7) ranked second on the test set, with performance very close to MA(2). During time-based cross-validation, SARIMA achieved slightly lower average MAE, RMSE and MAPE, as well as lower variability across the three folds.

Unlike MA(2), the SARIMA model also produced satisfactory residual diagnostics, indicating that it captured the temporal dependence more adequately.

Therefore, SARIMA(0,1,5)(0,1,1,7) is selected as the best classical forecasting model. It provides the strongest overall balance between forecasting accuracy, stability across different periods and residual quality.

The SARIMA forecast will later be compared with Prophet and HistGradientBoostingRegressor to determine the best overall model for daily NO₂ forecasting.

## 21. Prophet Model

Prophet is now used as an alternative forecasting approach for the daily NO₂ series.

Unlike ARIMA and SARIMA, Prophet models a time series through separate components such as trend and seasonality. This makes it particularly useful for capturing recurring patterns and gradual changes over time.

The model is trained on the same training period from 2010 to 2022 and evaluated on the unseen 2023 test set. Its forecasts are assessed using MAE, RMSE and MAPE to ensure a fair comparison with the classical models.

### 21.1 Preparing Data for Prophet

Prophet requires the time series to follow a specific tabular structure:

- `ds`: the date column;
- `y`: the numerical variable to forecast.

The daily NO₂ training and test series are therefore converted into this format. The same chronological split used for the previous models is preserved: data from 2010 to 2022 are used for training, while 2023 remains completely unseen for evaluation.

Unlike ARIMA-based models, Prophet can be trained directly on the original NO₂ levels. First-order or seasonal differencing is therefore not required.

In [ ]:
from prophet import Prophet

# Prepare training data for Prophet
prophet_train: pd.DataFrame = train[["no2_level"]].reset_index()
prophet_train.columns = ["ds", "y"]

# Prepare test data for Prophet
prophet_test: pd.DataFrame = test[["no2_level"]].reset_index()
prophet_test.columns = ["ds", "y"]

# Convert columns to the correct types
prophet_train["ds"] = pd.to_datetime(prophet_train["ds"])
prophet_test["ds"] = pd.to_datetime(prophet_test["ds"])

prophet_train["y"] = prophet_train["y"].astype(float)
prophet_test["y"] = prophet_test["y"].astype(float)

# Check the periods
print(
    "Training period:",
    prophet_train["ds"].min().date(),
    "to",
    prophet_train["ds"].max().date()
)

print(
    "Testing period:",
    prophet_test["ds"].min().date(),
    "to",
    prophet_test["ds"].max().date()
)

display(prophet_train.head())
display(prophet_test.head())

### 21.2 Baseline Prophet Model

A baseline Prophet model is trained on the daily NO₂ observations from 2010 to 2022.

Weekly and yearly seasonalities are included to capture recurring patterns across the week and throughout the year. Daily seasonality is disabled because the dataset contains one aggregated observation per day.

No holidays, lockdown periods or external variables are included at this stage. This model will serve as a reference for evaluating future Prophet improvements.

In [ ]:
# Create the baseline Prophet model
prophet_baseline_model: Prophet = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    interval_width=0.95
)

# Fit the model using only the 2010–2022 training data
prophet_baseline_model.fit(prophet_train)

# Prophet only requires the dates for which predictions are needed
prophet_test_dates: pd.DataFrame = prophet_test[["ds"]].copy()

# Generate predictions for the unseen 2023 test period
prophet_baseline_forecast: pd.DataFrame = (
    prophet_baseline_model.predict(prophet_test_dates)
)

# Keep the main forecast columns
prophet_baseline_results: pd.DataFrame = (
    prophet_baseline_forecast[
        ["ds", "yhat", "yhat_lower", "yhat_upper"]
    ]
    .copy()
)

# Add the actual NO₂ values to facilitate comparison
prophet_baseline_results["Actual"] = (
    prophet_test["y"].to_numpy()
)

# Rename the forecast columns for readability
prophet_baseline_results = (
    prophet_baseline_results.rename(
        columns={
            "yhat": "Predicted",
            "yhat_lower": "Lower Bound",
            "yhat_upper": "Upper Bound"
        }
    )
)

display(prophet_baseline_results.head())

### 21.3 Baseline Prophet Evaluation

The baseline Prophet model is evaluated on the unseen 2023 observations using MAE, RMSE and MAPE.

MAE and RMSE measure the forecast errors in the original NO₂ unit. MAPE expresses the average error as a percentage, but it must be interpreted cautiously because very low actual NO₂ values can produce extremely large percentage errors.

In [ ]:
# Extract actual and predicted NO₂ values
prophet_y_true: pd.Series = prophet_baseline_results["Actual"]
prophet_y_pred: pd.Series = prophet_baseline_results["Predicted"]

# Calculate MAE
prophet_baseline_mae: float = mean_absolute_error(
    prophet_y_true,
    prophet_y_pred
)

# Calculate RMSE
prophet_baseline_rmse: float = np.sqrt(
    mean_squared_error(
        prophet_y_true,
        prophet_y_pred
    )
)

# Exclude zero actual values before calculating MAPE
prophet_non_zero_mask: pd.Series = prophet_y_true != 0

prophet_baseline_mape: float = np.mean(
    np.abs(
        (
            prophet_y_true[prophet_non_zero_mask]
            - prophet_y_pred[prophet_non_zero_mask]
        )
        / prophet_y_true[prophet_non_zero_mask]
    )
) * 100

# Store the evaluation metrics
prophet_baseline_evaluation: pd.DataFrame = pd.DataFrame({
    "Model": ["Baseline Prophet"],
    "MAE": [prophet_baseline_mae],
    "RMSE": [prophet_baseline_rmse],
    "MAPE (%)": [prophet_baseline_mape]
})

display(
    prophet_baseline_evaluation.style.format({
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "MAPE (%)": "{:.2f}"
    })
)

The baseline Prophet model achieved an MAE of 7.691 and an RMSE of 10.498 on the unseen 2023 test period.

This means that its daily NO₂ forecasts differ from the observed values by approximately 7.7 units on average. The higher RMSE indicates that some days contain relatively large forecasting errors.

The MAPE is 58.42%, which appears high. However, this metric is strongly affected by days when the actual NO₂ concentration is close to zero. Therefore, MAE and RMSE remain the primary evaluation metrics.

These results establish the reference performance for Prophet. The following steps will examine the forecast visually and determine whether adding UK holidays, exceptional events or tuning the model can improve its performance.

### 21.4 Visualisation of the Baseline Prophet Forecast

The baseline Prophet forecasts are plotted against the actual daily NO₂ observations for 2023.

The shaded region represents Prophet's uncertainty interval. This visual comparison helps determine whether the model captures the general evolution and seasonal variations of NO₂, as well as the periods during which forecast errors are larger.

In [ ]:
# Plot actual and predicted NO₂ values
plt.figure(figsize=(15, 6))

plt.plot(
    prophet_baseline_results["ds"],
    prophet_baseline_results["Actual"],
    label="Actual NO₂",
    color="steelblue",
    linewidth=1.5
)

plt.plot(
    prophet_baseline_results["ds"],
    prophet_baseline_results["Predicted"],
    label="Baseline Prophet",
    color="darkorange",
    linewidth=2
)

# Add Prophet's uncertainty interval
plt.fill_between(
    prophet_baseline_results["ds"],
    prophet_baseline_results["Lower Bound"],
    prophet_baseline_results["Upper Bound"],
    color="orange",
    alpha=0.2,
    label="Uncertainty Interval"
)

plt.title("Actual vs Baseline Prophet Forecast — 2023")
plt.xlabel("Date")
plt.ylabel("Daily Mean NO₂ Level")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The baseline Prophet model captures the broad seasonal evolution of daily NO₂ levels in 2023. It predicts higher concentrations during winter, lower concentrations during summer and a gradual increase during autumn.

The regular oscillations in the forecast reflect the weekly seasonality learned by Prophet. However, the predicted curve is considerably smoother than the actual series. The model therefore does not reproduce sudden pollution peaks, particularly those observed at the beginning of the year and during autumn and winter.

The uncertainty interval is very wide, and most observed values appear to fall within it. Nevertheless, some exceptionally high NO₂ peaks exceed the upper bound. The lower bound also becomes negative, even though negative NO₂ concentrations are not physically possible. This occurs because the standard Prophet model does not impose a non-negative constraint on its uncertainty interval.

Overall, the baseline model captures the general annual and weekly patterns but struggles with short-term fluctuations and extreme pollution events. The next analysis will examine Prophet's individual trend and seasonality components.

### 21.5 Prophet Trend and Seasonality Components

Prophet decomposes its forecast into several components. The following plots display the long-term trend, weekly seasonality and yearly seasonality learned from the training data.

This decomposition helps explain how each temporal pattern contributes to the final NO₂ forecast.

In [ ]:
# Plot the components learned by the baseline Prophet model
prophet_components_figure = (
    prophet_baseline_model.plot_components(
        prophet_baseline_forecast
    )
)

plt.tight_layout()
plt.show()

The Prophet component plots reveal three important temporal patterns in the daily NO₂ series.

The trend component decreases throughout 2023, from approximately 14.3 to 11.0. This reflects the long-term decline in NO₂ levels learned by Prophet from the training data.

The weekly component shows lower NO₂ levels during the weekend, particularly on Sunday. In contrast, the positive effects observed from Tuesday to Friday suggest higher concentrations during working days, potentially because of increased road traffic and human activity.

The yearly component shows a strong seasonal pattern. NO₂ levels are generally higher during winter, particularly in January and from November to December. They gradually decrease during spring and reach their lowest level in summer, around July and August, before increasing again during autumn.

This annual component also allows us to observe the evolution across the months without aggregating the original daily data. The winter increase may be associated with greater heating emissions, less atmospheric dispersion and seasonal traffic patterns, while summer conditions generally favour pollutant dispersion.

These component values represent additive effects contributing to the final forecast; they are not the actual daily NO₂ concentrations.

## 22 Prophet with UK Public Holidays

### 22.1 Creating the Prophet Model with UK Public Holidays

Public holidays may influence daily NO₂ concentrations by changing commuting behaviour, road traffic and human activity.

A second Prophet model is trained using the same settings as the baseline model, with the addition of the built-in United Kingdom public holiday calendar. Using the same training and test periods ensures a fair comparison between the two models.

In [ ]:
# Create a Prophet model with UK public holidays
prophet_holidays_model: Prophet = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True
)

# Add the United Kingdom public holiday calendar
prophet_holidays_model.add_country_holidays(
    country_name="UK"
)

# Train the model
prophet_holidays_model.fit(prophet_train)

# Generate forecasts for the 2023 test period
prophet_holidays_forecast: pd.DataFrame = (
    prophet_holidays_model.predict(
        prophet_test[["ds"]]
    )
)

# Prepare the results
prophet_holidays_results: pd.DataFrame = (
    prophet_holidays_forecast[
        ["ds", "yhat", "yhat_lower", "yhat_upper"]
    ]
    .copy()
)

# Add the actual NO₂ values
prophet_holidays_results["Actual"] = (
    prophet_test["y"].to_numpy()
)

# Rename the columns
prophet_holidays_results = (
    prophet_holidays_results.rename(
        columns={
            "yhat": "Predicted",
            "yhat_lower": "Lower Bound",
            "yhat_upper": "Upper Bound"
        }
    )
)

display(prophet_holidays_results.head())

### 22.2 Evaluation and Comparison

The Prophet model with UK public holidays is evaluated on the unseen 2023 test data using MAE, RMSE and MAPE.

Its performance is compared with the baseline Prophet model to determine whether adding UK public holidays improves the daily NO₂ forecasts. Lower metric values indicate better forecasting performance.

In [ ]:
# Extract the actual and predicted values
holidays_y_true: pd.Series = prophet_holidays_results["Actual"]
holidays_y_pred: pd.Series = prophet_holidays_results["Predicted"]

# Calculate MAE
prophet_holidays_mae: float = mean_absolute_error(
    holidays_y_true,
    holidays_y_pred
)

# Calculate RMSE
prophet_holidays_rmse: float = np.sqrt(
    mean_squared_error(
        holidays_y_true,
        holidays_y_pred
    )
)

# Calculate MAPE
non_zero_mask: pd.Series = holidays_y_true != 0

prophet_holidays_mape: float = np.mean(
    np.abs(
        (
            holidays_y_true[non_zero_mask]
            - holidays_y_pred[non_zero_mask]
        )
        / holidays_y_true[non_zero_mask]
    )
) * 100

# Compare both Prophet models
prophet_comparison: pd.DataFrame = pd.DataFrame({
    "Model": [
        "Baseline Prophet",
        "Prophet with UK Holidays"
    ],
    "MAE": [
        prophet_baseline_mae,
        prophet_holidays_mae
    ],
    "RMSE": [
        prophet_baseline_rmse,
        prophet_holidays_rmse
    ],
    "MAPE (%)": [
        prophet_baseline_mape,
        prophet_holidays_mape
    ]
})

display(
    prophet_comparison.style.format({
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "MAPE (%)": "{:.2f}"
    })
)

The addition of UK public holidays does not meaningfully improve the Prophet model.

Compared with the baseline model, the holiday model produces a slightly higher MAE (7.704 versus 7.691) and RMSE (10.511 versus 10.498), indicating a marginal deterioration in the absolute forecast errors.

Its MAPE decreases slightly from 58.42% to 58.11%. However, this small improvement is insufficient to compensate for the deterioration in MAE and RMSE. MAPE is also sensitive to very low observed NO₂ values.

Overall, UK public holidays provide little additional predictive value for this daily NO₂ series. The baseline Prophet model is therefore preferred at this stage because it is simpler and performs slightly better according to the primary MAE and RMSE metrics.

## 23. Prophet Parameter Tuning and Cross-Validation

### 23.1 Setting Up Time-Series Cross-Validation

Time-series cross-validation evaluates the Prophet model on several historical periods while preserving chronological order.

The model is first trained on an initial historical window and then evaluated repeatedly on future validation periods. This provides a more reliable performance estimate than evaluating the model on a single test period.

The final 2023 test set remains completely separate and will only be used after selecting the best Prophet parameters.

In [ ]:
from prophet.diagnostics import cross_validation, performance_metrics
from itertools import product

# Cross-validation settings
cv_initial: str = "3652 days"  # About 10 years of initial training data
cv_period: str = "180 days"    # Start a new validation every 6 months
cv_horizon: str = "365 days"   # Forecast one year ahead

print("Initial training window:", cv_initial)
print("Validation period:", cv_period)
print("Forecast horizon:", cv_horizon)

### 23.2 Defining the Parameter Grid

A small parameter grid is defined to test different levels of trend flexibility, seasonality strength and seasonality mode.

A limited number of combinations is used to keep the computation time manageable.

In [ ]:
# Define the Prophet parameters to test
prophet_parameter_grid: dict = {
    "changepoint_prior_scale": [0.01, 0.1],
    "seasonality_prior_scale": [1.0, 10.0],
    "seasonality_mode": ["additive", "multiplicative"]
}

# Create all parameter combinations
prophet_parameter_combinations: list = [
    dict(zip(prophet_parameter_grid.keys(), values))
    for values in product(*prophet_parameter_grid.values())
]

print(
    "Number of parameter combinations:",
    len(prophet_parameter_combinations)
)

display(
    pd.DataFrame(prophet_parameter_combinations)
)

### 23.3 Testing the Parameter Combinations

Each parameter combination is evaluated using time-series cross-validation on the training period.

For every configuration, Prophet is repeatedly trained on historical observations and evaluated on later validation periods. The average MAE, RMSE and MAPE are recorded.

The 2023 test data remain excluded from this process to prevent data leakage.

In [ ]:
# Store the cross-validation results
prophet_tuning_results: list = []

# Test each parameter combination
for number, parameters in enumerate(
    prophet_parameter_combinations,
    start=1
):
    print(
        f"Testing combination "
        f"{number}/{len(prophet_parameter_combinations)}"
    )

    # Create the model with the current parameters
    model: Prophet = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        **parameters
    )

    # Train the model
    model.fit(prophet_train)

    # Perform time-series cross-validation
    cv_predictions: pd.DataFrame = cross_validation(
        model,
        initial=cv_initial,
        period=cv_period,
        horizon=cv_horizon
    )

    # Calculate validation metrics
    cv_metrics: pd.DataFrame = performance_metrics(
        cv_predictions
    )

    # Save the parameters and average metrics
    prophet_tuning_results.append({
        **parameters,
        "MAE": cv_metrics["mae"].mean(),
        "RMSE": cv_metrics["rmse"].mean(),
        "MAPE (%)": cv_metrics["mape"].mean() * 100
    })

    # Create and rank the results table
prophet_tuning_table: pd.DataFrame = (
    pd.DataFrame(prophet_tuning_results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(
    prophet_tuning_table.style.format({
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "MAPE (%)": "{:.2f}"
    })
)

The best parameter combination uses a changepoint prior scale of 0.1, a seasonality prior scale of 1.0 and multiplicative seasonality.

This configuration achieved the lowest cross-validation MAE (7.295) and RMSE (9.689). Its MAPE was 50.13%, although MAPE remains sensitive to days with very low observed NO₂ values.

A changepoint prior scale of 0.1 allows Prophet's trend to adapt more flexibly to historical changes. The multiplicative mode indicates that the magnitude of the seasonal variations changes according to the overall NO₂ level.

The seasonality prior scale had only a limited effect, since the results obtained with values of 1.0 and 10.0 were very similar.

This configuration is therefore selected for the final optimized Prophet model.

### 23.4 Training the Optimized Prophet Model

The best parameter configuration identified through time-series cross-validation is used to train the optimized Prophet model on the complete 2010–2022 training period.

The optimized model then generates daily forecasts for the unseen 2023 test period. These predictions will subsequently be evaluated and compared with the baseline Prophet model.

In [ ]:
# Extract the best parameters from the tuning results
best_prophet_parameters: dict = {
    "changepoint_prior_scale": prophet_tuning_table.loc[
        0, "changepoint_prior_scale"
    ],
    "seasonality_prior_scale": prophet_tuning_table.loc[
        0, "seasonality_prior_scale"
    ],
    "seasonality_mode": prophet_tuning_table.loc[
        0, "seasonality_mode"
    ]
}

print("Selected parameters:")
print(best_prophet_parameters)

# Create the optimized Prophet model
optimized_prophet_model: Prophet = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    **best_prophet_parameters
)

# Train the model on the complete training period
optimized_prophet_model.fit(prophet_train)

# Generate forecasts for the unseen 2023 period
optimized_prophet_forecast: pd.DataFrame = (
    optimized_prophet_model.predict(
        prophet_test[["ds"]]
    )
)

# Prepare the forecast results
optimized_prophet_results: pd.DataFrame = (
    optimized_prophet_forecast[
        ["ds", "yhat", "yhat_lower", "yhat_upper"]
    ]
    .copy()
    .rename(
        columns={
            "yhat": "Predicted",
            "yhat_lower": "Lower Bound",
            "yhat_upper": "Upper Bound"
        }
    )
)

# Add the actual observations
optimized_prophet_results["Actual"] = (
    prophet_test["y"].to_numpy()
)

display(optimized_prophet_results.head())

### 23.5 Evaluation of the Optimized Prophet Model

The optimized Prophet model is evaluated on the unseen 2023 test period using MAE, RMSE and MAPE.

Its performance is compared with the baseline Prophet model to determine whether parameter tuning and time-series cross-validation improved the final forecasts.

In [ ]:
# Extract actual and predicted values
optimized_y_true: pd.Series = optimized_prophet_results["Actual"]
optimized_y_pred: pd.Series = optimized_prophet_results["Predicted"]

# Calculate MAE
optimized_prophet_mae: float = mean_absolute_error(
    optimized_y_true,
    optimized_y_pred
)

# Calculate RMSE
optimized_prophet_rmse: float = np.sqrt(
    mean_squared_error(
        optimized_y_true,
        optimized_y_pred
    )
)

# Calculate MAPE
optimized_non_zero_mask: pd.Series = optimized_y_true != 0

optimized_prophet_mape: float = np.mean(
    np.abs(
        (
            optimized_y_true[optimized_non_zero_mask]
            - optimized_y_pred[optimized_non_zero_mask]
        )
        / optimized_y_true[optimized_non_zero_mask]
    )
) * 100

# Compare baseline and optimized Prophet models
prophet_final_comparison: pd.DataFrame = pd.DataFrame({
    "Model": [
        "Baseline Prophet",
        "Optimized Prophet"
    ],
    "MAE": [
        prophet_baseline_mae,
        optimized_prophet_mae
    ],
    "RMSE": [
        prophet_baseline_rmse,
        optimized_prophet_rmse
    ],
    "MAPE (%)": [
        prophet_baseline_mape,
        optimized_prophet_mape
    ]
})

display(
    prophet_final_comparison.style.format({
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "MAPE (%)": "{:.2f}"
    })
)

The optimized Prophet model outperforms the baseline Prophet model on all three evaluation metrics.

The MAE decreases from 7.691 to 6.886, representing an improvement of approximately 10.5% in the average absolute forecast error. The RMSE decreases more modestly from 10.498 to 10.323, indicating that large forecast errors and extreme NO₂ peaks remain difficult to predict.

The MAPE also decreases from 58.42% to 52.18%. However, this metric remains relatively high and should be interpreted cautiously because it is sensitive to days with very low observed NO₂ concentrations.

Overall, parameter tuning and time-series cross-validation improved Prophet's performance on the unseen 2023 test period. The optimized Prophet model is therefore retained as the best Prophet configuration for the next comparisons.

## 24. Advanced Prophet

This section investigates whether additional Prophet components can improve the optimized model selected in the previous section.

The reference model already includes weekly and yearly seasonalities, automatically detected changepoints, and the best hyperparameters identified through time-series cross-validation:

- `changepoint_prior_scale = 0.1`;
- `seasonality_prior_scale = 1.0`;
- `seasonality_mode = "multiplicative"`.

Three advanced configurations are evaluated:

1. a custom monthly seasonality represented by Fourier terms;
2. COVID-19 lockdown periods represented as one-off historical events;
3. informed changepoints based on documented events, including the introduction of the London ULEZ and the COVID-19 lockdowns.

The potential use of external regressors, such as weather or traffic variables, is also discussed. However, no external regressor is tested because the current dataset does not contain complete and reliable daily external variables.

Each tested component adds complexity and is retained only if it improves forecasting performance during time-series cross-validation on the 2010–2022 training period.

The unseen 2023 test period remains excluded from model selection and is reserved for the final evaluation of the selected Prophet model.

### 24.1 Custom Monthly Seasonality with Fourier Terms

The optimized Prophet model already includes weekly and yearly seasonalities. This experiment investigates whether an additional pattern repeating approximately every 30.5 days provides useful information for forecasting daily NO₂ concentrations.

Prophet represents seasonal patterns using Fourier terms. These are combinations of sine and cosine waves that allow the model to reproduce recurring patterns with different shapes.

The custom monthly seasonality is defined using:

- `period = 30.5`, representing a cycle of approximately 30.5 days;
- `fourier_order = 5`, producing five sine terms and five cosine terms;
- `mode = "multiplicative"`, following the seasonality mode selected during parameter tuning.

A Fourier order of 5 creates ten seasonal features in total. This provides enough flexibility to represent a smooth monthly pattern while limiting excessive complexity and overfitting.

The monthly seasonality is evaluated using the same time-series cross-validation settings used during Prophet tuning. Only the 2010–2022 training period is used. The unseen 2023 test period remains excluded from model selection.

The purpose is not to assume that a monthly cycle exists, but to test whether it reduces the cross-validation forecast errors.

In [ ]:
# Create the Prophet model with custom monthly seasonality
monthly_prophet_cv_model: Prophet = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    **best_prophet_parameters
)

# Add an approximately monthly cycle using Fourier terms
monthly_prophet_cv_model.add_seasonality(
    name="monthly",
    period=30.5,
    fourier_order=5,
    mode=best_prophet_parameters["seasonality_mode"]
)

# Train the model using only the 2010–2022 training data
monthly_prophet_cv_model.fit(prophet_train)

# Perform time-series cross-validation
monthly_prophet_cv_predictions: pd.DataFrame = cross_validation(
    monthly_prophet_cv_model,
    initial=cv_initial,
    period=cv_period,
    horizon=cv_horizon
)

# Calculate the cross-validation metrics
monthly_prophet_cv_metrics: pd.DataFrame = performance_metrics(
    monthly_prophet_cv_predictions
)

# Calculate the average metrics
monthly_prophet_cv_summary: dict = {
    "Model": "Optimized Prophet + Monthly Seasonality",
    "MAE": monthly_prophet_cv_metrics["mae"].mean(),
    "RMSE": monthly_prophet_cv_metrics["rmse"].mean(),
    "MAPE (%)": monthly_prophet_cv_metrics["mape"].mean() * 100
}

print("Monthly seasonality cross-validation results:")

display(
    pd.DataFrame([monthly_prophet_cv_summary]).style.format({
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "MAPE (%)": "{:.2f}"
    })
)

#### Interpretation

The custom monthly seasonality did not improve Prophet's cross-validation performance.

Compared with the optimized Prophet model:

- MAE increased from 7.295 to 7.370;
- RMSE increased from 9.689 to 9.761;
- MAPE increased from 50.13% to 50.72%.

Lower values indicate better forecasting performance. Therefore, the model with monthly seasonality performs slightly worse across all three metrics.

This result suggests that the additional 30.5-day cycle does not provide useful predictive information beyond the weekly and yearly seasonal patterns already captured by Prophet. The ten additional Fourier features may instead introduce unnecessary complexity or capture noise.

The custom monthly seasonality is therefore rejected. The optimized Prophet model without this additional component remains the reference model for the next advanced experiments.

### 24.2 COVID-19 Lockdowns as One-Off Events

The COVID-19 lockdowns caused exceptional changes in mobility, road traffic and economic activity. These disruptions may also have temporarily affected daily NO₂ concentrations in London.

Prophet allows unusual historical periods to be represented as custom events. In this experiment, each lockdown is defined as a one-off event with a start date and an end date.

The following three main lockdown periods in England are considered:

- first lockdown: 26 March to 4 July 2020;
- second lockdown: 5 November to 2 December 2020;
- third lockdown: 6 January to 29 March 2021.

These periods are encoded using Prophet's holiday framework. They are not ordinary public holidays and they do not repeat annually. The name `lockdown` simply tells Prophet to estimate a specific effect associated with these exceptional periods.

The lockdown model is based on the optimized Prophet parameters without the rejected monthly seasonality. It is evaluated using time-series cross-validation on the 2010–2022 training period. The unseen 2023 test data remain excluded from model selection.

This experiment tests whether explicitly identifying the lockdown periods improves forecasting performance beyond the trend and seasonal patterns already learned by Prophet.

In [ ]:
# Define the main COVID-19 lockdown periods in England
lockdown_periods: list = [
    ("2020-03-26", "2020-07-04"),
    ("2020-11-05", "2020-12-02"),
    ("2021-01-06", "2021-03-29")
]

# Create one row for each day covered by a lockdown
lockdown_dates: list = []

for start_date, end_date in lockdown_periods:
    lockdown_dates.extend(
        pd.date_range(
            start=start_date,
            end=end_date,
            freq="D"
        )
    )

# Create the custom events DataFrame required by Prophet
covid_lockdown_events: pd.DataFrame = pd.DataFrame({
    "holiday": "covid_lockdown",
    "ds": pd.to_datetime(lockdown_dates)
})

print(
    "Number of lockdown days:",
    len(covid_lockdown_events)
)

display(covid_lockdown_events.head())
display(covid_lockdown_events.tail())

#### 24.2.a. Cross-Validation of the Lockdown Event Model

The COVID-19 lockdown dates are now incorporated into the optimized Prophet model through the `holidays` parameter.

Prophet estimates a specific coefficient for the lockdown event. This coefficient represents the average change in NO₂ associated with the dates labelled as lockdown periods, after accounting for the model's trend and seasonal components.

The lockdown effect is treated as an additional explanatory component rather than as a recurring annual seasonality. It is therefore applied only to the specified dates in 2020 and 2021.

The model is evaluated using the same time-series cross-validation configuration as the previous experiments. Its MAE, RMSE and MAPE are compared with those of the optimized Prophet reference model.

An improvement would indicate that explicitly identifying the lockdown periods provides information not already captured by Prophet's flexible trend and seasonalities.

In [ ]:
# Create the optimized Prophet model with COVID-19 lockdown events
lockdown_prophet_cv_model: Prophet = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    holidays=covid_lockdown_events,
    **best_prophet_parameters
)

# Train the model using only the 2010–2022 training data
lockdown_prophet_cv_model.fit(prophet_train)

# Perform time-series cross-validation
lockdown_prophet_cv_predictions: pd.DataFrame = cross_validation(
    lockdown_prophet_cv_model,
    initial=cv_initial,
    period=cv_period,
    horizon=cv_horizon
)

# Calculate cross-validation metrics
lockdown_prophet_cv_metrics: pd.DataFrame = performance_metrics(
    lockdown_prophet_cv_predictions
)

# Calculate the average metrics
lockdown_prophet_cv_summary: dict = {
    "Model": "Optimized Prophet + COVID Lockdowns",
    "MAE": lockdown_prophet_cv_metrics["mae"].mean(),
    "RMSE": lockdown_prophet_cv_metrics["rmse"].mean(),
    "MAPE (%)": lockdown_prophet_cv_metrics["mape"].mean() * 100
}

# Create a comparison with the optimized Prophet reference model
lockdown_cv_comparison: pd.DataFrame = pd.DataFrame([
    {
        "Model": "Optimized Prophet",
        "MAE": 7.295,
        "RMSE": 9.689,
        "MAPE (%)": 50.13
    },
    lockdown_prophet_cv_summary
])

display(
    lockdown_cv_comparison.style.format({
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "MAPE (%)": "{:.2f}"
    })
)

#### 24.2.b. Interpretation

Adding the COVID-19 lockdown periods as one-off events did not improve Prophet's cross-validation performance.

Compared with the optimized Prophet reference model:

- MAE increased from 7.295 to 7.405;
- RMSE increased from 9.689 to 9.724;
- MAPE increased from 50.13% to 51.73%.

Because lower error values indicate better forecasting performance, the lockdown event model performs slightly worse across all three metrics.

This result suggests that explicitly identifying the lockdown periods does not provide enough additional predictive information. Prophet's flexible trend and automatically detected changepoints may already capture part of the structural changes observed during 2020 and 2021.

Moreover, a single event label assumes a relatively similar lockdown effect across all selected days. In reality, changes in traffic, mobility and NO₂ concentrations may have varied throughout each lockdown period.

The COVID-19 lockdown events are therefore not retained in the final model. The optimized Prophet model without these events remains the reference configuration.

### 24.3 Additional External Regressors

Prophet can incorporate additional variables that may help explain changes in the target time series. These variables are called external regressors.

For daily NO₂ concentrations, potentially useful regressors include:

- daily temperature;
- wind speed and direction;
- precipitation;
- road traffic volume;
- atmospheric pressure;
- industrial activity indicators.

These variables could improve the forecasts because NO₂ concentrations are influenced not only by trend and seasonality, but also by meteorological conditions and emission-generating activities.

For example, stronger winds may disperse pollutants, while heavy road traffic may increase NO₂ concentrations. A model that knows these conditions may therefore distinguish between recurring seasonal patterns and short-term environmental effects.

However, an external regressor can only be used correctly when:

1. reliable historical values are available for the entire training period;
2. the variable has the same daily frequency as the NO₂ series;
3. missing values are treated consistently;
4. its future values are known or can be forecast for the prediction period;
5. the regressor does not contain information that would have been unavailable at forecast time.

The current dataset does not contain complete meteorological, traffic or industrial variables for 2010–2023. Creating artificial values or using information unavailable at prediction time would produce unreliable results and could introduce data leakage.

The previously tested lockdown event is a binary event indicator rather than a continuous numerical environmental regressor. It did not improve cross-validation performance and was therefore rejected.

Consequently, no additional external regressor is added to the current Prophet model. This is retained as a limitation and a potential direction for future work.

A future extension could combine the NO₂ series with historical weather and traffic data and evaluate their contribution using the same time-series cross-validation framework.

### 24.4 Informed Changepoints

A changepoint is a date at which the rate of change in the time-series trend may shift.

For example, NO₂ concentrations may already be decreasing, but after a changepoint this decrease could become faster, slower or temporarily reverse. A changepoint therefore modifies the slope of the trend; it does not directly represent a temporary event or isolated anomaly.

By default, Prophet automatically identifies potential changepoints from the historical data. In this experiment, automatic changepoint locations are replaced with dates associated with documented changes that could plausibly have influenced traffic and NO₂ concentrations in London.

The following dates are considered:

- `8 April 2019`: introduction of the Ultra Low Emission Zone (ULEZ) in central London;
- `26 March 2020`: beginning of the first legally enforced COVID-19 lockdown in England;
- `5 November 2020`: beginning of the second national lockdown;
- `6 January 2021`: beginning of the third national lockdown.

The ULEZ was introduced to reduce emissions from the most polluting vehicles. The lockdowns produced major changes in mobility, road traffic and economic activity.

These dates are plausible candidates, but their inclusion does not prove that they caused a change in NO₂ at the selected monitoring site. Other environmental, meteorological and behavioural factors may have contributed to the observed trend.

Passing dates through Prophet's `changepoints` parameter replaces the automatically selected changepoint locations. This is therefore a substantial modelling assumption and must be evaluated using time-series cross-validation.

The model uses the optimized Prophet parameters and excludes the previously rejected monthly seasonality and lockdown holiday events. The unseen 2023 test period remains excluded from model selection.

In [ ]:
# Define documented dates that may correspond to trend changes
informed_changepoints: list = pd.to_datetime([
    "2019-04-08",  # Introduction of the central London ULEZ
    "2020-03-26",  # First legally enforced lockdown
    "2020-11-05",  # Second national lockdown
    "2021-01-06"   # Third national lockdown
])

# Create Prophet with manually specified changepoints
informed_changepoint_model: Prophet = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    changepoints=informed_changepoints,
    **best_prophet_parameters
)

# Fit using only the 2010–2022 training period
informed_changepoint_model.fit(prophet_train)

# Perform time-series cross-validation
informed_changepoint_cv_predictions: pd.DataFrame = cross_validation(
    informed_changepoint_model,
    initial=cv_initial,
    period=cv_period,
    horizon=cv_horizon
)

# Calculate the cross-validation metrics
informed_changepoint_cv_metrics: pd.DataFrame = performance_metrics(
    informed_changepoint_cv_predictions
)

# Summarize the average performance
informed_changepoint_cv_summary: dict = {
    "Model": "Optimized Prophet + Informed Changepoints",
    "MAE": informed_changepoint_cv_metrics["mae"].mean(),
    "RMSE": informed_changepoint_cv_metrics["rmse"].mean(),
    "MAPE (%)": (
        informed_changepoint_cv_metrics["mape"].mean() * 100
    )
}

# Compare with the optimized Prophet reference model
informed_changepoint_comparison: pd.DataFrame = pd.DataFrame([
    {
        "Model": "Optimized Prophet",
        "MAE": 7.295,
        "RMSE": 9.689,
        "MAPE (%)": 50.13
    },
    informed_changepoint_cv_summary
])

display(
    informed_changepoint_comparison.style.format({
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "MAPE (%)": "{:.2f}"
    })
)

#### Interpretation

The manually specified changepoints did not improve Prophet's cross-validation performance.

Compared with the optimized Prophet model using automatically detected changepoints:

- MAE increased from 7.295 to 7.761;
- RMSE increased from 9.689 to 10.442;
- MAPE increased from 50.13% to 50.39%.

Since lower values indicate better forecasting performance, the informed-changepoint model performs worse across all three metrics. The increase in RMSE is particularly noticeable, suggesting that the model produces larger forecast errors during some validation periods.

Although the selected dates correspond to documented events that could plausibly affect road traffic and NO₂ concentrations, they do not necessarily represent the most important changes in the trend of this specific monitoring-site series.

Furthermore, manually providing these dates replaces Prophet's automatically selected changepoint locations. Restricting the model to only four predetermined dates appears to reduce its ability to capture other relevant changes occurring throughout the 2010–2022 training period.

The informed changepoints are therefore rejected. The optimized Prophet model with automatically detected changepoints remains the preferred configuration.

### 24.5 Advanced Prophet Model Comparison

The advanced Prophet experiments are now compared using their average time-series cross-validation errors.

All configurations were evaluated on the 2010–2022 training period using the same cross-validation settings. The unseen 2023 test period was not used to select among these variants.

The comparison includes:

- the optimized Prophet reference model;
- the model with custom monthly seasonality;
- the model with COVID-19 lockdown events;
- the model with informed changepoints.

The external-regressor approach is not included because the dataset does not contain complete and reliable meteorological, traffic or industrial variables.

For MAE, RMSE and MAPE, lower values indicate better forecasting performance. A more complex model should only be retained if it produces a consistent and meaningful reduction in forecasting errors.

In [ ]:
# Combine the cross-validation results of all advanced Prophet experiments
advanced_prophet_cv_comparison: pd.DataFrame = pd.DataFrame([
    {
        "Model": "Optimized Prophet",
        "MAE": 7.295,
        "RMSE": 9.689,
        "MAPE (%)": 50.13
    },
    monthly_prophet_cv_summary,
    lockdown_prophet_cv_summary,
    informed_changepoint_cv_summary
])

# Sort the models from the lowest to the highest RMSE
advanced_prophet_cv_comparison = (
    advanced_prophet_cv_comparison
    .sort_values(by="RMSE")
    .reset_index(drop=True)
)

display(
    advanced_prophet_cv_comparison.style
    .format({
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "MAPE (%)": "{:.2f}"
    })
    .highlight_min(
        subset=["MAE", "RMSE", "MAPE (%)"],
        color="lightgreen"
    )
)

In [ ]:
# Prepare the 2023 test dates for Prophet
future_test: pd.DataFrame = pd.DataFrame({
    "ds": test.index
})

# Generate predictions with the optimized Prophet model
optimized_forecast: pd.DataFrame = (
    optimized_prophet_model
    .predict(future_test)
)

# Plot actual values and Optimized Prophet predictions
fig: plt.Figure
ax: plt.Axes

fig, ax = plt.subplots(figsize=(16, 6))

# Actual NO₂ values
ax.plot(
    test.index,
    test["no2_level"],
    color="black",
    linewidth=1,
    label="Actual NO₂"
)

# Optimized Prophet predictions
ax.plot(
    optimized_forecast["ds"],
    optimized_forecast["yhat"],
    color="royalblue",
    linewidth=2,
    label="Optimized Prophet"
)

# Prediction interval
ax.fill_between(
    optimized_forecast["ds"],
    optimized_forecast["yhat_lower"],
    optimized_forecast["yhat_upper"],
    color="royalblue",
    alpha=0.20,
    label="Prediction interval"
)

ax.set_title(
    "Optimized Prophet Forecast vs Actual NO₂ — Test Period (2023)"
)
ax.set_xlabel("Date")
ax.set_ylabel("Daily Mean NO₂ Concentration (µg/m³)")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

#### Interpretation

None of the advanced Prophet techniques improved cross-validation performance. The selected model remains the optimized Prophet configuration with:

- `changepoint_prior_scale = 0.1`;
- `seasonality_prior_scale = 1.0`;
- `seasonality_mode = "multiplicative"`;
- weekly and yearly seasonalities enabled;
- automatically detected changepoints.

No monthly seasonality, COVID-19 events or external regressors are retained, confirming that additional complexity does not necessarily improve forecasts.

## 25. Fourier Features

The previous Prophet experiments used Fourier terms internally to represent seasonal patterns. In this section, Fourier features are created manually for the feature-based machine-learning model.

A calendar variable such as `day_of_year` increases from 1 to 365 and then suddenly returns to 1. A machine-learning model may not naturally understand that the end and beginning of the year are adjacent and belong to the same recurring cycle.

Fourier features solve this problem by transforming time into sine and cosine waves. These transformations preserve the cyclical nature of seasonality:

- the end of December remains close to the beginning of January;
- recurring annual patterns can be represented smoothly;
- several harmonics can describe seasonal patterns that are more complex than a single wave.

For each harmonic \(k\), two features are created:

\[
\sin\left(\frac{2\pi kt}{P}\right)
\]

\[
\cos\left(\frac{2\pi kt}{P}\right)
\]

where:

- \(t\) represents the position in time;
- \(P\) is the length of the seasonal cycle;
- \(k\) is the harmonic number.

The first harmonic represents one complete cycle per year. Higher harmonics add more waves within the same year and allow more complex seasonal shapes to be represented.

These features will later be combined with lag, rolling and calendar features in the `HistGradientBoostingRegressor`. Their contribution will be evaluated through time-based cross-validation.

### 25.1 Creating Annual Fourier Terms

The daily NO₂ series exhibits an annual seasonal pattern. Annual Fourier features are therefore created using a period of approximately 365.25 days.

Three harmonics are initially used:

- harmonic 1 represents one cycle per year;
- harmonic 2 represents two cycles per year;
- harmonic 3 represents three cycles per year.

Each harmonic produces one sine feature and one cosine feature. Three harmonics therefore create six annual Fourier features.

Using several harmonics gives the machine-learning model enough flexibility to represent a non-sinusoidal annual pattern while keeping the number of additional features limited.

In [ ]:
def add_fourier_features(
    df: pd.DataFrame,
    period: float = 365.25,
    fourier_order: int = 3
) -> pd.DataFrame:
    """Add annual Fourier sine and cosine features."""

    df_with_fourier: pd.DataFrame = df.copy()

    # Convert each date into the number of elapsed days
    time_in_days: np.ndarray = (
        df_with_fourier.index - df_with_fourier.index.min()
    ).days.to_numpy()

    # Create one sine and one cosine feature per harmonic
    for harmonic in range(1, fourier_order + 1):
        df_with_fourier[f"yearly_sin_{harmonic}"] = np.sin(
            2 * np.pi * harmonic * time_in_days / period
        )

        df_with_fourier[f"yearly_cos_{harmonic}"] = np.cos(
            2 * np.pi * harmonic * time_in_days / period
        )

    return df_with_fourier

### 25.2 Creating and Inspecting the Fourier Features

The Fourier transformation is now applied to the complete daily NO₂ series.

The full chronological index is used so that the time variable remains continuous between the training period and the 2023 test period. This transformation does not create data leakage because the Fourier features depend only on the date, not on current or future NO₂ values.

With a Fourier order of three, six annual features are generated: one sine and one cosine feature for each harmonic.

In [ ]:
# Combine the training and test series while preserving chronological order
full_daily_series: pd.DataFrame = pd.concat([
    train,
    test
]).sort_index()

# Add six annual Fourier features
fourier_daily_data: pd.DataFrame = add_fourier_features(
    full_daily_series,
    period=365.25,
    fourier_order=3
)

# Identify the generated Fourier columns
fourier_columns: list = [
    column
    for column in fourier_daily_data.columns
    if column.startswith("yearly_")
]

print("Fourier features created:")
print(fourier_columns)

display(
    fourier_daily_data[fourier_columns].head()
)

### 25.3 Visualizing the Fourier Features

The Fourier features are visualized over one complete year to understand how they represent annual seasonality.

The first harmonic completes one cycle per year. The second and third harmonics complete two and three cycles respectively. Combining these waves allows the future machine-learning model to represent a more complex annual seasonal pattern.

All Fourier features remain between -1 and 1. They describe the position within the seasonal cycle but do not directly contain information about NO₂ concentrations.

In [ ]:
# Select one complete year for visualization
fourier_visualization: pd.DataFrame = (
    fourier_daily_data
    .loc["2019-01-01":"2019-12-31", fourier_columns]
)

# Plot the Fourier features
fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(14, 10),
    sharex=True
)

for harmonic, axis in enumerate(axes, start=1):
    axis.plot(
        fourier_visualization.index,
        fourier_visualization[f"yearly_sin_{harmonic}"],
        label=f"Sine – harmonic {harmonic}"
    )

    axis.plot(
        fourier_visualization.index,
        fourier_visualization[f"yearly_cos_{harmonic}"],
        label=f"Cosine – harmonic {harmonic}",
        linestyle="--"
    )

    axis.set_ylabel("Feature value")
    axis.set_title(f"Annual Fourier Harmonic {harmonic}")
    axis.legend()
    axis.grid(alpha=0.3)

axes[-1].set_xlabel("Date")

plt.suptitle(
    "Annual Fourier Features over One Year",
    fontsize=15,
    y=1.02
)

plt.tight_layout()
plt.show()

#### Interpretation

The graph confirms that the Fourier features correctly represent annual seasonality at different frequencies.

- harmonic 1 completes one cycle per year and captures the broad annual pattern;
- harmonic 2 completes two cycles per year and captures intermediate seasonal variations;
- harmonic 3 completes three cycles per year and captures finer variations within the year.

For each harmonic, the sine and cosine curves are shifted relative to each other. Their combination allows every position within the annual cycle to be represented correctly.

All values remain between -1 and 1, and the curves repeat smoothly. Unlike a simple calendar variable, these features avoid a discontinuity between the end of December and the beginning of January.

Together, the six Fourier features provide the future machine-learning model with a flexible representation of annual seasonality.

## 26. Feature-Based Machine Learning with HistGradientBoostingRegressor

This section develops a feature-based machine-learning model for forecasting daily NO₂ concentrations.

Unlike ARIMA, SARIMA and Prophet, `HistGradientBoostingRegressor` does not automatically understand the chronological structure of a time series. Temporal information must therefore be transformed into numerical explanatory variables.

Four groups of features are created:

1. **lag features**, representing previous NO₂ observations;
2. **rolling features**, summarizing recent historical behaviour;
3. **calendar features**, describing the position of each observation within the calendar;
4. **Fourier features**, representing annual seasonality through smooth cyclical transformations.

The model uses these historical and calendar variables to predict the daily NO₂ concentration.

`HistGradientBoostingRegressor` is a tree-based ensemble model that builds decision trees successively. Each new tree attempts to correct part of the prediction errors produced by the preceding trees. Its histogram-based implementation is computationally efficient and can model nonlinear relationships and interactions between features.

To prevent data leakage:

- every lag feature uses only observations recorded before the prediction date;
- rolling statistics are calculated after shifting the target by one day;
- calendar and Fourier features are derived only from the date;
- the chronological train-test split is preserved;
- the 2023 test data are not used during model training or hyperparameter selection.

The evaluation performed in this section follows a **one-day-ahead forecasting strategy with observed historical updates**. This means that each prediction may use actual NO₂ observations from preceding days. It is not a recursive multi-step forecast of the complete year without intermediate observations.

The initial model provides a reference configuration. Its preliminary performance is evaluated on the 2023 test period. Time-based cross-validation and hyperparameter tuning will subsequently be performed using only the 2011–2022 training period.

Because the initial model results for 2023 are examined before hyperparameter optimization, they should be considered a **preliminary test benchmark** rather than a completely untouched final evaluation. The 2023 data will nevertheless remain excluded from the cross-validation and tuning processes.

### 26.1 Creating Lag Features

Lag features provide the machine-learning model with previous NO₂ observations.

For a date \(t\), a lag of \(k\) days is defined as:

\[
\text{lag}_k(t) = y_{t-k}
\]

where \(y_{t-k}\) is the NO₂ concentration observed \(k\) days before the prediction date.

The following lags are created:

- `lag_1`: the previous day;
- `lag_2`: two days earlier;
- `lag_3`: three days earlier;
- `lag_7`: the same weekday one week earlier;
- `lag_14`: two weeks earlier;
- `lag_30`: approximately one month earlier;
- `lag_365`: approximately one year earlier.

Short lags capture recent temporal dependence, while longer lags may help represent weekly, monthly and annual recurrence.

All lag values are calculated from previous observations only. Therefore, the target value for the prediction date is never included in its own explanatory variables.

In [ ]:
def add_lag_features(
    df: pd.DataFrame,
    target_column: str,
    lags: list = [1, 2, 3, 7, 14, 30, 365]
) -> pd.DataFrame:
    """Create lagged target features using past observations only."""

    df_with_lags: pd.DataFrame = df.copy()

    for lag in lags:
        df_with_lags[f"lag_{lag}"] = (
            df_with_lags[target_column].shift(lag)
        )

    return df_with_lags

### 26.2 Applying and Inspecting Lag Features

The lag-feature function is now applied to the complete chronological dataset containing the daily NO₂ series and the previously created Fourier features.

Lag features naturally generate missing values at the beginning of the dataset because no earlier observation exists for these dates. For example:

- `lag_1` has one missing value;
- `lag_7` has seven missing values;
- `lag_365` has 365 missing values.

These missing values are expected and do not indicate a data-quality problem. The incomplete initial rows will be removed only after all lag and rolling features have been created.

In [ ]:
# Identify the target column
target_column: str = full_daily_series.columns[0]

print("Target column:", target_column)

# Add lag features to the complete chronological dataset
ml_feature_data: pd.DataFrame = add_lag_features(
    fourier_daily_data,
    target_column=target_column
)

# Identify the generated lag columns
lag_columns: list = [
    column
    for column in ml_feature_data.columns
    if column.startswith("lag_")
]

print("\nLag features created:")
print(lag_columns)

# Display the target and lag values around the first available weekly lag
display(
    ml_feature_data[
        [target_column] + lag_columns
    ].head(10)
)

# Count the missing values created by each lag
lag_missing_values: pd.Series = (
    ml_feature_data[lag_columns]
    .isna()
    .sum()
    .sort_values()
)

print("\nMissing values generated by lag features:")
display(lag_missing_values.to_frame(name="Missing values"))

#### Interpretation

The lag features were created correctly using past NO₂ observations.

For each date, `lag_k` contains the NO₂ value observed exactly \(k\) days earlier. For example, on 8 January 2010, `lag_1` corresponds to 7 January, while `lag_7` corresponds to 1 January.

The number of missing values matches each lag length:

- `lag_1` contains 1 missing value;
- `lag_2` contains 2 missing values;
- `lag_3` contains 3 missing values;
- `lag_7` contains 7 missing values;
- `lag_14` contains 14 missing values;
- `lag_30` contains 30 missing values;
- `lag_365` contains 365 missing values.

These missing values occur only at the beginning of the dataset, where the required historical observations do not exist. They are expected consequences of feature engineering rather than data-quality issues.

The initial incomplete rows will not be removed yet. They will be handled after all lag and rolling features have been created.

### 26.3 Creating Rolling Features

Rolling features summarize the recent behaviour of the NO₂ series over a fixed historical window.

For each prediction date, the following statistics are calculated:

- the rolling mean, representing the recent average concentration;
- the rolling standard deviation, representing recent variability;
- the rolling minimum and maximum, representing the recent range.

Two windows are used:

- 7 days to describe short-term weekly behaviour;
- 30 days to describe the broader monthly context.

Before calculating each rolling statistic, the target series is shifted by one day. Therefore, the rolling window ends on the day before the prediction date and never includes the NO₂ value being predicted.

For example, `rolling_mean_7` for date \(t\) is calculated using the observations from \(t-7\) to \(t-1\). This prevents target leakage.

In [ ]:
def add_rolling_features(
    df: pd.DataFrame,
    target_column: str,
    windows: list = [7, 30]
) -> pd.DataFrame:
    """Create rolling features using past target observations only."""

    df_with_rolling: pd.DataFrame = df.copy()

    # Shift first to exclude the current target value
    shifted_target: pd.Series = (
        df_with_rolling[target_column].shift(1)
    )

    for window in windows:
        rolling_window = shifted_target.rolling(
            window=window,
            min_periods=window
        )

        df_with_rolling[f"rolling_mean_{window}"] = (
            rolling_window.mean()
        )

        df_with_rolling[f"rolling_std_{window}"] = (
            rolling_window.std()
        )

        df_with_rolling[f"rolling_min_{window}"] = (
            rolling_window.min()
        )

        df_with_rolling[f"rolling_max_{window}"] = (
            rolling_window.max()
        )

    return df_with_rolling

### 26.4 Applying and Inspecting Rolling Features

The rolling-feature function is now applied to the complete chronological dataset.

Because the target is shifted by one day before calculating the rolling statistics, each feature contains only information that would have been available before the prediction date.

The first valid value appears:

- after 7 previous observations for the 7-day features;
- after 30 previous observations for the 30-day features.

The missing values at the beginning of these columns are therefore expected.

In [ ]:
# Add rolling features using past observations only
ml_feature_data = add_rolling_features(
    ml_feature_data,
    target_column=target_column
)

# Identify the generated rolling columns
rolling_columns: list = [
    column
    for column in ml_feature_data.columns
    if column.startswith("rolling_")
]

print("Rolling features created:")
print(rolling_columns)

# Display the first rows containing valid 7-day rolling features
display(
    ml_feature_data[
        [target_column] + rolling_columns
    ].head(35)
)

# Count the missing values generated by each rolling feature
rolling_missing_values: pd.Series = (
    ml_feature_data[rolling_columns]
    .isna()
    .sum()
    .sort_values()
)

print("\nMissing values generated by rolling features:")
display(
    rolling_missing_values.to_frame(name="Missing values")
)

#### Interpretation

The rolling features were created correctly using only past NO₂ observations.

Each 7-day rolling feature contains seven missing values because seven complete previous observations are required. Similarly, each 30-day rolling feature contains 30 missing values.

The first valid 30-day rolling values appear on 31 January 2010. For this date, the statistics are calculated from the observations between 1 and 30 January and do not include the target value from 31 January.

The rolling features describe different aspects of recent NO₂ behaviour:

- the rolling mean represents the recent average concentration;
- the rolling standard deviation measures recent variability;
- the rolling minimum and maximum describe the recent range.

These initial missing values are expected consequences of feature engineering and not data-quality problems. Because the target was shifted by one day before applying the rolling windows, the current NO₂ value is excluded and no target leakage is introduced.

In [ ]:
# Verify the first available 7-day rolling mean manually
first_valid_rolling_date = (
    ml_feature_data["rolling_mean_7"]
    .first_valid_index()
)

previous_seven_days: pd.Series = (
    ml_feature_data
    .loc[:first_valid_rolling_date, target_column]
    .iloc[:-1]
    .tail(7)
)

print("First valid rolling date:", first_valid_rolling_date)
print("\nPrevious seven NO₂ values:")
display(previous_seven_days)

print(
    "\nManual 7-day mean:",
    previous_seven_days.mean()
)

print(
    "Generated rolling_mean_7:",
    ml_feature_data.loc[
        first_valid_rolling_date,
        "rolling_mean_7"
    ]
)

#### Manual Verification

The first valid 7-day rolling mean is available on 8 January 2010.

It is calculated from the seven NO₂ observations recorded between 1 and 7 January 2010. The target value for 8 January is excluded.

The manually calculated mean and the generated `rolling_mean_7` are identical:

- manual mean: 45.3899;
- generated rolling mean: 45.3899.

This confirms that the rolling feature was calculated correctly using only information available before the prediction date. Therefore, no target leakage is introduced.

### 26.5 Creating Calendar Features

Calendar features describe the position of each observation within the week, month and year.

The following variables are created:

- `day_of_week`: Monday = 0 and Sunday = 6;
- `day_of_month`: day number within the month;
- `day_of_year`: day number within the year;
- `week_of_year`: ISO week number;
- `month`: month number;
- `quarter`: quarter of the year;
- `year`: calendar year;
- `is_weekend`: indicates Saturday or Sunday;
- `is_month_start` and `is_month_end`: indicate month boundaries.

These variables may help the model identify calendar-related patterns in NO₂ concentrations. Annual Fourier features remain useful because they represent seasonality smoothly, whereas raw calendar variables are discrete.

All calendar features depend only on the date and therefore do not introduce data leakage.

In [ ]:
def add_calendar_features(
    df: pd.DataFrame
) -> pd.DataFrame:
    """Create calendar-based features from a DatetimeIndex."""

    df_with_calendar: pd.DataFrame = df.copy()

    df_with_calendar["day_of_week"] = (
        df_with_calendar.index.dayofweek
    )

    df_with_calendar["day_of_month"] = (
        df_with_calendar.index.day
    )

    df_with_calendar["day_of_year"] = (
        df_with_calendar.index.dayofyear
    )

    df_with_calendar["week_of_year"] = (
        df_with_calendar.index.isocalendar().week.astype(int)
    )

    df_with_calendar["month"] = (
        df_with_calendar.index.month
    )

    df_with_calendar["quarter"] = (
        df_with_calendar.index.quarter
    )

    df_with_calendar["year"] = (
        df_with_calendar.index.year
    )

    df_with_calendar["is_weekend"] = (
        df_with_calendar.index.dayofweek >= 5
    ).astype(int)

    df_with_calendar["is_month_start"] = (
        df_with_calendar.index.is_month_start
    ).astype(int)

    df_with_calendar["is_month_end"] = (
        df_with_calendar.index.is_month_end
    ).astype(int)

    return df_with_calendar

### 26.6 Applying and Inspecting Calendar Features

In [ ]:
# Add calendar features
ml_feature_data = add_calendar_features(ml_feature_data)

# Identify the generated calendar columns
calendar_columns: list = [
    "day_of_week",
    "day_of_month",
    "day_of_year",
    "week_of_year",
    "month",
    "quarter",
    "year",
    "is_weekend",
    "is_month_start",
    "is_month_end"
]

print("Calendar features created:")
print(calendar_columns)

display(
    ml_feature_data[calendar_columns].head(10)
)

print("\nMissing values in calendar features:")
display(
    ml_feature_data[calendar_columns]
    .isna()
    .sum()
    .to_frame(name="Missing values")
)

#### Interpretation

The calendar features were created correctly from the datetime index, with no missing values.

For example:

- 1 January 2010 was a Friday, represented by `day_of_week = 4`;
- 2 and 3 January were Saturday and Sunday, so `is_weekend = 1`;
- 1 January is correctly identified by `is_month_start = 1`;
- none of the displayed dates is the last day of the month, so `is_month_end = 0`.

The first three days of January belong to ISO week 53 of the previous ISO calendar cycle, while 4 January begins ISO week 1. This explains why `week_of_year` changes from 53 to 1.

These features contain only information derived from the date. Therefore, they introduce no data leakage and can safely be used with the lag, rolling and Fourier features in the machine-learning model.

### 26.7 Preparing the Machine-Learning Dataset

The Fourier, lag, rolling and calendar variables are now combined into a single feature matrix.

Rows containing missing explanatory variables are removed. These missing values occur at the beginning of the series because the longest feature, `lag_365`, requires 365 previous observations.

The target column is excluded from the explanatory variables to prevent direct target leakage.

The cleaned dataset is then divided chronologically:

- observations up to 31 December 2022 form the training set;
- observations from 1 January 2023 form the test set.

For the 2023 evaluation, lag and rolling features use the actual NO₂ observations available before each prediction date. The model is therefore evaluated using a one-day-ahead forecasting strategy.

In [ ]:
# Combine all explanatory-variable names
feature_columns: list = (
    lag_columns
    + rolling_columns
    + calendar_columns
    + fourier_columns
)

print("Number of features:", len(feature_columns))
print("\nFeatures used:")
print(feature_columns)

# Keep the target and explanatory variables
ml_model_data: pd.DataFrame = (
    ml_feature_data[
        [target_column] + feature_columns
    ]
    .dropna()
    .copy()
)

print("\nDataset shape after removing incomplete rows:")
print(ml_model_data.shape)

print(
    "\nFirst available date:",
    ml_model_data.index.min()
)

print(
    "Last available date:",
    ml_model_data.index.max()
)

# Reproduce the original chronological train-test boundary
train_end_date = train.index.max()
test_start_date = test.index.min()

ml_train_data: pd.DataFrame = (
    ml_model_data.loc[:train_end_date].copy()
)

ml_test_data: pd.DataFrame = (
    ml_model_data.loc[test_start_date:].copy()
)

# Separate explanatory variables and target
X_train_ml: pd.DataFrame = (
    ml_train_data[feature_columns]
)

y_train_ml: pd.Series = (
    ml_train_data[target_column]
)

X_test_ml: pd.DataFrame = (
    ml_test_data[feature_columns]
)

y_test_ml: pd.Series = (
    ml_test_data[target_column]
)

print("\nTraining period:")
print(X_train_ml.index.min(), "to", X_train_ml.index.max())

print("\nTest period:")
print(X_test_ml.index.min(), "to", X_test_ml.index.max())

print("\nTraining shapes:")
print("X_train_ml:", X_train_ml.shape)
print("y_train_ml:", y_train_ml.shape)

print("\nTest shapes:")
print("X_test_ml:", X_test_ml.shape)
print("y_test_ml:", y_test_ml.shape)

print("\nMissing values:")
print("X_train_ml:", X_train_ml.isna().sum().sum())
print("X_test_ml:", X_test_ml.isna().sum().sum())
print("y_train_ml:", y_train_ml.isna().sum())
print("y_test_ml:", y_test_ml.isna().sum())

#### Interpretation

The final machine-learning dataset contains 31 explanatory variables combining lag, rolling, calendar and annual Fourier features.

After removing the initial incomplete rows, 4,747 observations remain. The first usable date is 1 January 2011 because `lag_365`, the longest historical feature, requires 365 previous daily observations.

The chronological split was preserved:

- the training set contains 4,383 observations from 1 January 2011 to 31 December 2022;
- the test set contains 364 observations from 1 January to 30 December 2023.

The feature matrices and target vectors have matching numbers of observations, and no missing values remain in either dataset.

The target variable is not included among the explanatory variables. Lag and rolling features use only previous NO₂ observations, while calendar and Fourier features depend only on the date. Therefore, no direct target leakage is present.

The 2023 test features use the actual NO₂ observations from preceding days. Consequently, this evaluation represents a one-day-ahead forecasting strategy rather than a recursive multi-step forecast.

### 26.8 Training the Initial HistGradientBoostingRegressor

An initial `HistGradientBoostingRegressor` is trained using the complete set of 31 explanatory variables.

This first model uses a reference configuration rather than optimized hyperparameters. Its purpose is to verify that the feature-based forecasting pipeline works correctly and to establish an initial performance level.

The model combines information from:

- previous NO₂ observations through lag features;
- recent averages, variability and ranges through rolling features;
- calendar information;
- annual seasonality represented by Fourier features.

No feature scaling is required because histogram-based decision trees are not sensitive to differences in feature scale.

The 2023 test period remains unseen during training. Predictions are produced using a one-day-ahead forecasting strategy because the lag and rolling features contain actual observations available before each prediction date.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor


# Define the initial machine-learning model
initial_hgbr_model = HistGradientBoostingRegressor(
    learning_rate=0.1,
    max_iter=100,
    max_leaf_nodes=31,
    l2_regularization=0.0,
    random_state=42
)

# Train the model using the 2011–2022 period
initial_hgbr_model.fit(
    X_train_ml,
    y_train_ml
)

# Generate one-day-ahead predictions for 2023
initial_hgbr_predictions: np.ndarray = (
    initial_hgbr_model.predict(X_test_ml)
)

print("Model training completed.")
print("Number of predictions:", len(initial_hgbr_predictions))

display(
    pd.DataFrame(
        {
            "Actual NO₂": y_test_ml,
            "Predicted NO₂": initial_hgbr_predictions
        },
        index=y_test_ml.index
    ).head(10)
)

#### Initial Results

The initial `HistGradientBoostingRegressor` was trained successfully and generated 364 predictions, matching the size of the 2023 test set.

The first predictions generally follow the range of the observed NO₂ concentrations, but several substantial differences are visible. The model appears to overestimate some low-concentration days and does not always capture abrupt daily variations.

These first rows alone are not sufficient to assess overall forecasting performance. MAE, RMSE and MAPE will therefore be calculated across the complete test period.

### 26.9 Evaluating the Initial HistGradientBoostingRegressor

The initial machine-learning model is evaluated on the complete unseen 2023 test period.

The same metrics used for the previous forecasting models are calculated:

- MAE measures the average absolute prediction error;
- RMSE gives more importance to large errors;
- MAPE expresses the average error as a percentage of the actual concentration.

These results provide an initial reference for the feature-based machine-learning approach. The model has not yet been optimized; time-based cross-validation and hyperparameter tuning will be performed in the following sections.

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)


# Calculate evaluation metrics
initial_hgbr_mae: float = mean_absolute_error(
    y_test_ml,
    initial_hgbr_predictions
)

initial_hgbr_rmse: float = np.sqrt(
    mean_squared_error(
        y_test_ml,
        initial_hgbr_predictions
    )
)

initial_hgbr_mape: float = (
    mean_absolute_percentage_error(
        y_test_ml,
        initial_hgbr_predictions
    )
    * 100
)

# Store the results
initial_hgbr_metrics: pd.DataFrame = pd.DataFrame(
    {
        "Model": ["Initial HistGradientBoostingRegressor"],
        "MAE": [initial_hgbr_mae],
        "RMSE": [initial_hgbr_rmse],
        "MAPE (%)": [initial_hgbr_mape]
    }
)

display(
    initial_hgbr_metrics.round(3)
)

In [ ]:
# Create a prediction series with the correct datetime index
initial_hgbr_prediction_series: pd.Series = pd.Series(
    initial_hgbr_predictions,
    index=y_test_ml.index,
    name="Predicted NO₂"
)

# Plot actual and predicted values
plt.figure(figsize=(15, 6))

plt.plot(
    y_test_ml.index,
    y_test_ml,
    label="Actual NO₂",
    color="black",
    linewidth=1.5
)

plt.plot(
    initial_hgbr_prediction_series.index,
    initial_hgbr_prediction_series,
    label="Initial HistGradientBoostingRegressor",
    color="tab:green",
    linewidth=1.3,
    alpha=0.85
)

plt.title(
    "Actual vs Predicted Daily NO₂ Concentrations — 2023"
)
plt.xlabel("Date")
plt.ylabel("Daily mean NO₂ concentration")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

#### Interpretation

The initial `HistGradientBoostingRegressor` achieved:

- MAE: 5.868;
- RMSE: 7.986;
- MAPE: 51.511%.

The MAE indicates that the predicted daily NO₂ concentration differs from the actual value by approximately 5.87 units on average.

The RMSE is higher than the MAE, showing that some days contain relatively large prediction errors. This is visible during abrupt pollution peaks, particularly at the beginning of the year and during autumn and winter.

The predicted series follows the broad evolution of the observed concentrations. It captures the lower levels during spring and summer and the generally higher and more variable concentrations during winter.

However, the predictions are smoother than the actual series. The model tends to underestimate sharp peaks and overestimate some very low concentrations. This suggests that the existing lag, rolling, calendar and Fourier features capture the general temporal structure more effectively than sudden daily fluctuations.

The MAPE of 51.51% appears high. This metric must be interpreted cautiously because several actual NO₂ concentrations are close to zero. On these days, even a moderate absolute error produces a very large percentage error. For this dataset, MAE and RMSE are therefore more reliable for comparing models.

These results represent an initial, non-optimized configuration. The model should not yet be selected using the 2023 test performance. Time-based cross-validation on the training period will be used next to evaluate its stability and tune its hyperparameters while preserving the test set for final assessment.


The initial model successfully captures the broad temporal behaviour of daily NO₂ concentrations but struggles with abrupt peaks and very low values.

The next step is time-based cross-validation using only the training period.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Conceptual expanding-window cross-validation diagram
n_folds: int = 5
initial_train_size: int = 5
validation_size: int = 1

fig, ax = plt.subplots(figsize=(14, 6))

for fold in range(n_folds):
    train_end: int = initial_train_size + fold
    validation_start: int = train_end

    # Expanding training window
    ax.barh(
        y=fold,
        width=train_end,
        left=0,
        height=0.55,
        color="steelblue"
    )

    # Following validation period
    ax.barh(
        y=fold,
        width=validation_size,
        left=validation_start,
        height=0.55,
        color="darkorange"
    )

ax.set_yticks(range(n_folds))
ax.set_yticklabels([f"Fold {i}" for i in range(1, n_folds + 1)])
ax.invert_yaxis()

ax.set_xlabel("Chronological progression of the time series")
ax.set_ylabel("Cross-validation fold")
ax.set_title("Expanding-Window Time-Based Cross-Validation")

# Hide abstract numerical values
ax.set_xticks([])
ax.grid(False)

legend_elements = [
    Patch(facecolor="steelblue", label="Training observations"),
    Patch(facecolor="darkorange", label="Validation period")
]

ax.legend(
    handles=legend_elements,
    loc="lower right"
)

fig.tight_layout()
plt.show()

## 27. Time-Based Cross-Validation for HistGradientBoostingRegressor

The initial `HistGradientBoostingRegressor` was evaluated once on the 2023 test period. However, a single train-test split does not show whether the model performs consistently across different periods.

Time-based cross-validation evaluates the model through several chronological training and validation windows.

Unlike standard random cross-validation, observations are never shuffled because doing so would mix past and future information. For every fold:

- the training period occurs before the validation period;
- the training window expands progressively;
- the validation observations remain chronologically later;
- the final 2023 test set is excluded entirely.

The cross-validation process therefore reproduces a realistic forecasting situation in which only historical observations are available when predicting a future period.

`TimeSeriesSplit` will be used to create five chronological folds from the 2011–2022 training dataset. MAE, RMSE and MAPE will be calculated for each fold to evaluate both average performance and model stability.

### 27.1 Creating the Time-Based Cross-Validation Splits

Five expanding-window folds are created using `TimeSeriesSplit`.

In each successive fold:

- the training set becomes larger;
- the validation set contains observations occurring after the training set;
- no future observation is used to train a model that predicts the past.

Only the 2011–2022 training dataset is used. The 2023 test period remains untouched for the final evaluation of the selected model.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit


# Define five chronological cross-validation folds
hgbr_time_series_cv = TimeSeriesSplit(n_splits=5)

# Store information about each fold
cv_fold_periods: list = []

for fold_number, (train_indices, validation_indices) in enumerate(
    hgbr_time_series_cv.split(X_train_ml),
    start=1
):
    fold_train_dates = X_train_ml.index[train_indices]
    fold_validation_dates = X_train_ml.index[validation_indices]

    cv_fold_periods.append(
        {
            "Fold": fold_number,
            "Training start": fold_train_dates.min(),
            "Training end": fold_train_dates.max(),
            "Training observations": len(train_indices),
            "Validation start": fold_validation_dates.min(),
            "Validation end": fold_validation_dates.max(),
            "Validation observations": len(validation_indices)
        }
    )

# Display the chronological structure of the folds
cv_fold_periods_df: pd.DataFrame = pd.DataFrame(cv_fold_periods)

display(cv_fold_periods_df)

#### Interpretation

The five time-based cross-validation folds were created correctly using an expanding training window.

The first training period contains 733 observations, from 1 January 2011 to 2 January 2013. In each subsequent fold, 730 additional observations are added to the training set.

Each validation window contains 730 observations, corresponding to approximately two years:

- Fold 1 validates from 3 January 2013 to 2 January 2015;
- Fold 2 validates from 3 January 2015 to 1 January 2017;
- Fold 3 validates from 2 January 2017 to 1 January 2019;
- Fold 4 validates from 2 January 2019 to 31 December 2020;
- Fold 5 validates from 1 January 2021 to 31 December 2022.

For every fold, the training period ends before the validation period begins. Therefore, no future observation is used to train a model that predicts an earlier date.

The 2023 test period is not included in any fold and remains excluded from cross-validation and hyperparameter tuning.

### 27.2 Evaluating the Initial Model Across the Five Folds

The initial `HistGradientBoostingRegressor` configuration is now evaluated across the five chronological folds.

For each fold:

1. a new model is trained using only the corresponding training window;
2. predictions are generated for the subsequent validation window;
3. MAE, RMSE and MAPE are calculated;
4. the model is discarded before moving to the next fold.

The model is therefore retrained independently in every fold.

The validation features contain actual NO₂ observations from preceding dates. Consequently, the evaluation follows the same one-day-ahead forecasting strategy with observed historical updates used in the preliminary 2023 evaluation.

The 2023 test period remains excluded from this cross-validation process.

In [ ]:
# Store the performance of each cross-validation fold
initial_hgbr_cv_results: list = []

for fold_number, (train_indices, validation_indices) in enumerate(
    hgbr_time_series_cv.split(X_train_ml),
    start=1
):
    # Create the chronological fold datasets
    X_fold_train: pd.DataFrame = X_train_ml.iloc[train_indices]
    y_fold_train: pd.Series = y_train_ml.iloc[train_indices]

    X_fold_validation: pd.DataFrame = (
        X_train_ml.iloc[validation_indices]
    )
    y_fold_validation: pd.Series = (
        y_train_ml.iloc[validation_indices]
    )

    # Create a new model for each fold
    fold_hgbr_model = HistGradientBoostingRegressor(
        learning_rate=0.1,
        max_iter=100,
        max_leaf_nodes=31,
        l2_regularization=0.0,
        random_state=42
    )

    # Train the model on the fold training period
    fold_hgbr_model.fit(
        X_fold_train,
        y_fold_train
    )

    # Predict the subsequent validation period
    fold_predictions: np.ndarray = (
        fold_hgbr_model.predict(X_fold_validation)
    )

    # Calculate fold metrics
    fold_mae: float = mean_absolute_error(
        y_fold_validation,
        fold_predictions
    )

    fold_rmse: float = np.sqrt(
        mean_squared_error(
            y_fold_validation,
            fold_predictions
        )
    )

    fold_mape: float = (
        mean_absolute_percentage_error(
            y_fold_validation,
            fold_predictions
        )
        * 100
    )

    # Store the fold results
    initial_hgbr_cv_results.append(
        {
            "Fold": fold_number,
            "Training end": X_fold_train.index.max(),
            "Validation start": X_fold_validation.index.min(),
            "Validation end": X_fold_validation.index.max(),
            "MAE": fold_mae,
            "RMSE": fold_rmse,
            "MAPE (%)": fold_mape
        }
    )

# Convert the results to a DataFrame
initial_hgbr_cv_results_df: pd.DataFrame = pd.DataFrame(
    initial_hgbr_cv_results
)

display(
    initial_hgbr_cv_results_df.round(3)
)

In [ ]:
# Summarize average performance and variability
initial_hgbr_cv_summary: pd.DataFrame = pd.DataFrame(
    {
        "Metric": ["MAE", "RMSE", "MAPE (%)"],
        "Mean": [
            initial_hgbr_cv_results_df["MAE"].mean(),
            initial_hgbr_cv_results_df["RMSE"].mean(),
            initial_hgbr_cv_results_df["MAPE (%)"].mean()
        ],
        "Standard deviation": [
            initial_hgbr_cv_results_df["MAE"].std(),
            initial_hgbr_cv_results_df["RMSE"].std(),
            initial_hgbr_cv_results_df["MAPE (%)"].std()
        ],
        "Minimum": [
            initial_hgbr_cv_results_df["MAE"].min(),
            initial_hgbr_cv_results_df["RMSE"].min(),
            initial_hgbr_cv_results_df["MAPE (%)"].min()
        ],
        "Maximum": [
            initial_hgbr_cv_results_df["MAE"].max(),
            initial_hgbr_cv_results_df["RMSE"].max(),
            initial_hgbr_cv_results_df["MAPE (%)"].max()
        ]
    }
)

display(
    initial_hgbr_cv_summary.round(3)
)

#### Interpretation

The initial `HistGradientBoostingRegressor` obtained an average cross-validation performance of:

- MAE: 8.501 ± 1.269;
- RMSE: 10.962 ± 1.719;
- MAPE: 39.940% ± 9.775 percentage points.

The MAE decreases progressively from 9.739 in Fold 1 to 6.563 in Fold 5. Similarly, the RMSE decreases from 12.497 to 8.457. This indicates that the model performs better in absolute terms during the more recent validation periods.

Several factors may explain this improvement. The expanding training window provides the later models with more historical observations, while changes in the level and variability of NO₂ concentrations across the years may also make the recent periods easier to predict.

The RMSE is consistently higher than the MAE in every fold. This confirms that the model produces some relatively large errors, particularly when abrupt NO₂ peaks occur.

In contrast, MAPE increases from 31.725% in Fold 1 to 54.404% in Fold 5, despite the improvement in MAE and RMSE. This apparent contradiction is likely caused by lower actual NO₂ concentrations in the more recent periods. When the observed concentration is close to zero, even a moderate absolute error produces a large percentage error.

The variability across folds is moderate for MAE and RMSE but higher for MAPE. Therefore, MAE and RMSE provide more reliable indicators of model stability for this dataset, while MAPE should be interpreted cautiously.

Overall, the model does not perform equally across all historical periods. Nevertheless, its absolute prediction accuracy improves in the later folds.

#### Comparison with the Preliminary 2023 Evaluation

The average cross-validation errors are higher than the preliminary 2023 test errors:

- cross-validation mean MAE: 8.501, compared with 5.868 in 2023;
- cross-validation mean RMSE: 10.962, compared with 7.986 in 2023;
- cross-validation mean MAPE: 39.940%, compared with 51.511% in 2023.

The lower 2023 MAE and RMSE are consistent with the improvement observed in the most recent cross-validation folds. However, the higher 2023 MAPE again suggests that low observed concentrations amplify percentage errors.

These results show why a single test period is insufficient to assess model robustness. Time-based cross-validation reveals substantial variation in performance across historical periods.

## 28. Hyperparameter Tuning and Selection of the Best Machine-Learning Model

### 28.1 Selecting the Optimization Metric

The cross-validation results show that MAPE is highly sensitive to low NO₂ concentrations. Therefore, it is not selected as the primary metric for hyperparameter tuning.

MAE is used as the optimization metric because:

- it measures the average prediction error in the original NO₂ unit;
- it is less influenced by unusually large errors than RMSE;
- it remains interpretable when actual concentrations are close to zero;
- it is suitable for comparing performance across chronological folds.

RMSE and MAPE will still be reported as complementary evaluation metrics.

In scikit-learn, the MAE scoring function is represented by `neg_mean_absolute_error`. Scores are returned as negative values because the model-selection tools always attempt to maximize the score. The absolute MAE is therefore obtained by multiplying the reported score by `-1`.

Hyperparameter combinations will be evaluated exclusively on the 2011–2022 training period using the same five chronological folds. The 2023 test data will not participate in the optimization process.

In [ ]:
# Define the primary optimization metric
hgbr_scoring: str = "neg_mean_absolute_error"

print("Optimization metric:", hgbr_scoring)
print(
    "The best model will be the configuration "
    "with the lowest cross-validated MAE."
)

### 28.2 Defining the Hyperparameter Grid

A compact hyperparameter grid is defined for the `HistGradientBoostingRegressor`.

The following hyperparameters are explored:

- `learning_rate`: controls the contribution of each successive tree;
- `max_iter`: determines the maximum number of boosting iterations;
- `max_leaf_nodes`: controls the complexity of each tree;
- `min_samples_leaf`: specifies the minimum number of observations required in a leaf;
- `l2_regularization`: penalizes overly complex models and may reduce overfitting.

The grid is deliberately limited to a reasonable number of combinations. Each configuration will be evaluated using the five chronological cross-validation folds, which makes an excessively large grid computationally expensive.

The 2023 test set remains excluded from the search.

In [ ]:
from sklearn.model_selection import ParameterGrid


# Define a compact hyperparameter grid
hgbr_parameter_grid: dict = {
    "learning_rate": [0.05, 0.1],
    "max_iter": [100, 200],
    "max_leaf_nodes": [15, 31],
    "min_samples_leaf": [20, 40],
    "l2_regularization": [0.0, 1.0]
}

# Count the number of configurations
number_of_configurations: int = len(
    list(ParameterGrid(hgbr_parameter_grid))
)

number_of_model_fits: int = (
    number_of_configurations
    * hgbr_time_series_cv.get_n_splits()
)

print("Hyperparameter grid:")
for parameter, values in hgbr_parameter_grid.items():
    print(f"{parameter}: {values}")

print(
    "\nNumber of hyperparameter configurations:",
    number_of_configurations
)

print(
    "Number of cross-validation model fits:",
    number_of_model_fits
)

#### Grid Size

The grid contains 32 hyperparameter configurations. With five chronological folds, the search requires 160 separate model fits.

This grid explores different compromises between learning speed, model complexity, regularization and training duration while keeping the computational cost manageable.

### 28.3 Running the Time-Based Hyperparameter Search

`GridSearchCV` is used to evaluate the 32 hyperparameter configurations.

Each configuration is assessed using the same five chronological folds and MAE as the optimization metric. This results in 160 model fits.

The model with the lowest average validation MAE is selected. The 2023 test set remains excluded from the search.

`refit=True` means that, after identifying the best configuration, `GridSearchCV` automatically retrains it using the complete 2011–2022 training dataset.

In [ ]:
from sklearn.model_selection import GridSearchCV


# Define the base model
hgbr_base_model = HistGradientBoostingRegressor(
    random_state=42
)

# Configure the time-based grid search
hgbr_grid_search = GridSearchCV(
    estimator=hgbr_base_model,
    param_grid=hgbr_parameter_grid,
    scoring=hgbr_scoring,
    cv=hgbr_time_series_cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True,
    verbose=1
)

# Run the hyperparameter search using only 2011–2022
hgbr_grid_search.fit(
    X_train_ml,
    y_train_ml
)

print("Hyperparameter search completed.")

print("\nBest hyperparameters:")
print(hgbr_grid_search.best_params_)

best_cv_mae: float = -hgbr_grid_search.best_score_

print(
    "\nBest mean cross-validated MAE:",
    round(best_cv_mae, 3)
)

### 28.4 Hyperparameter Search Results

The time-based grid search evaluated 32 hyperparameter configurations across five chronological folds, resulting in 160 model fits.

The best configuration was:

- learning rate: 0.05;
- maximum iterations: 100;
- maximum leaf nodes: 15;
- minimum samples per leaf: 20;
- L2 regularization: 0.0.

This configuration achieved a mean cross-validated MAE of 8.134.

The initial configuration obtained a mean cross-validated MAE of 8.501. Hyperparameter tuning therefore reduced the average validation MAE by 0.367, corresponding to an improvement of approximately 4.3%.

The selected model uses a lower learning rate and fewer leaf nodes than the initial model. This suggests that a slower and less complex boosting process generalizes better across the chronological validation periods.

The absence of L2 regularization in the best configuration indicates that additional regularization did not improve the average validation MAE within the tested grid. However, this conclusion applies only to the values explored in this search.

The 2023 test data were not used during the hyperparameter search. The selected configuration was determined exclusively from the 2011–2022 training period.

In [ ]:
# Convert all search results into a DataFrame
hgbr_search_results: pd.DataFrame = pd.DataFrame(
    hgbr_grid_search.cv_results_
)

# Convert negative MAE scores into positive errors
hgbr_search_results["Mean validation MAE"] = (
    -hgbr_search_results["mean_test_score"]
)

hgbr_search_results["Validation MAE std"] = (
    hgbr_search_results["std_test_score"]
)

hgbr_search_results["Mean training MAE"] = (
    -hgbr_search_results["mean_train_score"]
)

# Select and rank the ten best configurations
top_hgbr_configurations: pd.DataFrame = (
    hgbr_search_results[
        [
            "rank_test_score",
            "param_learning_rate",
            "param_max_iter",
            "param_max_leaf_nodes",
            "param_min_samples_leaf",
            "param_l2_regularization",
            "Mean training MAE",
            "Mean validation MAE",
            "Validation MAE std"
        ]
    ]
    .sort_values("rank_test_score")
    .head(10)
    .rename(
        columns={
            "rank_test_score": "Rank",
            "param_learning_rate": "Learning rate",
            "param_max_iter": "Max iterations",
            "param_max_leaf_nodes": "Max leaf nodes",
            "param_min_samples_leaf": "Min samples leaf",
            "param_l2_regularization": "L2 regularization"
        }
    )
)

display(top_hgbr_configurations.round(3))

### 28.5 Interpretation of the Top Configurations

The best configuration achieved a mean validation MAE of 8.134, with a standard deviation of 0.980 across the five chronological folds.

The second-best configuration differs only by the addition of L2 regularization and obtains a slightly higher validation MAE of 8.170. This confirms that L2 regularization did not improve performance for the selected model structure.

Most of the ten best configurations use:

- 100 boosting iterations;
- 15 maximum leaf nodes;
- a learning rate of 0.05 or 0.10.

This indicates that relatively small trees generalize better than the more complex initial configuration with 31 leaf nodes.

Increasing the number of iterations from 100 to 200 substantially reduces the training MAE. For example, one configuration reaches a training MAE of 5.387, compared with 6.514 for the selected model. However, its validation MAE remains higher at 8.192.

The larger difference between training and validation performance suggests that additional boosting iterations improve the fit to the training data without improving generalization. This is evidence of increased overfitting.

The selected configuration provides the best compromise between predictive accuracy, model complexity and stability. Its validation MAE standard deviation of 0.980 is also the lowest among the displayed configurations, indicating relatively consistent performance across the five chronological folds.

### 28.6 Selected Hyperparameters

Based on the lowest mean time-based cross-validation MAE, the selected configuration is:

- `learning_rate = 0.05`;
- `max_iter = 100`;
- `max_leaf_nodes = 15`;
- `min_samples_leaf = 20`;
- `l2_regularization = 0.0`.

This tuned model will now be evaluated on the 2023 test period and compared with the initial `HistGradientBoostingRegressor`.

### 28.7 Evaluating the Tuned HistGradientBoostingRegressor on 2023

After hyperparameter tuning, the selected `HistGradientBoostingRegressor` is evaluated on the 2023 test period.

The selected model was determined exclusively through time-based cross-validation on the 2011–2022 training data. It was then automatically refitted on the complete training period by `GridSearchCV`.

Its 2023 performance is compared with that of the initial model using:

- MAE;
- RMSE;
- MAPE.

Because the initial model results for 2023 were already examined, this comparison should be interpreted as an assessment against the preliminary test benchmark rather than a completely untouched final model-selection step.

The evaluation continues to represent one-day-ahead forecasting with observed historical updates.

In [ ]:
# Retrieve the selected model refitted on the complete training set
tuned_hgbr_model = hgbr_grid_search.best_estimator_

# Generate predictions for the 2023 test period
tuned_hgbr_predictions: np.ndarray = (
    tuned_hgbr_model.predict(X_test_ml)
)

# Calculate test metrics
tuned_hgbr_mae: float = mean_absolute_error(
    y_test_ml,
    tuned_hgbr_predictions
)

tuned_hgbr_rmse: float = np.sqrt(
    mean_squared_error(
        y_test_ml,
        tuned_hgbr_predictions
    )
)

tuned_hgbr_mape: float = (
    mean_absolute_percentage_error(
        y_test_ml,
        tuned_hgbr_predictions
    )
    * 100
)

# Compare the initial and tuned models
hgbr_test_comparison: pd.DataFrame = pd.DataFrame(
    {
        "Model": [
            "Initial HistGradientBoostingRegressor",
            "Tuned HistGradientBoostingRegressor"
        ],
        "MAE": [
            initial_hgbr_mae,
            tuned_hgbr_mae
        ],
        "RMSE": [
            initial_hgbr_rmse,
            tuned_hgbr_rmse
        ],
        "MAPE (%)": [
            initial_hgbr_mape,
            tuned_hgbr_mape
        ]
    }
)

display(hgbr_test_comparison.round(3))

In [ ]:
# Calculate the percentage change relative to the initial model
hgbr_metric_changes: pd.DataFrame = pd.DataFrame(
    {
        "Metric": ["MAE", "RMSE", "MAPE"],
        "Initial model": [
            initial_hgbr_mae,
            initial_hgbr_rmse,
            initial_hgbr_mape
        ],
        "Tuned model": [
            tuned_hgbr_mae,
            tuned_hgbr_rmse,
            tuned_hgbr_mape
        ]
    }
)

hgbr_metric_changes["Change (%)"] = (
    (
        hgbr_metric_changes["Tuned model"]
        - hgbr_metric_changes["Initial model"]
    )
    / hgbr_metric_changes["Initial model"]
    * 100
)

display(hgbr_metric_changes.round(3))

### 28.8 Comparing the Initial and Tuned Models

The tuned `HistGradientBoostingRegressor` achieved the following performance on the 2023 test period:

- MAE: 5.913;
- RMSE: 8.152;
- MAPE: 54.240%.

Compared with the initial model, the tuned model produced:

- a 0.773% increase in MAE;
- a 2.078% increase in RMSE;
- a 5.298% increase in MAPE.

Because all three percentage changes are positive, the tuned model performed slightly worse than the initial model on the 2023 test period.

This result does not mean that the hyperparameter search failed. The selected configuration reduced the mean cross-validated MAE from 8.501 to 8.134 across the five historical validation periods. It therefore generalized better on average during time-based cross-validation, but this improvement did not transfer to the specific conditions observed in 2023.

The larger increase in RMSE compared with MAE suggests that the tuned model made slightly larger errors on some difficult days, particularly during abrupt pollution peaks. The increase in MAPE must again be interpreted cautiously because this metric is highly sensitive to actual NO₂ concentrations close to zero.

These results illustrate that improved average cross-validation performance does not guarantee improved performance on every future period. Temporal changes in the level, variability and dynamics of NO₂ concentrations may explain why the configuration selected from 2011–2022 did not outperform the initial model in 2023.

#### Model-Selection Decision

The tuned model remains the configuration selected through the predefined time-based cross-validation procedure because its hyperparameters were chosen using only the 2011–2022 training period.

However, the initial model achieved slightly better results on the preliminary 2023 test benchmark across MAE, RMSE and MAPE.

This distinction should be reported transparently:

- the tuned model performed better on average across historical cross-validation folds;
- the initial model performed better on the specific 2023 test period.

Therefore, hyperparameter tuning improved historical validation performance but did not improve the final 2023 forecasting results.

### 28.9 Visualizing the Tuned Model Predictions

The predictions produced by the tuned `HistGradientBoostingRegressor` are now compared visually with the actual daily NO₂ concentrations observed in 2023.

This visualization helps assess whether the model captures:

- the general evolution of NO₂ concentrations;
- seasonal differences across the year;
- periods of low pollution;
- abrupt pollution peaks and daily fluctuations.

The initial model predictions are also included to show how strongly hyperparameter tuning changed the forecasting behaviour.

In [ ]:
# Create a series for the tuned predictions
tuned_hgbr_prediction_series: pd.Series = pd.Series(
    tuned_hgbr_predictions,
    index=y_test_ml.index,
    name="Tuned model"
)

# Plot actual values and both model predictions
plt.figure(figsize=(15, 6))

plt.plot(
    y_test_ml.index,
    y_test_ml,
    label="Actual NO₂",
    color="black",
    linewidth=1.5
)

plt.plot(
    initial_hgbr_prediction_series.index,
    initial_hgbr_prediction_series,
    label="Initial model",
    color="tab:green",
    linewidth=1.1,
    alpha=0.65
)

plt.plot(
    tuned_hgbr_prediction_series.index,
    tuned_hgbr_prediction_series,
    label="Tuned model",
    color="tab:orange",
    linewidth=1.2,
    alpha=0.85
)

plt.title(
    "Actual vs Initial and Tuned HistGradientBoosting Predictions — 2023"
)
plt.xlabel("Date")
plt.ylabel("Daily mean NO₂ concentration")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

#### Interpretation

The initial and tuned `HistGradientBoostingRegressor` predictions follow very similar trajectories throughout 2023. The two predicted curves frequently overlap, indicating that hyperparameter tuning produced only limited changes in the model's forecasting behaviour.

Both models capture the broad temporal pattern of daily NO₂ concentrations:

- higher and more variable concentrations during winter;
- generally lower concentrations during spring and summer;
- a renewed increase in concentrations and variability during autumn.

However, both prediction series are considerably smoother than the actual observations. They capture the general evolution of the series more effectively than abrupt daily fluctuations.

The models tend to underestimate several sharp pollution peaks, particularly during January and February, around early September, and during November and December. They also overestimate some very low concentrations, especially during late spring and summer.

The tuned model occasionally produces slightly lower or smoother predictions than the initial model. This may be explained by its lower learning rate and smaller maximum number of leaf nodes, which reduce model complexity. Although this simpler structure improved the mean cross-validated MAE, it did not improve performance on the specific 2023 test period.

The limited visual difference between the two models is consistent with their similar MAE values:

- initial model MAE: 5.868;
- tuned model MAE: 5.913.

Overall, hyperparameter tuning did not fundamentally change the forecasting behaviour. The tuned configuration remains the model selected through time-based cross-validation, while the initial configuration achieved slightly better observed performance in 2023.

The main limitation of both models is their tendency to smooth predictions, underestimate abrupt peaks and overestimate very low concentrations. Improving peak prediction may require additional explanatory variables, such as weather conditions, traffic activity or information about unusual pollution events.

### 28.10 Analyzing the Tuned Model Residuals

Residual analysis is used to examine the errors produced by the tuned `HistGradientBoostingRegressor`.

A residual is calculated as:

\[
\text{Residual}_t = \text{Actual NO₂}_t - \text{Predicted NO₂}_t
\]

Therefore:

- a positive residual indicates that the model underestimated the actual concentration;
- a negative residual indicates that the model overestimated the actual concentration;
- a residual close to zero indicates an accurate prediction.

A well-performing forecasting model should ideally produce residuals that fluctuate randomly around zero without a clear temporal pattern.

The residual time series and its distribution are examined to identify systematic bias, extreme errors and periods during which the model performs less effectively.

In [ ]:
# Calculate the tuned model residuals
tuned_hgbr_residuals: pd.Series = pd.Series(
    y_test_ml.values - tuned_hgbr_predictions,
    index=y_test_ml.index,
    name="Residual"
)

# Create residual diagnostic plots
fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(15, 10)
)

# Residuals over time
axes[0].plot(
    tuned_hgbr_residuals.index,
    tuned_hgbr_residuals,
    color="tab:blue",
    linewidth=1
)

axes[0].axhline(
    y=0,
    color="black",
    linestyle="--",
    linewidth=1
)

axes[0].set_title(
    "Tuned HistGradientBoostingRegressor Residuals — 2023"
)
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Residual")
axes[0].grid(alpha=0.3)

# Residual distribution
sns.histplot(
    tuned_hgbr_residuals,
    bins=30,
    kde=True,
    color="tab:blue",
    ax=axes[1]
)

axes[1].axvline(
    x=0,
    color="black",
    linestyle="--",
    linewidth=1
)

axes[1].set_title("Distribution of Residuals")
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Frequency")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Summarize the residual distribution
tuned_hgbr_residual_summary: pd.DataFrame = pd.DataFrame(
    {
        "Statistic": [
            "Mean",
            "Median",
            "Standard deviation",
            "Minimum",
            "Maximum"
        ],
        "Value": [
            tuned_hgbr_residuals.mean(),
            tuned_hgbr_residuals.median(),
            tuned_hgbr_residuals.std(),
            tuned_hgbr_residuals.min(),
            tuned_hgbr_residuals.max()
        ]
    }
)

display(tuned_hgbr_residual_summary.round(3))

#### Interpretation and Conclusion

The tuned `HistGradientBoostingRegressor` residuals fluctuate on both sides of zero throughout 2023, without a persistent long-term trend. However, they are not perfectly centred around zero:

- mean residual: -1.258;
- median residual: -1.879;
- standard deviation: 8.065;
- minimum residual: -35.644;
- maximum residual: 43.498.

Because residuals are calculated as actual values minus predictions, the negative mean and median indicate that the model has a slight overall tendency to overestimate daily NO₂ concentrations.

The residual standard deviation of 8.065 shows that prediction errors remain relatively dispersed around their average. This is consistent with the test RMSE of 8.152.

The residual variation is not constant throughout the year. Particularly large errors occur during January and February, while another period of increased variability appears during autumn and early winter. The residuals are generally smaller and more stable during spring and summer.

The maximum positive residual of 43.498 represents a substantial underestimation of an observed pollution peak. Conversely, the minimum residual of -35.644 represents a strong overestimation, probably on a day when the actual concentration was unusually low.

The residual distribution is concentrated mostly between approximately -10 and 10 but is not perfectly symmetric or normally distributed. Its positive tail is extended by several underestimated pollution peaks. The mean is consequently less negative than the median because these large positive residuals pull it towards the right.

Overall, the tuned model captures the general evolution of daily NO₂ concentrations but its residuals do not behave like ideal random noise. The slight negative bias, extreme errors and changing variability over time indicate that some temporal structure and unusual pollution events remain unexplained.

These findings confirm the main limitation observed in the prediction plot: the model tends to smooth abrupt daily variations, underestimate major pollution peaks and overestimate some unusually low concentrations. Additional explanatory variables, such as weather conditions, traffic activity and exceptional pollution events, could help explain these remaining errors.

### 28.11 Checking Residual Autocorrelation

The residual time series and its distribution revealed a slight negative bias, several extreme errors and changing variability throughout 2023.

The next step is to determine whether the residuals remain correlated over time.

If significant autocorrelation is present, prediction errors from neighbouring days are related. This would indicate that the model has not captured all the temporal information contained in the NO₂ series.

Two complementary diagnostics are used:

- the residual autocorrelation function (ACF), which displays correlation at individual lags;
- the Ljung–Box test, which jointly tests whether autocorrelations up to selected lags are equal to zero.

For the Ljung–Box test:

- the null hypothesis states that the residuals are independently distributed up to the tested lag;
- a p-value below 0.05 leads to rejection of the null hypothesis and suggests remaining autocorrelation;
- a p-value greater than or equal to 0.05 provides insufficient evidence of residual autocorrelation.

The tests are applied to the 2023 residuals of the tuned model.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox


# Plot the residual autocorrelation function
fig, ax = plt.subplots(figsize=(14, 5))

plot_acf(
    tuned_hgbr_residuals.dropna(),
    lags=40,
    alpha=0.05,
    zero=False,
    ax=ax
)

ax.set_title(
    "ACF of Tuned HistGradientBoostingRegressor Residuals — 2023"
)
ax.set_xlabel("Lag (days)")
ax.set_ylabel("Autocorrelation")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Test residual autocorrelation at selected horizons
ljung_box_lags: list[int] = [7, 14, 30]

tuned_hgbr_ljung_box: pd.DataFrame = acorr_ljungbox(
    tuned_hgbr_residuals.dropna(),
    lags=ljung_box_lags,
    return_df=True
)

tuned_hgbr_ljung_box.index.name = "Lag"

tuned_hgbr_ljung_box = tuned_hgbr_ljung_box.rename(
    columns={
        "lb_stat": "Ljung-Box statistic",
        "lb_pvalue": "p-value"
    }
)

display(tuned_hgbr_ljung_box.round(4))

In [ ]:
# Interpret the Ljung-Box results at the 5% significance level
ljung_box_interpretation: pd.DataFrame = (
    tuned_hgbr_ljung_box.copy()
)

ljung_box_interpretation["Decision at 5%"] = np.where(
    ljung_box_interpretation["p-value"] < 0.05,
    "Reject H0: residual autocorrelation remains",
    "Do not reject H0: no significant evidence of autocorrelation"
)

display(ljung_box_interpretation)

#### Interpretation and Conclusion

The residual ACF shows that most autocorrelation coefficients remain within the 95% confidence interval and are close to zero.

A few isolated lags, particularly around lag 35, approach or slightly exceed the confidence limits. However, an isolated ACF spike does not necessarily indicate a systematic temporal pattern, especially when many lags are examined.

The Ljung–Box test provides the following p-values:

- lag 7: 0.2705;
- lag 14: 0.0660;
- lag 30: 0.0646.

All p-values are greater than the 5% significance threshold. Therefore, the null hypothesis of no residual autocorrelation is not rejected at any of the three tested horizons.

The results provide no statistically significant evidence that the residuals remain jointly autocorrelated over periods of 7, 14 or 30 days. This suggests that the lag, rolling, calendar and Fourier features captured most of the systematic temporal dependence available in the daily NO₂ history.

However, the p-values at lags 14 and 30 are relatively close to 0.05. Therefore, some weak residual dependence may remain, although the evidence is insufficient to declare it statistically significant at the selected threshold.

Overall, the residuals are largely uncorrelated over time, which is a positive diagnostic result. Nevertheless, they are not ideal white noise because the previous analysis revealed a slight negative bias, extreme errors and non-constant variability.

The model therefore captures the temporal dependence reasonably well, while its main remaining limitations concern prediction bias, abrupt pollution peaks and changing error variance rather than strong residual autocorrelation.

### 28.12 Final Assessment of the Tuned Model

Time-based cross-validation and hyperparameter tuning were used to select a `HistGradientBoostingRegressor` without using the 2023 test period during model selection.

The selected hyperparameters were:

- `learning_rate = 0.05`;
- `max_iter = 100`;
- `max_leaf_nodes = 15`;
- `min_samples_leaf = 20`;
- `l2_regularization = 0.0`.

The tuned model reduced the mean cross-validated MAE from 8.501 to 8.134, corresponding to an improvement of approximately 4.3% across the five historical validation periods.

However, this improvement did not transfer to the 2023 test period. The tuned model obtained:

- MAE: 5.913;
- RMSE: 8.152;
- MAPE: 54.240%.

The initial model performed slightly better in 2023, with an MAE of 5.868 and an RMSE of 7.986. Nevertheless, the tuned model remains the configuration selected through the predefined cross-validation procedure because choosing the initial model retrospectively based on its 2023 performance would use the test period for model selection.

The prediction and residual analyses revealed that the tuned model:

- captures the broad temporal evolution of daily NO₂ concentrations;
- produces predictions that are smoother than the actual observations;
- tends to underestimate abrupt pollution peaks;
- overestimates some unusually low concentrations;
- has a slight overall overestimation bias;
- produces several large errors during periods of high variability.

Despite these limitations, the Ljung–Box tests found no statistically significant residual autocorrelation at lags 7, 14 or 30. This indicates that the engineered lag, rolling, calendar and Fourier features captured most of the systematic temporal dependence present in the historical NO₂ series.

Overall, hyperparameter tuning produced a model that generalized better across the historical cross-validation folds but did not improve the specific 2023 forecasting performance. Among the final representative models included in the comparison, the tuned `HistGradientBoostingRegressor` achieved the lowest MAE and RMSE. It was retained because it was selected through the predefined time-based cross-validation procedure. However, the initial configuration performed slightly better on the specific 2023 test period.

The remaining errors appear to be associated mainly with abrupt pollution events, changing variance and missing external explanatory information rather than strong unmodelled temporal autocorrelation.

Potential improvements could include weather variables, traffic activity, public holidays, emission-related information and indicators of exceptional pollution events.

## 29. Final Comparison of All Forecasting Models

All forecasting approaches are now compared on the 2023 test period using the same evaluation metrics:

- MAE;
- RMSE;
- MAPE.

The comparison includes:

- the Mean baseline;
- the Naïve baseline;
- the Seasonal Naïve baseline;
- the Drift baseline;
- the selected classical statistical model;
- Prophet;
- the tuned `HistGradientBoostingRegressor`.

Using the same test period and evaluation metrics makes it possible to determine whether the more advanced forecasting methods provide a meaningful improvement over simple baseline strategies.

The models will first be compared numerically. Their predictive behaviour, residual diagnostics, complexity and practical limitations will then be considered before selecting the final forecasting approach.

### 29.1 Identifying the Existing Model-Result Variables

Before constructing the final comparison table, the existing notebook variables containing model predictions, evaluation metrics and comparison results must be identified.

This preliminary step prevents:

- inventing variable names that do not exist in the notebook;
- unnecessarily recalculating metrics;
- using inconsistent evaluation periods or formulas;
- accidentally comparing results obtained from different datasets.

The search focuses on variables associated with the baseline models, ARIMA or SARIMA, Prophet, `HistGradientBoostingRegressor`, evaluation metrics and previous comparison tables.

This step only inspects the variables already stored in the notebook environment. It does not modify the data, retrain the models or recalculate their predictions.

In [ ]:
# Search for existing variables containing model results or metrics
search_terms: tuple[str, ...] = (
    "baseline",
    "arima",
    "sarima",
    "prophet",
    "hgbr",
    "metric",
    "result",
    "comparison"
)

matching_variables: list[str] = sorted(
    variable_name
    for variable_name in globals()
    if any(
        term in variable_name.lower()
        for term in search_terms
    )
)

print("Potential result variables:\n")

for variable_name in matching_variables:
    variable_value = globals()[variable_name]

    print(
        f"{variable_name}: "
        f"{type(variable_value).__name__}"
    )

### 29.2 Inspecting the Existing Final Evaluation Tables

The existing evaluation tables are now inspected to identify the final 2023 results for each forecasting approach.

The inspection focuses on:

- the four baseline models;
- the final ARIMA and SARIMA models;
- the selected Prophet model;
- the initial and tuned `HistGradientBoostingRegressor`.

Displaying the existing tables before constructing the final comparison ensures that the correct model names, test metrics and evaluation periods are used without unnecessary recalculation.

In [ ]:
# Collect the most relevant existing evaluation tables
candidate_final_result_tables: dict[str, pd.DataFrame] = {
    "Baseline models": baseline_evaluation,
    "ARIMA and SARIMA models": final_arima_sarima_comparison,
    "Advanced Prophet model": advanced_prophet_results,
    "HistGradientBoostingRegressor models": hgbr_test_comparison
}

# Display each candidate table
for table_name, result_table in candidate_final_result_tables.items():
    print(f"\n{table_name}")
    display(result_table)

### 29.3 Calculating the Final Prophet Test Metrics

The `advanced_prophet_results` table contains the daily Prophet predictions and actual NO₂ concentrations for the complete 2023 test period, rather than an aggregated evaluation table.

The final Prophet test metrics are therefore calculated directly from:

- `Actual`, containing the observed daily NO₂ concentrations;
- `Predicted`, containing the corresponding Prophet forecasts.

MAE, RMSE and MAPE are calculated using the same metric functions applied to the other forecasting models. This ensures that Prophet is evaluated consistently on the same 364-day test period.

In [ ]:
# Extract the actual values and Prophet predictions
prophet_actual: pd.Series = advanced_prophet_results[
    "Actual"
]

prophet_predictions: pd.Series = advanced_prophet_results[
    "Predicted"
]

# Calculate Prophet test metrics
advanced_prophet_mae: float = mean_absolute_error(
    prophet_actual,
    prophet_predictions
)

advanced_prophet_rmse: float = np.sqrt(
    mean_squared_error(
        prophet_actual,
        prophet_predictions
    )
)

advanced_prophet_mape: float = (
    mean_absolute_percentage_error(
        prophet_actual,
        prophet_predictions
    )
    * 100
)

# Store the final Prophet metrics
advanced_prophet_test_metrics: pd.DataFrame = pd.DataFrame(
    {
        "Model": ["Advanced Prophet"],
        "MAE": [advanced_prophet_mae],
        "RMSE": [advanced_prophet_rmse],
        "MAPE (%)": [advanced_prophet_mape]
    }
)

display(advanced_prophet_test_metrics.round(3))

### 29.4 Building the Final Model Comparison Table

The final test results are now combined into a single comparison table.

The comparison includes:

- the four baseline models;
- the selected ARIMA and SARIMA models;
- the advanced Prophet model;
- the tuned `HistGradientBoostingRegressor`.

All metrics were calculated on the same 2023 test period. The models can therefore be compared consistently using MAE, RMSE and MAPE.

Lower values indicate better forecasting performance for all three metrics.

In [ ]:
# Inspect the column names before combining the result tables
tables_to_inspect: dict[str, pd.DataFrame] = {
    "Baselines": baseline_evaluation,
    "ARIMA/SARIMA": final_arima_sarima_comparison,
    "Prophet": advanced_prophet_test_metrics,
    "HistGradientBoostingRegressor": hgbr_test_comparison
}

for table_name, result_table in tables_to_inspect.items():
    print(f"{table_name}: {result_table.columns.tolist()}")

In [ ]:
# Standardize the baseline results
final_baseline_results: pd.DataFrame = baseline_evaluation[
    ["Model", "MAE", "RMSE", "MAPE (%)"]
].copy()

# Keep the final ARIMA and SARIMA results
final_classical_results: pd.DataFrame = (
    final_arima_sarima_comparison[
        ["Model", "MAE", "RMSE", "MAPE (%)"]
    ].copy()
)

# Keep the final Prophet result
final_prophet_results: pd.DataFrame = (
    advanced_prophet_test_metrics[
        ["Model", "MAE", "RMSE", "MAPE (%)"]
    ].copy()
)

# Keep only the tuned HistGradientBoostingRegressor
final_hgbr_results: pd.DataFrame = (
    hgbr_test_comparison.loc[
        hgbr_test_comparison["Model"]
        .str.contains("Tuned", case=False, na=False),
        ["Model", "MAE", "RMSE", "MAPE (%)"]
    ].copy()
)

# Combine all final test results
final_model_comparison: pd.DataFrame = pd.concat(
    [
        final_baseline_results,
        final_classical_results,
        final_prophet_results,
        final_hgbr_results
    ],
    ignore_index=True
)

# Rank models from the lowest to the highest MAE
final_model_comparison = (
    final_model_comparison
    .sort_values(by="MAE", ascending=True)
    .reset_index(drop=True)
)

# Add the MAE ranking
final_model_comparison.insert(
    0,
    "MAE Rank",
    range(1, len(final_model_comparison) + 1)
)

display(final_model_comparison.round(3))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Final test results for the representative models
final_results: pd.DataFrame = pd.DataFrame({
    "Model": [
        "Tuned HistGradientBoosting",
        "Advanced Prophet",
        "SARIMA",
        "ARIMA",
        "Naive",
        "Seasonal Naive",
        "Drift",
        "Mean"
    ],
    "MAE": [
        5.913,
        6.968,
        8.889,
        9.311,
        9.402,
        10.018,
        10.190,
        16.484
    ],
    "RMSE": [
        8.152,
        10.395,
        11.381,
        11.658,
        14.163,
        14.130,
        14.880,
        18.114
    ]
})

# Convert to long format for grouped bars
results_long: pd.DataFrame = final_results.melt(
    id_vars="Model",
    value_vars=["MAE", "RMSE"],
    var_name="Metric",
    value_name="Error"
)

# Create the comparison chart
fig, ax = plt.subplots(figsize=(15, 7))

sns.barplot(
    data=results_long,
    x="Model",
    y="Error",
    hue="Metric",
    palette={
        "MAE": "steelblue",
        "RMSE": "darkorange"
    },
    ax=ax
)

ax.set_title(
    "MAE and RMSE Comparison on the Unseen 2023 Test Period",
    fontsize=15
)
ax.set_xlabel("Forecasting Model")
ax.set_ylabel("Error (µg/m³)")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.3)

# Display the exact value above each bar
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.3f",
        padding=3,
        fontsize=8
    )

ax.legend(title="Metric")
fig.tight_layout()
plt.show()

Comparison of MAE and RMSE across the final representative forecasting models evaluated on the unseen 2023 test period. Lower values indicate better performance.

#### Interpretation

The tuned `HistGradientBoostingRegressor` achieved the best overall performance on the 2023 test period:

- MAE: 5.913;
- RMSE: 8.152;
- MAPE: 54.240%.

It ranks first according to MAE and also produces the lowest RMSE. On average, its daily forecasts differ from the observed NO₂ concentrations by approximately 5.9 units. Its relatively low RMSE also indicates that it handles large forecasting errors better than the other models overall.

`Advanced Prophet` ranks second by MAE, with:

- MAE: 6.968;
- RMSE: 10.395;
- MAPE: 52.742%.

Prophet therefore outperforms the statistical models and all baseline methods according to MAE and RMSE, but it remains less accurate than the tuned machine-learning model.

Among the classical statistical approaches, SARIMA performs better than ARIMA:

- SARIMA MAE: 8.889;
- ARIMA MAE: 9.311.

This suggests that explicitly modelling weekly seasonality improved the forecasts. Nevertheless, both models are outperformed by Prophet and the tuned `HistGradientBoostingRegressor`.

The `Naive` baseline achieves an MAE of 9.402 and performs only slightly worse than ARIMA. This confirms that daily NO₂ concentrations are persistent and that the previous day's value remains a relatively competitive forecast.

The Mean baseline performs worst across all three metrics, with an MAE of 16.484. A constant historical average cannot represent the temporal variation, seasonal behaviour or abrupt changes in daily NO₂ concentrations.

The ranking differs when MAPE is considered. The Naive baseline obtains the lowest MAPE at 50.099%, followed by Advanced Prophet and Drift. However, all MAPE values are relatively high. Because daily NO₂ concentrations sometimes approach zero, percentage errors can become disproportionately large. Therefore, MAPE should be interpreted cautiously and should not be used alone to select the final model.

Overall, the tuned `HistGradientBoostingRegressor` provides the strongest forecasting performance according to the two more reliable absolute-error metrics, MAE and RMSE. It is therefore retained as the best final forecasting model for this study.

### 29.5 Comparing Actual Values with the Final Model Predictions

The tuned `HistGradientBoostingRegressor` was selected as the final forecasting model because it achieved the lowest MAE and RMSE on the 2023 test period.

Its predictions are now compared directly with the actual daily NO₂ concentrations.

This visualization is used to assess whether the selected model:

- follows the general temporal evolution of NO₂ concentrations;
- captures short-term variations;
- reproduces major pollution peaks;
- overestimates unusually low concentrations;
- produces predictions that are smoother than the observed series.

The numerical evaluation and the visual comparison are considered together to assess the practical forecasting behaviour of the final model.

In [ ]:
# Create the final actual-versus-predicted comparison
final_prediction_comparison: pd.DataFrame = pd.DataFrame(
    {
        "Actual": y_test_ml.values,
        "Predicted": tuned_hgbr_predictions
    },
    index=y_test_ml.index
)

# Plot actual and predicted NO₂ concentrations
fig, ax = plt.subplots(figsize=(16, 7))

ax.plot(
    final_prediction_comparison.index,
    final_prediction_comparison["Actual"],
    label="Actual",
    color="black",
    linewidth=1.6,
    alpha=0.85
)

ax.plot(
    final_prediction_comparison.index,
    final_prediction_comparison["Predicted"],
    label="Tuned HistGradientBoostingRegressor",
    color="#F97316",
    linewidth=2.0,
    alpha=0.95
)

ax.set_title(
    "Actual vs Final Model Predictions — 2023"
)
ax.set_xlabel("Date")
ax.set_ylabel("Daily Mean NO₂ Concentration")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 29.6 Residual Analysis of the Final Model

Residual analysis helps evaluate the errors made by the selected model during the unseen 2023 test period.

A residual is calculated as:

\[
\text{Residual}_t = \text{Actual NO₂}_t - \text{Predicted NO₂}_t
\]

Therefore:

- a residual close to zero indicates an accurate prediction;
- a positive residual means that the model underestimated the NO₂ concentration;
- a negative residual means that the model overestimated the NO₂ concentration;
- a large absolute residual indicates a day that was particularly difficult to predict.

The residuals are expected to fluctuate around zero without a clear temporal pattern. However, large positive and negative errors may occur around abrupt changes or unusually high NO₂ concentrations. This would indicate that the model captures the general evolution of the series but struggles to anticipate sudden pollution events using historical and calendar-based features alone.

In [ ]:
# Create the final actual-versus-predicted comparison
final_prediction_comparison: pd.DataFrame = pd.DataFrame(
    {
        "Actual": y_test_ml.values,
        "Predicted": tuned_hgbr_predictions
    },
    index=y_test_ml.index
).dropna()

# Calculate residuals
final_prediction_comparison["Residual"] = (
    final_prediction_comparison["Actual"]
    - final_prediction_comparison["Predicted"]
)

# Plot residuals over time
fig, ax = plt.subplots(figsize=(16, 6))

ax.scatter(
    final_prediction_comparison.index,
    final_prediction_comparison["Residual"],
    color="steelblue",
    alpha=0.65,
    s=28
)

ax.axhline(
    y=0,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Zero error"
)

ax.set_title(
    "Residual Analysis — Tuned HistGradientBoostingRegressor (2023)"
)
ax.set_xlabel("Date")
ax.set_ylabel("Residual: Actual − Predicted (µg/m³)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

#### Interpretation of the Residual analysis of the selected Tuned HistGradientBoostingRegressor during the unseen 2023 test period.

Most residuals are relatively close to zero, indicating that the model produces reasonable predictions on many days. However, several large residuals remain. Positive residuals show days when the model underestimated the observed concentration, while negative residuals represent overestimations.

The largest errors appear to be associated with abrupt variations or extreme NO₂ concentrations. This confirms that the final model produces smoother forecasts and has difficulty capturing sudden pollution peaks. Additional explanatory variables—such as weather, traffic intensity or exceptional events—could help explain these unpredictable variations.

### 29.6 Final Model Selection and Conclusion

The tuned `HistGradientBoostingRegressor` is selected as the final forecasting model for this study.

On the 2023 test period, it achieved:

- MAE: 5.913;
- RMSE: 8.152;
- MAPE: 54.240%.

Among the final representative models included in the comparison, the tuned `HistGradientBoostingRegressor` achieved the lowest MAE and RMSE. It was retained because it was selected through the predefined time-based cross-validation procedure without using the 2023 test period for model selection. However, the initial configuration performed slightly better on the specific 2023 test period, with an MAE of 5.868 and an RMSE of 7.986.

The comparison also produced several important conclusions:

- the Mean baseline performed worst because a constant historical average cannot represent the temporal variability of daily NO₂ concentrations;
- the Naive baseline remained relatively competitive, demonstrating strong day-to-day persistence in the series;
- SARIMA performed slightly better than ARIMA, suggesting that weekly seasonality contributes useful forecasting information;
- Advanced Prophet outperformed the baseline and classical statistical models according to MAE and RMSE;
- the tuned `HistGradientBoostingRegressor` provided the strongest performance among the final representative models according to the two principal absolute-error metrics.

Although the Naive baseline achieved the lowest MAPE, this metric is less reliable for this dataset because some daily NO₂ observations are close to zero. Small actual values can generate disproportionately large percentage errors. Therefore, the final model was selected primarily according to MAE and RMSE.

The visual comparison shows that the selected model follows the main temporal evolution of daily NO₂ concentrations. However, its predictions are smoother than the actual observations. It tends to underestimate abrupt pollution peaks and overestimate some unusually low concentrations.

The residual analysis found no statistically significant autocorrelation at lags 7, 14 or 30. This suggests that the lag, rolling, calendar and Fourier features captured most of the systematic temporal dependence available in the historical NO₂ series.

Nevertheless, the remaining errors indicate that historical concentration data alone cannot fully explain exceptional pollution events. Important external factors such as weather conditions, traffic activity, public holidays, emission patterns and unusual local events were not included in the model.

Overall, the tuned `HistGradientBoostingRegressor` provides the best balance of forecasting accuracy and robust model-selection methodology among the evaluated approaches. It is therefore retained as the final model, while its predictions should be interpreted as expected daily NO₂ concentration estimates rather than precise forecasts of exceptional pollution peaks.

## 30. Overall Conclusion, Limitations and Future Improvements

This study developed and compared several approaches for forecasting daily mean NO₂ concentrations at the Kensington and Chelsea – North Kensington monitoring site.

The analysis covered the period from 2010 to 2023. Data from 2010–2022 were used for model development, while the complete 2023 period was preserved as an unseen test set.

The forecasting approaches included:

- Mean, Naive, Seasonal Naive and Drift baselines;
- autoregressive and moving-average models;
- ARIMA and SARIMA;
- Prophet;
- `HistGradientBoostingRegressor` with lag, rolling, calendar and Fourier features.

The tuned `HistGradientBoostingRegressor` was selected as the final model through the predefined time-based cross-validation procedure, without using the 2023 test period during model selection.

On the 2023 test period, it achieved:

- MAE: 5.913;
- RMSE: 8.152;
- MAPE: 54.240%.

Among the final representative models included in the comparison, the tuned `HistGradientBoostingRegressor` achieved the lowest MAE and RMSE, outperforming the baseline models, ARIMA, SARIMA and Advanced Prophet.

However, the initial `HistGradientBoostingRegressor` configuration performed slightly better on the specific 2023 test period, achieving an MAE of 5.868 and an RMSE of 7.986. The tuned model was nevertheless retained because it generalized better across the historical cross-validation folds. Selecting the initial model retrospectively based on its 2023 performance would have meant using the test set for model selection.

These findings show that combining historical NO₂ observations with engineered temporal features can provide better forecasting accuracy than the baseline, classical statistical and Prophet approaches evaluated in this study.

However, the selected model produces smoother forecasts than the observed series. It captures the overall temporal evolution but frequently underestimates abrupt pollution peaks and overestimates exceptionally low concentrations.

The study has several important limitations:

- only one monitoring site and one pollutant were modelled;
- the model relied primarily on historical NO₂ concentrations and temporal features;
- meteorological variables were unavailable;
- traffic intensity and emission-related information were not included;
- exceptional local events could not be represented;
- interpolated observations may introduce additional uncertainty;
- MAPE was unstable because some actual concentrations were close to zero.

Future work could improve the forecasting system by:

- integrating temperature, wind speed, wind direction, rainfall and atmospheric pressure;
- including traffic, roadworks and emission-related variables;
- adding public holidays and exceptional-event indicators;
- evaluating other monitoring sites and pollutants;
- testing additional machine-learning models such as XGBoost and LightGBM;
- producing prediction intervals to quantify forecast uncertainty;
- using rolling model updates as new observations become available;
- evaluating performance separately by season and pollution level.

Overall, the project demonstrates that time-series analysis and machine-learning feature engineering can produce useful daily NO₂ forecasts. The tuned `HistGradientBoostingRegressor` offered the best balance between forecasting performance and a methodologically robust model-selection procedure.

Nevertheless, accurately predicting exceptional pollution peaks requires external explanatory information beyond the historical concentration series alone.